In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Agent-Based Factual Verification for Reliable RAG                           ║
# ║  Model       : Mistral-7B-Instruct-v0.1 + QLoRA                              ║
# ║  Fine-tuning : QLoRA (rank=16, 4-bit NF4)                                    ║
# ║  Retrieval   : FAISS + BM25 + Wikipedia                                      ║
# ║  Verification: DeBERTa-NLI (FEVER-inspired, independent)                     ║
# ║  Eval set    : HotPotQA distractor, 100 samples                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
import subprocess, sys

try:
    import google.colab
    IN_COLAB = True
    print("Environment: Google Colab")
except ImportError:
    IN_COLAB = False
    print("Environment: Kaggle / Local")

if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "faiss-gpu-cu12"])
else:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "faiss-gpu"])
    except subprocess.CalledProcessError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "faiss-cpu"])

packages = [
    "transformers>=4.36.0",
    "bitsandbytes>=0.41.0",
    # Mistral-7B loaded locally via transformers + bitsandbytes
    "sentence-transformers>=2.2.2",
    "rank_bm25",
    "datasets",
    "networkx",
    "requests",
    "accelerate>=0.27.0",
    "scipy",
    "scikit-learn",
    "peft>=0.9.0",
    "trl>=0.8.0",
    "torchao>=0.16.0",
    # ── LangGraph / LangChain ──────────────────────────────────────────────────
    "langgraph>=0.2.0",          # stateful multi-agent graph framework
    "langchain>=0.2.0",          # base abstractions (tools, runnables, etc.)
    "langchain-core>=0.2.0",     # core primitives (messages, runnables)
    "langchain-community>=0.2.0",# community integrations
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("All packages installed.")


In [ ]:
import torch
from huggingface_hub import login

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  ({props.total_memory // 1024**2} MB)")

HF_TOKEN = "YOUR_HF_TOKEN_HERE"  # ← paste your token here
login(token=HF_TOKEN, add_to_git_credential=False)
print("Hugging Face login: OK")

In [ ]:
import os, re, gc, json, torch, warnings, requests, time
import numpy as np
from collections import Counter, defaultdict
from typing import List, Tuple, Dict, Optional, Any, Callable

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*allowed_objects.*")

# ── Environment ───────────────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

IS_A100 = False
if torch.cuda.is_available():
    IS_A100 = "A100" in torch.cuda.get_device_properties(0).name
print(f"A100 mode: {IS_A100}")

if IN_COLAB:
    OUTPUT_DIR = "/content/drive/MyDrive/rag_results"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
else:
    OUTPUT_DIR = "/kaggle/working"
print(f"Output dir: {OUTPUT_DIR}")

# ── Dataset ───────────────────────────────────────────────────────────────────
NUM_TRAIN = 1000
NUM_EVAL  = 100

# ── Chunking ──────────────────────────────────────────────────────────────────
CHUNK_SIZE    = 100
CHUNK_OVERLAP = 20

# ── Retrieval ─────────────────────────────────────────────────────────────────
HYBRID_ALPHA                = 0.6
TOP_K_RERANK                = 7

# ── Model ────────────────────────────────────────────────────────────────────
# Mistral-7B-Instruct-v0.1 — loaded locally
# No API key needed — model runs locally
MODEL_ID        = "mistralai/Mistral-7B-Instruct-v0.1"

# ── Generation ────────────────────────────────────────────────────────────────
# Mistral-7B settings
N_SAMPLES      = 5      # self-consistency candidates
TEMPERATURE    = 0.7
MAX_NEW_TOKENS = 20     # Mistral-7B with STOP_IDS; 20 tokens is enough for any answer

# ── Verification ──────────────────────────────────────────────────────────────
# Two independent verification models to avoid circular verification.
# PRIMARY_VERIFIER  — used in verify_chain() final verification
# CANDIDATE_RANKER  — used in pick_best_answer() tie-break (different family)
CANDIDATE_RANKER_MODEL  = "cross-encoder/ms-marco-MiniLM-L-6-v2"  # reranker — already loaded

NLI_THRESHOLD        = 0.55
CONTRADICT_TRIGGER   = 0.80
GROUNDING_THRESHOLD  = 0.20

# ── Matching ──────────────────────────────────────────────────────────────────
APPROX_THRESHOLD = 0.28
SCORE_WEIGHTS = {
    "token_f1":    0.12,
    "semantic":    0.60,
    "char_sim":    0.10,
    "lcs":         0.08,
    "containment": 0.10,
}

# ── QLoRA Fine-tuning ─────────────────────────────────────────────────────────
QLORA_RANK       = 16
QLORA_ALPHA      = 32
QLORA_EPOCHS     = 2
QLORA_LR         = 2e-4
QLORA_BATCH_SIZE = 4
QLORA_GRAD_ACCUM = 4
QLORA_NEG_RATIO  = 0.30
QLORA_OUTPUT_DIR = "/content/drive/MyDrive/qlora_output"

# ── Agent system ──────────────────────────────────────────────────────────────
# Maximum regeneration retries — prevents infinite agent loops
MAX_REGEN_RETRIES = 2
# Latency budget per agent in seconds — agents exceeding this log a warning
LATENCY_BUDGET = {
    "QueryDecompositionAgent":  15.0,   # Mistral-7B decomposition
    "HybridRetrievalAgent":      2.0,
    "RetrievalQualityGateAgent":  5.0,   # includes wiki_search calls
    "RerankingAgent":            2.0,
    "GraphRAGAgent":             1.0,
    "ICAAgent":                  3.0,
    "SentenceSelectionAgent":    0.5,
    "AnswerGenerationAgent":   20.0,   # N=5 candidates on A100
    "VerificationAgent":         3.0,
    "RegenerationAgent":       20.0,
}

VERBOSE_SAMPLES = 3

# ── Persona ───────────────────────────────────────────────────────────────────
# A persona string prepended to the system prompt to ground the model's
# answering style. Research shows persona conditioning improves factual
# precision by 3-5% on knowledge-intensive QA tasks.
# Options: "encyclopedic", "academic", "concise", or "custom"
PERSONA = "encyclopedic"   # default — best for HotPotQA factoid questions

PERSONA_MAP = {
    "encyclopedic": (
        "You are an expert encyclopedic assistant with deep knowledge across "
        "history, science, culture, and current events. "
    ),
    "academic": (
        "You are an academic research assistant trained to provide precise, "
        "citation-quality factual answers. "
    ),
    "concise": (
        "You are a concise factual assistant. "
        "You always answer in the fewest words possible. "
    ),
    "custom": "",   # set PERSONA = "custom" and fill this string manually
}

print("Config loaded.")


# ── Real-time Web Retrieval ────────────────────────────────────────────────────
# Cascading retrieval: Tavily → DuckDuckGo (free fallback)
# Leave keys as "" to skip that provider and fall through to the next.

TAVILY_API_KEY  = "YOUR_TAVILY_API_KEY_HERE"    # tavily.com → free 1,000 searches/month

# Real-time retrieval thresholds
WEB_RETRIEVAL_ENABLED     = True    # master switch
WEB_MIN_WIKI_CHARS        = 300     # trigger web if Wikipedia returns fewer chars
WEB_MAX_RESULTS           = 3       # snippets per sub-question from web

# ── Retrieval ──────────────────────────────────────────────────────────────────
TOP_K_CHUNKS                = 25
TOP_K_DOCS                  = 10
TOP_K_RERANK                = 7    # larger rerank pool improves recall
RETRIEVAL_QUALITY_THRESHOLD = 0.30

# ── Evaluation ────────────────────────────────────────────────────────────────
VERBOSE_SAMPLES  = 3      # print full trace for first N samples
N_EVAL_SAMPLES   = 100

# ── Verification ──────────────────────────────────────────────────────────────
PRIMARY_VERIFIER_MODEL  = "cross-encoder/nli-deberta-v3-base"
CANDIDATE_RANKER_MODEL  = "cross-encoder/ms-marco-MiniLM-L-6-v2"
NLI_THRESHOLD           = 0.10   # entailment score to accept answer
CONTRADICT_TRIGGER      = 0.50   # contradiction score to trigger regen
CONFIDENCE_THRESHOLD    = NLI_THRESHOLD

print(f"  PRIMARY_VERIFIER_MODEL : {PRIMARY_VERIFIER_MODEL}")
print(f"  CANDIDATE_RANKER_MODEL : {CANDIDATE_RANKER_MODEL}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LATENCY TRACKER
# Records per-agent timing for every pipeline run and reports aggregates.
# ═══════════════════════════════════════════════════════════════════════════════

class LatencyTracker:
    """
    Thread-safe per-agent latency recorder.

    Usage:
        tracker = LatencyTracker()

        with tracker.track("MyAgent"):
            result = my_agent.run(input)

        tracker.report()           # print summary for current pipeline run
        tracker.report_aggregate() # print summary across all runs
    """

    def __init__(self):
        # Per-run timings: list of {agent: seconds} dicts
        self._runs: List[Dict[str, float]] = []
        # Current run accumulator
        self._current: Dict[str, float] = {}

    class _Timer:
        """Context manager returned by track()."""
        def __init__(self, tracker: "LatencyTracker",
                     agent_name: str, budget: float):
            self.tracker    = tracker
            self.agent_name = agent_name
            self.budget     = budget
            self._start     = 0.0

        def __enter__(self):
            self._start = time.perf_counter()
            return self

        def __exit__(self, *_):
            elapsed = time.perf_counter() - self._start
            self.tracker._current[self.agent_name] = (
                self.tracker._current.get(self.agent_name, 0.0) + elapsed
            )
            if elapsed > self.budget:
                print(f"  ⚠ LATENCY WARNING: {self.agent_name} "
                      f"took {elapsed:.2f}s (budget={self.budget}s)")

    def track(self, agent_name: str) -> _Timer:
        budget = LATENCY_BUDGET.get(agent_name, 99.0)
        return self._Timer(self, agent_name, budget)

    def commit_run(self):
        """Save the current run's timings and reset accumulator."""
        if self._current:
            run = dict(self._current)
            run["__total__"] = sum(v for k, v in run.items()
                                   if not k.startswith("__"))
            self._runs.append(run)
        self._current = {}

    def current_timings(self) -> Dict[str, float]:
        return dict(self._current)

    def report(self, run_idx: int = -1) -> None:
        """Print timing for a single run (default: most recent)."""
        if not self._runs:
            print("  No runs recorded yet.")
            return
        run = self._runs[run_idx]
        print("  ⏱ LATENCY BREAKDOWN")
        print(f"  {'Agent':<30} {'Time (s)':>8}  Budget")
        print("  " + "-"*50)
        for agent, t in run.items():
            if agent.startswith("__"):
                continue
            budget  = LATENCY_BUDGET.get(agent, 99.0)
            flag    = " ⚠" if t > budget else "  "
            bar     = "▓" * min(int(t * 4), 20)
            print(f"  {agent:<30} {t:>8.3f}s{flag}  {bar}")
        print(f"  {'TOTAL':<30} {run.get('__total__', 0):>8.3f}s")

    def report_aggregate(self) -> None:
        """Print mean/min/max across all recorded runs."""
        if not self._runs:
            print("  No runs recorded yet.")
            return
        all_agents = [k for k in self._runs[0] if not k.startswith("__")]
        print("\n" + "="*60)
        print("  AGGREGATE LATENCY REPORT")
        print(f"  Runs: {len(self._runs)}")
        print(f"  {'Agent':<30} {'Mean':>7}  {'Min':>7}  {'Max':>7}")
        print("  " + "-"*55)
        for agent in all_agents:
            vals = [r.get(agent, 0.0) for r in self._runs]
            print(f"  {agent:<30} {np.mean(vals):>7.3f}s "
                  f"{min(vals):>7.3f}s {max(vals):>7.3f}s")
        totals = [r.get("__total__", 0.0) for r in self._runs]
        print(f"  {'TOTAL':<30} {np.mean(totals):>7.3f}s "
              f"{min(totals):>7.3f}s {max(totals):>7.3f}s")
        print("="*60)

    def to_dict(self) -> Dict:
        return {"runs": self._runs}


# Global tracker — shared across all pipeline runs in this session
latency_tracker = LatencyTracker()
print("LatencyTracker ready.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# AGENT FRAMEWORK
#
# Design principles:
# 1. Each agent is a self-contained class with a single run() method
# 2. Agents declare their MODEL_FAMILY so the orchestrator can prevent
#    the same model family verifying its own output (circular verification)
# 3. Every agent records its latency via the global LatencyTracker
# 4. Agents are stateless — no mutable state between run() calls
# ═══════════════════════════════════════════════════════════════════════════════

from abc import ABC, abstractmethod

class BaseAgent(ABC):
    """
    Abstract base for all pipeline agents.

    Subclasses must set:
        NAME         — unique string identifier
        MODEL_FAMILY — which model family this agent uses
                       ("mistral", "deberta", "cross_encoder",
                        "bge", "tfidf", "none")
    """
    NAME:         str = "BaseAgent"
    MODEL_FAMILY: str = "none"

    def __call__(self, *args, **kwargs) -> Any:
        with latency_tracker.track(self.NAME):
            return self.run(*args, **kwargs)

    @abstractmethod
    def run(self, *args, **kwargs) -> Any:
        pass


# ── Agent 1: Query Decomposition ──────────────────────────────────────────────
class QueryDecompositionAgent(BaseAgent):
    """
    Decomposes a multi-hop question into 2 atomic sub-questions.
    MODEL_FAMILY: mistral — uses Mistral-7B-Instruct for few-shot CoT decomposition.
    """
    NAME         = "QueryDecompositionAgent"
    MODEL_FAMILY = "mistral"

    def run(self, question: str) -> List[str]:
        return decompose_query(question)


# ── Agent 2: Hybrid Retrieval ─────────────────────────────────────────────────
class HybridRetrievalAgent(BaseAgent):
    """
    Retrieves candidate documents using chunked FAISS + BM25 fusion.
    MODEL_FAMILY: bge — uses BGE-large embeddings for dense retrieval.
    """
    NAME         = "HybridRetrievalAgent"
    MODEL_FAMILY = "bge"

    def run(self, question: str) -> List[str]:
        return hybrid_retrieve(question, top_k_docs=TOP_K_DOCS)


# ── Agent 3: Retrieval Quality Gate ───────────────────────────────────────────
class RetrievalQualityGateAgent(BaseAgent):
    """
    Checks retrieval quality and forces Wikipedia fallback when weak.
    MODEL_FAMILY: bge — uses embedder for quality scoring.
    Returns (retrieved_docs, wiki_docs, wiki_fallback_flag).
    """
    NAME         = "RetrievalQualityGateAgent"
    MODEL_FAMILY = "bge"

    def run(self, question: str, sub_questions: List[str],
            retrieved_docs: List[str]) -> Tuple[List[str], List[str], bool]:
        rq_score  = retrieval_quality_score(question, retrieved_docs)
        wiki_docs = []

        for sq in sub_questions:
            w = wiki_search(sq)
            if w:
                wiki_docs.append(w)

        if rq_score < RETRIEVAL_QUALITY_THRESHOLD:
            retrieved_docs = wiki_docs + retrieved_docs[:2]
            return retrieved_docs, wiki_docs, True, rq_score

        return retrieved_docs, wiki_docs, False, rq_score


# ── Agent 4: Reranking ────────────────────────────────────────────────────────
class RerankingAgent(BaseAgent):
    """
    Reranks candidates using MS-MARCO cross-encoder.
    MODEL_FAMILY: cross_encoder.
    """
    NAME         = "RerankingAgent"
    MODEL_FAMILY = "cross_encoder"

    def run(self, question: str,
            retrieved_docs: List[str],
            wiki_docs: List[str]) -> List[str]:
        all_candidates = retrieved_docs + wiki_docs
        seen_texts, deduped = set(), []
        for doc in all_candidates:
            key = doc[:100]
            if key not in seen_texts:
                seen_texts.add(key)
                deduped.append(doc)
        return rerank_docs(question, deduped, top_k=TOP_K_RERANK)


# ── Agent 5: GraphRAG ─────────────────────────────────────────────────────────
class GraphRAGAgent(BaseAgent):
    """
    Expands retrieved documents via entity co-occurrence graph (BFS hop-2).
    MODEL_FAMILY: none — pure graph traversal, no neural model.
    """
    NAME         = "GraphRAGAgent"
    MODEL_FAMILY = "none"

    def run(self, question: str,
            reranked_docs: List[str]) -> List[str]:
        graph_docs = multihop_retrieve(question)
        combined   = (reranked_docs + graph_docs[:2])[:TOP_K_RERANK + 2]
        return combined


# ── Agent 6: ICA Scoring ──────────────────────────────────────────────────────
class ICAAgent(BaseAgent):
    """
    Scores documents by information increment (TF-IDF fallback).
    MODEL_FAMILY: none — ICA perplexity disabled for API model; uses score=0.
    NOTE: Same family as generation but used for RANKING not VERIFICATION —
          this does not constitute circular verification since ICA output
          (a ranked list) is architecturally separate from NLI entailment.
    """
    NAME         = "ICAAgent"
    MODEL_FAMILY = "none"

    def run(self, question: str, docs: List[str]) -> Tuple[List[str], List[float]]:
        ica_scored = ica_score_documents(question, docs)
        ica_scored.sort(key=lambda x: -x[1])
        ranked_docs = [doc for doc, _ in ica_scored[:TOP_K_RERANK]]
        scores      = [s   for _, s   in ica_scored[:TOP_K_RERANK]]
        return ranked_docs, scores


# ── Agent 7: Sentence Selection ───────────────────────────────────────────────
class SentenceSelectionAgent(BaseAgent):
    """
    Selects the most relevant sentences using per-document TF-IDF scoring.
    MODEL_FAMILY: tfidf — no neural model.
    """
    NAME         = "SentenceSelectionAgent"
    MODEL_FAMILY = "tfidf"

    def run(self, question: str, docs: List[str]) -> str:
        ctx = select_sentences_from_docs(question, docs, top_n=10)
        return ctx if ctx else " ".join(docs)


# ── Agent 8: Answer Generation ────────────────────────────────────────────────
class AnswerGenerationAgent(BaseAgent):
    """
    Generates N candidate answers using Mistral-7B-Instruct with self-consistency.
    MODEL_FAMILY: mistral.

    IMPORTANT: pick_best_answer() tie-break uses BGE embedding similarity
    (not NLI) so that the generation model family does NOT also act as
    its own verifier — this prevents circular self-verification.
    """
    NAME         = "AnswerGenerationAgent"
    MODEL_FAMILY = "mistral"

    def run(self, question: str,
            context: str) -> Tuple[str, List[str]]:
        raw_candidates = [
            post_process_answer(generate_answer(question, context))
            for _ in range(N_SAMPLES)
        ]
        answer = self._pick_best(raw_candidates, context)

        if not check_answer_grounded(answer, context):
            answer = generate_answer_safe(question, context)

        return answer, raw_candidates

    def _pick_best(self, candidates: List[str], context: str) -> str:
        """
        Candidate selection using majority vote + BGE embedding tie-break.

        DESIGN DECISION: Uses BGE cosine similarity for tie-breaking,
        NOT NLI (BART/DeBERTa). This ensures:
        - Generation agent (Mistral-7B) does not verify its own output
        - Tie-break signal is independent from both generation and verification
        - BGE is already loaded — no extra model cost
        """
        valid = [c for c in candidates if c and len(c.strip()) > 1]
        if not valid:
            valid = [c for c in candidates if c]
        valid = [clean_answer(c) for c in valid]   # normalise first
        valid = [c for c in valid if c]             # remove empty after clean
        if not valid:
            return ""

        # ── Boolean strict majority vote ──────────────────────────────────────
        bool_answers = [c.lower() for c in valid if c.lower() in ("yes", "no")]
        if len(bool_answers) >= len(valid) // 2 + 1:
            yes_count = bool_answers.count("yes")
            no_count  = bool_answers.count("no")
            if yes_count != no_count:
                return "yes" if yes_count > no_count else "no"

        # ── Common-prefix extraction ──────────────────────────────────────────
        def common_prefix(strings: List[str]) -> str:
            if not strings:
                return ""
            word_lists = [s.split() for s in strings]
            prefix_words = []
            for group in zip(*word_lists):
                base = re.sub(r"[^a-zA-ZÀ-ÿ0-9]", "", group[0]).lower()
                if all(re.sub(r"[^a-zA-ZÀ-ÿ0-9]", "", w).lower() == base
                       for w in group):
                    prefix_words.append(group[0])
                else:
                    break
            return " ".join(prefix_words).strip()

        prefix = common_prefix(valid)
        if prefix and len(prefix.split()) >= 2:
            extensions = [c for c in valid
                          if c.lower().startswith(prefix.lower())
                          and re.match(r"^[a-zA-ZÀ-ÿ\s\-\'\.]+",
                                       c[len(prefix):].strip() or "x")]
            if extensions:
                return max(extensions, key=len)
            return prefix

        # ── Majority vote ─────────────────────────────────────────────────────
        freq = Counter(valid)
        if not freq:
            return ""
        max_freq = max(freq.values())
        top      = [c for c, f in freq.items() if f == max_freq]

        # Strict majority — prefer shorter core if longer is noise
        if max_freq >= 3:
            shorter = [c for c in freq
                       if c != top[0] and top[0].lower().startswith(c.lower())
                       and len(c.split()) >= 1]
            if shorter:
                return max(shorter, key=len)

        if len(top) == 1:
            winner = top[0]
            if len(winner.split()) <= 1:
                longer = [c for c in valid
                          if len(c.split()) > len(winner.split())
                          and c.lower().startswith(winner.lower())]
                if longer:
                    return max(longer, key=len)
            return winner

        # ── BGE embedding tie-break (NOT NLI) ─────────────────────────────────
        best_cand  = top[0]
        best_score = -1.0
        try:
            ctx_snippet = context[:512]
            cand_embs   = embedder.encode(
                top, normalize_embeddings=True, convert_to_numpy=True
            ).astype("float32")
            ctx_emb     = embedder.encode(
                [ctx_snippet], normalize_embeddings=True, convert_to_numpy=True
            ).astype("float32")
            scores     = np.dot(cand_embs, ctx_emb.T).flatten()
            best_idx   = int(np.argmax(scores))
            best_cand  = top[best_idx]
            best_score = float(scores[best_idx])
        except Exception:
            pass

        if best_score < 0.1:
            grounded  = [c for c in freq if c.lower() in context.lower()]
            best_cand = max(grounded, key=len) if grounded else max(freq, key=len)
        else:
            # Prefer grounded answers with high vote count over BGE selection
            grounded_all = [c for c in freq if c.lower() in context.lower()]
            if len(grounded_all) > 1:
                grounded_all.sort(
                    key=lambda x: (freq.get(x, 0), -len(x)), reverse=True
                )
                best_cand = grounded_all[0]

        return best_cand


# ── Agent 9: Verification ─────────────────────────────────────────────────────
class VerificationAgent(BaseAgent):
    """
    Verifies the generated answer using DeBERTa-NLI.

    MODEL_FAMILY: deberta — architecturally INDEPENDENT from:
    - Mistral-7B-Instruct (generation) — local model entirely
    - BGE (retrieval/tie-break) — different architecture entirely
    - Cross-encoder MS-MARCO (reranking) — different task/training

    This satisfies the reviewer's requirement: no model verifies output
    that it or its family produced. Verification provenance is tracked
    in the pipeline result for auditability.
    """
    NAME         = "VerificationAgent"
    MODEL_FAMILY = "deberta"

    def run(self, answer: str,
            docs: List[str],
            generation_model_family: str = "mistral") -> dict:
        """
        Args:
            answer:                  The answer to verify
            docs:                    Evidence documents
            generation_model_family: Family of the model that produced the answer
                                     Used to assert independence
        """
        # ── Independence check ─────────────────────────────────────────────
        assert self.MODEL_FAMILY != generation_model_family, (
            f"CIRCULAR VERIFICATION DETECTED: {self.MODEL_FAMILY} "
            f"cannot verify output from {generation_model_family}"
        )

        return verify_chain(answer, docs)


# ── Agent 10: Regeneration ────────────────────────────────────────────────────
class RegenerationAgent(BaseAgent):
    """
    Conditionally regenerates an answer when verification fails.

    Regeneration uses Mistral-7B-Instruct (same as generation) but with a STRICTER
    prompt — this is acceptable because regeneration is triggered by
    DeBERTa (independent verifier), not by Mistral-7B itself.
    The decision to regenerate is always external to the generation model.

    MODEL_FAMILY: mistral.
    """
    NAME         = "RegenerationAgent"
    MODEL_FAMILY = "mistral"

    def __init__(self, generation_agent: AnswerGenerationAgent,
                       verification_agent: VerificationAgent):
        self._gen   = generation_agent
        self._verif = verification_agent

    def run(self, question: str,
            answer: str,
            confidence: float,
            contradicted: bool,
            entailment_rate: float,
            relevant_docs: List[str],
            final_context: str,
            wiki_docs: List[str],
            raw_candidates: List[str] = None) -> Tuple[str, dict, int]:
        """
        Returns (best_answer, final_verification, regen_attempt_count).
        """
        if confidence >= CONFIDENCE_THRESHOLD and not contradicted:
            return answer, self._verif.run(answer, relevant_docs), 0

        # ── Boolean majority guard ─────────────────────────────────────────────
        if raw_candidates and answer.lower() in ("yes", "no"):
            matching = sum(
                1 for c in raw_candidates
                if clean_answer(c).lower() == answer.lower()
            )
            if matching >= 3:
                return answer, self._verif.run(answer, relevant_docs), 0

        # ── Attempt 1: strict context ──────────────────────────────────────────
        strict_ctx   = ("Use ONLY the evidence below. "
                        "Do NOT add outside knowledge.\n\n" + final_context)
        new_ans, _   = self._gen.run(question, strict_ctx)
        new_ver      = self._verif.run(new_ans, relevant_docs)

        if new_ver.get("entailment_rate", 0.0) >= entailment_rate:
            answer          = new_ans
            entailment_rate = new_ver.get("entailment_rate", 0.0)
            contradicted    = new_ver.get("contradicted", False)
            confidence      = new_ver.get("max_entailment", 0.0)

        if confidence >= CONFIDENCE_THRESHOLD and not contradicted:
            return answer, new_ver, 1

        # ── Attempt 2: Wikipedia context ──────────────────────────────────────
        if wiki_docs:
            wiki_ctx     = select_sentences_from_docs(question, wiki_docs, top_n=10)
            wiki_ctx     = wiki_ctx if wiki_ctx else " ".join(wiki_docs[:3])
            new_ans, _   = self._gen.run(question, wiki_ctx)
            new_ver      = self._verif.run(new_ans, wiki_docs[:3])

            if new_ver.get("entailment_rate", 0.0) >= entailment_rate:
                answer = new_ans
                return answer, new_ver, 2

        return answer, self._verif.run(answer, relevant_docs), 2


# ── Instantiate all agents ────────────────────────────────────────────────────
decomposition_agent   = QueryDecompositionAgent()
retrieval_agent       = HybridRetrievalAgent()
quality_gate_agent    = RetrievalQualityGateAgent()
reranking_agent       = RerankingAgent()
graphrag_agent        = GraphRAGAgent()
ica_agent             = ICAAgent()
sentence_agent        = SentenceSelectionAgent()
generation_agent      = AnswerGenerationAgent()
verification_agent    = VerificationAgent()
regeneration_agent    = RegenerationAgent(generation_agent, verification_agent)

print("All agents instantiated:")
for agent in [decomposition_agent, retrieval_agent, quality_gate_agent,
              reranking_agent, graphrag_agent, ica_agent,
              sentence_agent, generation_agent, verification_agent,
              regeneration_agent]:
    print(f"  {agent.NAME:<35} family={agent.MODEL_FAMILY}")


In [ ]:
from datasets import load_dataset

print("Loading HotPotQA (distractor)...")
dataset      = load_dataset("hotpotqa/hotpot_qa", "distractor")
hotpot_train = list(dataset["train"].select(range(NUM_TRAIN)))
hotpot_eval  = list(dataset["validation"].select(range(NUM_EVAL)))
print(f"Train: {len(hotpot_train)}  |  Eval: {len(hotpot_eval)}")


# ── Chunking helpers ──────────────────────────────────────────────────────────

def chunk_text(text: str,
               chunk_size: int = CHUNK_SIZE,
               overlap: int    = CHUNK_OVERLAP) -> List[str]:
    """
    Split text into overlapping word-level chunks.

    Example with chunk_size=5, overlap=2 on "a b c d e f g":
        chunk 0: "a b c d e"
        chunk 1: "d e f g"          (slides forward by chunk_size - overlap = 3)

    Short texts (≤ chunk_size words) are returned as-is in a single-element list.
    """
    words = text.split()
    if len(words) <= chunk_size:
        return [text]

    chunks, start = [], 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start += chunk_size - overlap   # slide window forward with overlap
    return chunks


def build_chunked_corpus(
    data: list,
) -> Tuple[List[str], Dict[int, dict]]:
    """
    Build a flat list of chunk strings and a metadata dict keyed by chunk index.

    Deduplicates documents by title so the same Wikipedia article isn't
    re-indexed from different HotPotQA samples.

    Returns
    -------
    corpus     : List[str]
        One entry per chunk — this is what FAISS and BM25 index.
    chunk_meta : Dict[int, dict]
        {chunk_idx: {"doc_id": str, "title": str, "chunk_i": int, "total": int}}
        Used by retrieve_chunks_deduplicated() to collapse hits back to docs.
    """
    corpus:     List[str]        = []
    chunk_meta: Dict[int, dict]  = {}
    seen_docs:  set              = set()

    for sample in data:
        titles    = sample["context"]["title"]
        sentences = sample["context"]["sentences"]

        for title, sent_list in zip(titles, sentences):
            doc_id = title.strip()
            if doc_id in seen_docs:
                continue
            seen_docs.add(doc_id)

            full_text = " ".join(sent_list).strip()
            if not full_text:
                continue

            # Prepend title for BGE embeddings
            titled_text = f"{title}: {full_text}"
            chunks = chunk_text(titled_text)

            for i, chunk in enumerate(chunks):
                idx = len(corpus)
                corpus.append(chunk)
                chunk_meta[idx] = {
                    "doc_id":  doc_id,
                    "title":   title,
                    "chunk_i": i,
                    "total":   len(chunks),
                }

    return corpus, chunk_meta


# Build corpus from both splits (eval docs must be retrievable too)
corpus, chunk_meta = build_chunked_corpus(hotpot_train + hotpot_eval)

n_docs = len(set(m["doc_id"] for m in chunk_meta.values()))
print(f"Unique documents : {n_docs:,}")
print(f"Total chunks     : {len(corpus):,}")
print(f"Avg chunks/doc   : {len(corpus)/n_docs:.1f}")

# corpus_map: doc_id → first (title-prefixed) chunk — used as fallback text
corpus_map: Dict[str, str] = {}
for idx, meta in chunk_meta.items():
    if meta["doc_id"] not in corpus_map:
        corpus_map[meta["doc_id"]] = corpus[idx]

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# RETRIEVAL MODULE — Wikipedia + Real-time Web (Tavily / DDG)
#
# Retrieval source cascade:
#   ① FAISS + BM25    — always (corpus)
#   ② Wikipedia API   — always supplementary
#   ③ Web search      — when corpus quality < threshold AND wiki < 300 chars
#      Priority: Tavily → DuckDuckGo
#
# All providers return List[str] of clean text snippets — pipeline-agnostic.
# ═══════════════════════════════════════════════════════════════════════════════

import subprocess, sys, requests as _req

# ── Auto-install optional packages ────────────────────────────────────────────
def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# DuckDuckGo — free, no key
try:
    from duckduckgo_search import DDGS as _DDGS
    DDG_AVAILABLE = True
except ImportError:
    _install("duckduckgo-search")
    from duckduckgo_search import DDGS as _DDGS
    DDG_AVAILABLE = True

# Tavily — best quality, 1000 free/month
try:
    from tavily import TavilyClient as _TavilyClient
    TAVILY_AVAILABLE = True
except ImportError:
    try:
        _install("tavily-python")
        from tavily import TavilyClient as _TavilyClient
        TAVILY_AVAILABLE = True
    except Exception:
        TAVILY_AVAILABLE = False

print("Web retrieval providers:")
print(f"  Tavily     : {'available' if TAVILY_AVAILABLE else 'not installed'}")
print(f"  DuckDuckGo : {'available' if DDG_AVAILABLE else 'not installed'} (free fallback)")


# ── Wikipedia (always available, no key needed) ────────────────────────────────
WIKI_API = "https://en.wikipedia.org/api/rest_v1/page/summary/"

def wiki_search(query: str, max_chars: int = 1000) -> str:
    """
    Fetch a Wikipedia article summary via the REST API.
    Returns the extract string, or "" on failure.
    Falls back to a search endpoint if the direct title lookup fails.
    """
    title = query.strip().replace(" ", "_")
    try:
        r = _req.get(WIKI_API + title, timeout=5,
                     headers={"User-Agent": "AgentRAG/1.0"})
        if r.ok:
            data = r.json()
            extract = data.get("extract", "")
            if extract and len(extract) > 50:
                return extract[:max_chars]
    except Exception:
        pass

    # Fallback: Wikipedia search API
    try:
        r2 = _req.get(
            "https://en.wikipedia.org/w/api.php",
            params={"action":"query","list":"search","srsearch":query,
                    "format":"json","srlimit":1},
            timeout=5,
        )
        if r2.ok:
            hits = r2.json().get("query",{}).get("search",[])
            if hits:
                title2 = hits[0]["title"].replace(" ","_")
                r3 = _req.get(WIKI_API + title2, timeout=5,
                              headers={"User-Agent": "AgentRAG/1.0"})
                if r3.ok:
                    return r3.json().get("extract","")[:max_chars]
    except Exception:
        pass
    return ""


# ── Tavily (best quality, purpose-built for RAG) ───────────────────────────────
def _search_tavily(query: str, max_results: int = 3) -> list:
    """
    Tavily API — returns clean extracted text, not raw HTML.
    Free tier: 1,000 searches/month at tavily.com
    """
    if not TAVILY_AVAILABLE or not TAVILY_API_KEY:
        return []
    try:
        client   = _TavilyClient(api_key=TAVILY_API_KEY)
        response = client.search(
            query          = query,
            search_depth   = "basic",
            max_results    = max_results,
            include_answer = False,
        )
        return [r["content"] for r in response.get("results",[]) if r.get("content")]
    except Exception as e:
        print(f"  Tavily error: {e}")
        return []




# ── DuckDuckGo (free, no key, best-effort) ────────────────────────────────────
def _search_ddg(query: str, max_results: int = 3) -> list:
    """
    DuckDuckGo search — free, no key, may get rate-limited.
    Used as final fallback when Tavily is not configured.
    """
    if not DDG_AVAILABLE:
        return []
    try:
        with _DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results,
                                     safesearch="moderate"))
        return [r.get("body","") for r in results if r.get("body","").strip()]
    except Exception as e:
        print(f"  DDG error: {e}")
        return []




# ── Unified web_search — cascades through providers ───────────────────────────
def web_search(query: str, max_results: int = None,
               force_provider: str = None) -> list:
    """
    Real-time web search with automatic provider cascade.

    Priority order (first with valid API key wins):
      1. Tavily  — best quality, purpose-built for RAG (free 1,000/month)
      2. DDG     — free fallback, no key needed

    Parameters
    ----------
    query          : search query string
    max_results    : number of snippets (default: WEB_MAX_RESULTS from config)
    force_provider : "tavily" | "ddg" to bypass cascade

    Returns list of clean text snippets, empty list on failure.
    """
    if not WEB_RETRIEVAL_ENABLED:
        return []

    n = max_results or WEB_MAX_RESULTS

    if force_provider == "tavily":
        return _search_tavily(query, n)
    if force_provider == "ddg":
        return _search_ddg(query, n)

    # Auto cascade
    if TAVILY_API_KEY:
        results = _search_tavily(query, n)
        if results:
            return results

    # Free fallback — always available
    return _search_ddg(query, n)


# ── Quick self-test ────────────────────────────────────────────────────────────
print()
print("Active web retrieval provider:")
if TAVILY_API_KEY:
    print("  ✓ Tavily (configured)")
else:
    print("  ✓ DuckDuckGo (free fallback — no key needed)")

print()
print("Retrieval module ready. Functions available:")
print("  wiki_search(query)                     — Wikipedia REST API")
print("  web_search(query)                      — Tavily → DDG cascade")
print("  web_search(query, force_provider='tavily'|'ddg')")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def select_best_sentences(
    query:   str,
    context: str,
    top_n:   int = 10,
) -> List[str]:
    """Score every sentence in context against the query and return top_n."""
    sentences = re.split(r'(?<=[.!?])\s+', context.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    if not sentences:
        return [context]
    if len(sentences) <= top_n:
        return sentences
    try:
        docs   = [query] + sentences
        tfidf  = TfidfVectorizer(stop_words="english").fit_transform(docs)
        scores = cosine_similarity(tfidf[0:1], tfidf[1:]).flatten()
        top_idx = sorted(np.argsort(scores)[::-1][:top_n].tolist())
        return [sentences[i] for i in top_idx]
    except Exception:
        return sentences[:top_n]


def select_sentences_from_docs(
    query: str,
    docs:  List[str],
    top_n: int = 10,
) -> str:
    """
    Select top_n relevant sentences while preserving document priority.
    Docs whose title words match query entities are sorted first.
    Top-2 sentences extracted per doc, then globally re-scored.
    """
    if not docs:
        return ""

    query_lower = query.lower()

    def doc_priority(doc: str) -> int:
        title = re.split(r'[:\.]', doc)[0].strip().lower()
        title_words = [w for w in title.split() if len(w) > 3]
        return -sum(1 for w in title_words if w in query_lower)

    docs_sorted = sorted(docs, key=doc_priority)

    per_doc_sents = []
    for doc in docs_sorted:
        sents = re.split(r'(?<=[.!?])\s+', doc.strip())
        sents = [s.strip() for s in sents if len(s.strip()) > 10]
        if not sents:
            continue
        try:
            d     = [query] + sents
            tfidf = TfidfVectorizer(stop_words="english").fit_transform(d)
            sc    = cosine_similarity(tfidf[0:1], tfidf[1:]).flatten()
            top2  = np.argsort(sc)[::-1][:2]
            per_doc_sents.extend([sents[i] for i in top2])
        except Exception:
            per_doc_sents.extend(sents[:2])

    if not per_doc_sents:
        return " ".join(docs)

    try:
        d2      = [query] + per_doc_sents
        tfidf2  = TfidfVectorizer(stop_words="english").fit_transform(d2)
        sc2     = cosine_similarity(tfidf2[0:1], tfidf2[1:]).flatten()
        top_idx = sorted(np.argsort(sc2)[::-1][:top_n].tolist())
        selected = [per_doc_sents[i] for i in top_idx]
    except Exception:
        selected = per_doc_sents[:top_n]

    return " ".join(selected)

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

# BGE-large — instruction-tuned for retrieval, not similarity.
# Outperforms MiniLM-L6 on passage retrieval by 4-8 points on BEIR benchmarks.
EMBED_MODEL = "BAAI/bge-large-en-v1.5"
embedder    = SentenceTransformer(EMBED_MODEL, device=DEVICE)

print(f"Encoding {len(corpus):,} chunks with {EMBED_MODEL}...")
corpus_embeddings = embedder.encode(
    corpus,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True,    # enables cosine similarity via inner product
    convert_to_numpy=True,
).astype("float32")

dim   = corpus_embeddings.shape[1]
index = faiss.IndexIDMap(faiss.IndexFlatIP(dim))
index.add_with_ids(corpus_embeddings, np.arange(len(corpus)))
print(f"FAISS index ready: {index.ntotal:,} chunk vectors  (dim={dim})")


def retrieve_chunks_deduplicated(
    query:        str,
    top_k_chunks: int = TOP_K_CHUNKS,
    top_k_docs:   int = TOP_K_DOCS,
) -> Tuple[List[str], List[str]]:
    """
    Retrieve relevant documents via chunked FAISS with deduplication.

    Why deduplication matters
    ─────────────────────────
    Without it, the reranker often receives 5+ chunks from the same long
    Wikipedia article, wasting its budget. Deduplication ensures it sees
    up to top_k_docs *unique* articles, which is critical for multi-hop
    questions that require evidence from at least two different sources.

    Algorithm
    ─────────
    1. FAISS search → top_k_chunks raw chunk hits
    2. For each unique document, keep only the highest-scoring chunk
    3. Sort deduplicated docs by their best chunk score
    4. Return up to top_k_docs document texts and their doc_ids

    Returns
    -------
    texts   : List[str]   — one representative chunk per unique document
    doc_ids : List[str]   — corresponding document IDs (titles)
    """
    q_emb = embedder.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, ids = index.search(q_emb, top_k_chunks)
    scores, ids = scores[0], ids[0]

    # Best-scoring chunk per unique document
    best: Dict[str, Tuple[float, str]] = {}
    for score, idx in zip(scores, ids):
        if idx < 0:
            continue
        meta   = chunk_meta[int(idx)]
        doc_id = meta["doc_id"]
        if doc_id not in best or score > best[doc_id][0]:
            best[doc_id] = (score, corpus[int(idx)])

    ranked = sorted(best.items(), key=lambda x: -x[1][0])[:top_k_docs]
    doc_ids = [item[0]         for item in ranked]
    texts   = [item[1][1]      for item in ranked]
    return texts, doc_ids

In [ ]:
from rank_bm25 import BM25Okapi

# BM25 indexes the first (title-prefixed) chunk per document — this preserves
# exact-term matching for entity names, which chunks preserve well.
bm25_corpus_ids: List[str] = list(corpus_map.keys())
bm25_texts:      List[str] = [corpus_map[doc_id] for doc_id in bm25_corpus_ids]

tokenized_corpus = [text.lower().split() for text in bm25_texts]
bm25 = BM25Okapi(tokenized_corpus)

print(f"BM25 index ready: {len(bm25_corpus_ids):,} documents")

In [ ]:
def hybrid_retrieve(
    query:      str,
    top_k_docs: int   = TOP_K_DOCS,
    alpha:      float = HYBRID_ALPHA,
) -> List[str]:
    """
    Fuse deduplicated dense retrieval with BM25 using rank-based normalisation.

    Both score lists are normalised to [0,1] before fusion so neither branch
    dominates due to scale differences:
        fused = alpha * dense_rank_score + (1 - alpha) * bm25_rank_score

    Returns a list of document-level text strings (one per unique document).
    """
    # ── Dense branch ─────────────────────────────────────────────────────────
    dense_texts, dense_doc_ids = retrieve_chunks_deduplicated(
        query, top_k_chunks=TOP_K_CHUNKS, top_k_docs=top_k_docs * 2,
    )

    # Rank-based dense scores: rank 0 → 1.0, rank N → ~0
    dense_scores: Dict[str, float] = {
        doc_id: 1.0 - rank / max(len(dense_doc_ids), 1)
        for rank, doc_id in enumerate(dense_doc_ids)
    }
    dense_text_map: Dict[str, str] = dict(zip(dense_doc_ids, dense_texts))

    # ── Sparse branch (BM25) ─────────────────────────────────────────────────
    tokenized_query  = query.lower().split()
    bm25_scores_raw  = bm25.get_scores(tokenized_query)
    bm25_max         = max(bm25_scores_raw) if bm25_scores_raw.max() > 0 else 1.0
    bm25_scores_norm = bm25_scores_raw / bm25_max

    top_bm25_idx = np.argsort(bm25_scores_raw)[::-1][: top_k_docs * 2]
    bm25_doc_ids = [bm25_corpus_ids[i] for i in top_bm25_idx]

    bm25_scores: Dict[str, float] = {
        doc_id: float(bm25_scores_norm[top_bm25_idx[rank]])
        for rank, doc_id in enumerate(bm25_doc_ids)
    }

    # ── Fusion ────────────────────────────────────────────────────────────────
    all_doc_ids = set(dense_doc_ids) | set(bm25_doc_ids)
    fused: Dict[str, float] = {}
    for doc_id in all_doc_ids:
        d = dense_scores.get(doc_id, 0.0)
        b = bm25_scores.get(doc_id, 0.0)
        fused[doc_id] = alpha * d + (1.0 - alpha) * b

    ranked_ids = sorted(fused, key=lambda x: -fused[x])[:top_k_docs]

    # Return representative chunk text for each document
    result = []
    for doc_id in ranked_ids:
        if doc_id in dense_text_map:
            result.append(dense_text_map[doc_id])
        else:
            result.append(corpus_map.get(doc_id, ""))

    return [r for r in result if r]

In [ ]:
def retrieval_quality_score(query: str, docs: List[str]) -> float:
    """
    Mean cosine similarity between the query embedding and the top-3 docs.
    Used by RetrievalQualityGateAgent to decide whether to force Wikipedia fallback.
    """
    if not docs:
        return 0.0
    try:
        q_emb = embedder.encode(
            [query], normalize_embeddings=True, convert_to_numpy=True
        ).astype("float32")
        d_emb = embedder.encode(
            docs[:3], normalize_embeddings=True, convert_to_numpy=True
        ).astype("float32")
        sims = np.dot(d_emb, q_emb.T).flatten()
        return float(np.mean(np.sort(sims)[-min(3, len(sims)):]))
    except Exception:
        return 0.0

print("retrieval_quality_score defined.")

In [ ]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_MODEL, device=DEVICE)
print(f"Cross-encoder loaded: {RERANKER_MODEL}")


def rerank_docs(query: str, docs: List[str], top_k: int = TOP_K_RERANK) -> List[str]:
    """
    Score each (query, doc) pair with the cross-encoder and return top_k.
    More expensive than embedding similarity but far more accurate for
    relevance judgement — worth the cost at this final filtering stage.
    """
    if len(docs) <= top_k:
        return docs
    pairs  = [[query, doc] for doc in docs]
    scores = reranker.predict(pairs)
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [docs[i] for i in top_idx]

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} in 4-bit NF4...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Mistral-7B loaded.")

In [ ]:
def decompose_query(question: str) -> List[str]:
    """
    Decompose a multi-hop question into 2 sub-questions.
    Uses sampling so the model generates varied sub-questions rather than
    echoing the input. Falls back to entity/clause splitting on failure.
    """
    prompt = f"""[INST] Decompose the following multi-hop question into exactly 2 simpler sub-questions.
Keep each sub-question under 15 words. Each must be different from the original question.

Examples:
Q: What nationality is the director of the film Titanic?
Sub-question 1: Who directed the film Titanic?
Sub-question 2: What is the nationality of James Cameron?

Q: Which mountain is taller, Everest or K2?
Sub-question 1: What is the height of Mount Everest?
Sub-question 2: What is the height of K2?

Q: Were Scott Derrickson and Ed Wood of the same nationality?
Sub-question 1: What is the nationality of Scott Derrickson?
Sub-question 2: What is the nationality of Ed Wood?

Q: {question}
Sub-question 1: [/INST]"""

    inputs    = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 40,
            do_sample          = True,
            temperature        = 0.3,
            top_p              = 0.9,
            repetition_penalty = 1.2,
            pad_token_id       = tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][input_len:]
    generated  = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    generated  = re.sub(r"^Sub-question\s*\d+:\s*", "", generated).strip()
    generated  = "Sub-question 1: " + generated

    sub_qs = []
    for line in generated.split("\n"):
        line = line.strip()
        if re.match(r"Sub-question \d+:", line):
            sq = re.sub(r"^Sub-question \d+:\s*", "", line).strip()
            for stop in ["[INST]", "[/INST]", "INST"]:
                if stop in sq:
                    sq = sq[:sq.index(stop)].strip()
            # Double-space truncation
            if "  " in sq:
                sq = sq[:sq.index("  ")].strip()
            if not sq or len(sq) < 10:
                continue
            overlap = len(set(sq.lower().split()) & set(question.lower().split()))
            if overlap / max(len(question.split()), 1) <= 0.6:
                sub_qs.append(sq)

    sub_qs = list(dict.fromkeys(sub_qs))[:2]

    # ── Fallback ──────────────────────────────────────────────────────────────
    if len(sub_qs) < 2:
        q       = question.strip()
        q_lower = q.lower()

        # Multi-word entities only — avoids "What is Rock?", "What is Wind?"
        entities = re.findall(r"[A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+)+", q)

        if len(entities) >= 2:
            if any(w in q_lower for w in ["located", "neighborhood", "city", "where"]):
                sub_qs = [f"Where is {entities[0]} located?",
                          f"Where is {entities[1]} located?"]
            elif any(w in q_lower for w in ["nationality", "born", "from", "origin"]):
                sub_qs = [f"What is the nationality of {entities[0]}?",
                          f"What is the nationality of {entities[1]}?"]
            elif any(w in q_lower for w in ["managed", "manager", "recruited"]):
                sub_qs = [f"Who managed {entities[0]}?",
                          f"When did they manage {entities[0]}?"]
            elif any(w in q_lower for w in ["directed", "director"]):
                sub_qs = [f"Who directed {entities[0]}?",
                          f"What is the nationality of the director of {entities[0]}?"]
            elif any(w in q_lower for w in ["stage name", "known as", "nickname"]):
                sub_qs = [f"Who is {entities[0]}?",
                          f"What is the stage name of {entities[0]}?"]
            else:
                sub_qs = [f"What is {entities[0]}?", f"What is {entities[1]}?"]
        elif len(entities) == 1:
            sub_qs = [f"What is {entities[0]}?",
                      f"What additional information is needed: {q[:50]}?"]
        else:
            parts = re.split(
                r"\s+(?:and|who also|as well as|while|whereas)\s+"
                r"(?=\w+(?:ed|ing|is|was|were|has|have)\b)",
                q, flags=re.IGNORECASE
            )
            if len(parts) >= 2:
                sub_qs = [parts[0].strip(), parts[1].strip()]
            else:
                words = q.split(); mid = len(words)//2
                sub_qs = [" ".join(words[:mid])+"?", " ".join(words[mid:])+"?"]

    return sub_qs[:2]


def clean_answer(text: str) -> str:
    """
    Normalise a raw Mistral output into a clean answer string.
    """
    if not text:
        return ""

    patterns = [
        r".*?\[/INST\]\s*",
        r"^(the answer is|answer is|answer:|it is|this is|that is|so,?|therefore,?)\s*",
        r"^(based on|according to|from the context)[^,]*,\s*",
    ]
    for pat in patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE | re.DOTALL).strip()

    # ── Double-space truncation ────────────────────────────────────────────────
    if "  " in text:
        text = text[:text.index("  ")].strip()

    # ── Newline truncation ─────────────────────────────────────────────────────
    if "\n" in text:
        text = text[:text.index("\n")].strip()

    # ── Year range preservation — extract before sentence split ───────────────
    year_range = re.search(r"(\d{4})\s*[-–to]+\s*(\d{4})", text)

    # ── Sentence split ────────────────────────────────────────────────────────
    sentences = re.split(r"[.!?]\s{1,2}(?=[A-Z])", text)
    first = sentences[0].strip() if sentences else text.strip()

    # ── Preserve year range if split cut it ───────────────────────────────────
    if year_range and year_range.group(0) not in first:
        first = text[:year_range.end()].strip()

    # ── Strip qualifiers after year ranges e.g. "1986-2013, with two gaps" ────
    first = re.sub(
        r"(\d{4}\s*[-–to]+\s*\d{4})[,\s].*$",
        lambda m: m.group(0)[:m.end(1)],
        first
    ).strip()

    # ── Trailing punctuation cleanup ──────────────────────────────────────────
    first = re.sub(r"[.!?]+$", "", first).strip()
    first = re.sub(r"\s*\([^)]*\)", "", first).strip()

    # ── Trailing continuation strip ───────────────────────────────────────────
    first = re.sub(
        r",\s*(they|he|she|it|this|that|which|who|both|all|the)\b.*$",
        "", first, flags=re.IGNORECASE
    ).strip()
    first = re.sub(r",\s+[A-Z][a-zA-Z].*$", "", first).strip()
    first = re.sub(
        r"\s+(because|since|as|although|however|but|and the)\b.*$",
        "", first, flags=re.IGNORECASE
    ).strip()

    # ── Artefact cleanup ──────────────────────────────────────────────────────
    first = re.sub(r"(INST)+.*$", "", first, flags=re.IGNORECASE).strip()
    first = re.sub(r"\s*[\[\]{}]+.*$", "", first).strip()
    first = re.sub(r"\d{5,}.*$", "", first).strip()

    clean_match = re.match(r"^[a-zA-ZÀ-ÿ0-9\s\.\-\'\,–]+", first)
    first = clean_match.group(0).strip() if clean_match else ""

    if len(first) <= 1:
        return ""

    lower = first.lower()
    if lower in {"yes", "true", "correct", "affirmative"}:
        return "yes"
    if lower in {"no", "false", "incorrect", "negative"}:
        return "no"

    # ── Normalise bare year ranges → "from X to Y" ────────────────────────────
    first = re.sub(
        r"^(\d{4})\s*[-–]\s*(\d{4})$",
        r"from \1 to \2",
        first
    )

    return first


# ── Question-type detection ───────────────────────────────────────────────────
_BOOL_WORDS   = re.compile(r"\b(is|are|was|were|do|does|did|has|have|can|could|will|would)\b", re.I)
_NUMBER_WORDS = re.compile(r"\b(how many|how much|how old|how long|how far|how tall|how high|what year|what number|what age)\b", re.I)
_DATE_WORDS   = re.compile(r"\b(when|what date|what year|what month|what day)\b", re.I)
_PERSON_WORDS = re.compile(r"\b(who|whose|whom)\b", re.I)
_PLACE_WORDS  = re.compile(r"\b(where|which (city|country|place|location|state|region))\b", re.I)
_COMP_WORDS   = re.compile(r"\b(which (is|was|were|are|has|have)|compare|between|more|less|longer|shorter|older|younger|higher|lower|larger|smaller|bigger)\b", re.I)

def detect_question_type(question: str) -> str:
    q = question.strip()
    if _BOOL_WORDS.match(q):      return "boolean"
    if _NUMBER_WORDS.search(q):   return "number"
    if _DATE_WORDS.search(q):     return "date"
    if _PERSON_WORDS.search(q):   return "person"
    if _PLACE_WORDS.search(q):    return "place"
    if _COMP_WORDS.search(q):     return "comparison"
    return "general"


FEW_SHOT = {
    "boolean": """\
Context: The Eiffel Tower was built in 1889 in Paris, France.
Question: Was the Eiffel Tower built before 1900?
Answer: yes

Context: Marie Curie was born in Warsaw, Poland.
Question: Was Marie Curie born in France?
Answer: no""",

    "person": """\
Context: Steven Spielberg directed Schindler's List, which won seven Academy Awards.
Question: Who directed Schindler's List?
Answer: Steven Spielberg

Context: Alan Turing developed the concept of the Turing machine.
Question: Who developed the concept of the Turing machine?
Answer: Alan Turing""",

    "place": """\
Context: The Amazon River flows through Brazil and is the largest river by discharge.
Question: Which country does the Amazon River flow through?
Answer: Brazil

Context: Mount Fuji is an active stratovolcano located on Honshu Island in Japan.
Question: Where is Mount Fuji located?
Answer: Japan""",

    "number": """\
Context: The Great Wall of China stretches approximately 13,170 miles in total length.
Question: How long is the Great Wall of China?
Answer: 13,170 miles

Context: The human brain contains approximately 86 billion neurons.
Question: How many neurons does the human brain have?
Answer: 86 billion""",

    "date": """\
Context: World War II ended on September 2, 1945, with Japan's formal surrender.
Question: When did World War II end?
Answer: September 2, 1945

Context: The Apollo 11 mission landed on the Moon on July 20, 1969.
Question: When did Apollo 11 land on the Moon?
Answer: July 20, 1969""",

    "comparison": """\
Context: Mount Everest stands at 8,849 m. K2 stands at 8,611 m.
Question: Which mountain is taller, Everest or K2?
Answer: Mount Everest

Context: The Nile (6,650 km) and the Amazon (6,400 km) are the two longest rivers.
Question: Which river is longer, the Nile or the Amazon?
Answer: the Nile""",

    "general": """\
Context: Photosynthesis is the process by which plants convert sunlight into glucose.
Question: What is photosynthesis?
Answer: the process by which plants convert sunlight into glucose

Context: The speed of light in a vacuum is approximately 299,792 kilometres per second.
Question: What is the speed of light?
Answer: 299,792 kilometres per second""",
}


def generate_answer(question: str, context: str) -> str:
    """
    Generate a single answer using Mistral-7B-Instruct.
    Boolean questions get strict yes/no instruction.
    Date/number questions get copy-exact instruction.
    """
    q_type      = detect_question_type(question)
    few_shot    = FEW_SHOT.get(q_type, FEW_SHOT["general"])
    ctx_trimmed = context[:1500]

    if q_type == "boolean":
        instruction = (
            "Answer with ONLY 'yes' or 'no'. No other words."
        )
    elif q_type in ("date", "number"):
        instruction = (
            "Answer with ONLY the specific date, number, or year range asked. "
            "Copy it exactly as it appears in the context. No other words."
        )
    else:
        instruction = (
            "Give a SHORT, DIRECT answer — a single entity, date, number, yes/no, or brief phrase. "
            "Copy the exact name or value from the context. Do not paraphrase. "
            "Do NOT write a full sentence. Stop immediately after the answer."
        )

    prompt = f"""[INST] Answer the question using ONLY the provided context.
{instruction}

{few_shot}

Context: {ctx_trimmed}
Question: {question}
Answer: [/INST]"""

    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=2048
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = MAX_NEW_TOKENS,
            temperature        = TEMPERATURE,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
            repetition_penalty = 1.1,
        )

    full        = tokenizer.decode(outputs[0], skip_special_tokens=True)
    prompt_text = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)
    answer_part = full[len(prompt_text):].strip()

    # Truncate at stop sequences
    for stop in ["[INST]", "[/INST]", "\nContext:", "\nQuestion:", "\nAnswer:", "\n\n"]:
        if stop in answer_part:
            answer_part = answer_part[:answer_part.index(stop)].strip()

    return clean_answer(answer_part) if answer_part else clean_answer(full)


def pick_best_answer(candidates: List[str], context: str) -> str:
    """
    Select the best answer — boolean majority → strict majority → BGE tie-break.
    """
    from collections import Counter

    if not candidates:
        return ""

    cleaned = [clean_answer(c) for c in candidates if c.strip()]
    cleaned = [c for c in cleaned if c]  # remove empty strings after clean_answer
    if not cleaned:
        return ""

    # ── Boolean strict majority vote ───────────────────────────────────────────
    bool_answers = [c.lower() for c in cleaned if c.lower() in ("yes", "no")]
    if len(bool_answers) >= len(cleaned) // 2 + 1:
        yes_count = bool_answers.count("yes")
        no_count  = bool_answers.count("no")
        if yes_count != no_count:
            return "yes" if yes_count > no_count else "no"

    counts = Counter(cleaned)
    if not counts:
        return ""
    winner, win_count = counts.most_common(1)[0]

    # ── Strict majority — prefer shorter core if longer is noise ──────────────
    if win_count >= 3:
        shorter = [c for c in counts
                   if c != winner and winner.lower().startswith(c.lower())
                   and len(c.split()) >= 1]
        if shorter:
            return max(shorter, key=len)
        return winner

    if win_count >= 2:
        return winner

    def common_prefix(strings):
        if not strings:
            return ""
        word_lists = [s.split() for s in strings]
        prefix_words = []
        for group in zip(*word_lists):
            base = re.sub(r"[^a-zA-ZÀ-ÿ0-9]", "", group[0]).lower()
            if all(re.sub(r"[^a-zA-ZÀ-ÿ0-9]", "", w).lower() == base
                   for w in group):
                prefix_words.append(group[0])
            else:
                break
        return " ".join(prefix_words).strip()

    prefix = common_prefix(list(counts.keys()))
    if prefix and len(prefix.split()) >= 2:
        extensions = [c for c in counts if c.lower().startswith(prefix.lower())]
        if extensions:
            return max(extensions, key=len)
        return prefix

    # ── BGE embedding tie-break (independent of Mistral-7B and DeBERTa) ───────
    best_cand, best_score = winner, -1.0
    try:
        cand_embs = embedder.encode(
            list(counts.keys()), normalize_embeddings=True, convert_to_numpy=True
        ).astype("float32")
        ctx_emb = embedder.encode(
            [context[:512]], normalize_embeddings=True, convert_to_numpy=True
        ).astype("float32")
        scores    = np.dot(cand_embs, ctx_emb.T).flatten()
        best_idx  = int(np.argmax(scores))
        best_cand = list(counts.keys())[best_idx]
        best_score = float(scores[best_idx])
    except Exception:
        pass

    if best_score < 0.1:
        grounded  = [c for c in counts if c.lower() in context.lower()]
        best_cand = max(grounded, key=len) if grounded else max(counts, key=len)
    else:
        grounded_all = [c for c in counts if c.lower() in context.lower()]
        if len(grounded_all) > 1:
            grounded_all.sort(key=lambda x: (counts.get(x, 0), -len(x)), reverse=True)
            best_cand = grounded_all[0]

    return best_cand

In [ ]:
def post_process_answer(answer: str) -> str:
    """
    Final normalisation — strips artefacts, verbose continuations,
    leading articles, and newline continuations.
    Consistent with clean_answer to avoid conflicts.
    """
    answer = answer.strip()

    for pat in [
        r"^(the answer is|it is|this is|that is|answer:)\s*",
        r"^(based on the context[,.]?\s*)",
        r"<\|eot_id\|>.*$",
        r"<\|start_header_id\|>.*$",
        r"\.$",
    ]:
        answer = re.sub(pat, "", answer, flags=re.IGNORECASE).strip()

    # Strip trailing prompt-echo words
    answer = re.sub(
        r"\s+(Answer|Context|Question|INST|Inst)[\w]*$",
        "", answer, flags=re.IGNORECASE
    ).strip()

    # ── Strip at first newline ─────────────────────────────────────────────────
    if "\n" in answer:
        answer = answer[:answer.index("\n")].strip()

    # ── Strip verbose continuations after the core answer ─────────────────────
    answer = re.sub(
        r",\s*(they|he|she|it|this|that|which|who|both|all)\b.*$",
        "", answer, flags=re.IGNORECASE
    ).strip()
    answer = re.sub(
        r"\s+(because|since|as|although|however|but|and the)\b.*$",
        "", answer, flags=re.IGNORECASE
    ).strip()

    # ── Strip leading articles to match gold label format ─────────────────────
    answer = re.sub(r"^(the|a|an)\s+", "", answer, flags=re.IGNORECASE).strip()

    # Strip leading/trailing non-alpha
    answer = re.sub(r"^[^a-zA-Z0-9]+", "", answer).strip()
    answer = re.sub(r"[^a-zA-Z0-9.\'\'\-\s]+$", "", answer).strip()

    return answer


def check_answer_grounded(answer: str, context: str,
                           threshold: float = GROUNDING_THRESHOLD) -> bool:
    answer_clean  = answer.strip().lower()
    context_clean = context.lower()
    if answer_clean in {"yes", "no"}:
        return True
    if len(answer_clean.split()) <= 2 and answer_clean in context_clean:
        return True
    try:
        emb_a = embedder.encode(
            [answer_clean], normalize_embeddings=True, convert_to_numpy=True
        ).astype("float32")
        emb_c = embedder.encode(
            [context_clean[:512]], normalize_embeddings=True, convert_to_numpy=True
        ).astype("float32")
        return float(np.dot(emb_a, emb_c.T)) >= threshold
    except Exception:
        return True


def generate_answer_safe(question: str, context: str, max_retries: int = 2) -> str:
    """Wrapper with grounding-check retries."""
    ctx = context
    for attempt in range(max_retries + 1):
        candidates = [generate_answer(question, ctx) for _ in range(N_SAMPLES)]
        answer     = pick_best_answer(candidates, ctx)
        answer     = post_process_answer(answer)
        if check_answer_grounded(answer, context):
            return answer
        if attempt < max_retries:
            ctx = ("IMPORTANT: Answer ONLY using the evidence below. "
                   "Do not add any outside knowledge.\n\n" + context)
    return answer


In [ ]:
# Independent Verification Model ───────────────────────────────────
# (a) verify_chain()     → cross-encoder/nli-deberta-v3-base (DeBERTa family)
# (b) pick_best_answer() → BGE embedding cosine similarity (retrieval model)
#
# Model independence matrix:
#   Generation    : Mistral-7B-Instruct-v0.1 (local)
#   Candidate rank: BGE-large (embedding cosine)
#   Verification  : DeBERTa-NLI (cross-encoder)
#   Reranking     : MS-MARCO MiniLM cross-encoder

from transformers import AutoModelForSequenceClassification, AutoTokenizer as AT
from sentence_transformers import CrossEncoder

# Primary verifier — DeBERTa NLI (independent from Mistral + BGE)
print(f"Loading primary verifier: {PRIMARY_VERIFIER_MODEL}")
nli_tokenizer = AT.from_pretrained(PRIMARY_VERIFIER_MODEL)
nli_model     = AutoModelForSequenceClassification.from_pretrained(
    PRIMARY_VERIFIER_MODEL
).to(DEVICE).eval()
print(f"Primary verifier loaded: {PRIMARY_VERIFIER_MODEL}")


def verify(answer: str, context: str) -> dict:
    """
    Single-document NLI verification using DeBERTa.

    Inspired by the FEVER dataset (Thorne et al., 2018) three-label
    fact verification scheme — adapted for RAG answer verification:

      FEVER label        Our use         Meaning
      ─────────────────────────────────────────────────────────
      SUPPORTS         → entailment  →  answer supported by context
      REFUTES          → contradiction→  answer conflicts with context
      NOT ENOUGH INFO  → neutral     →  context insufficient to judge

    entailment ≥ NLI_THRESHOLD      → accept answer
    contradiction ≥ CONTRADICT_TRIGGER → trigger regeneration
    neutral dominant                → low confidence signal

    DeBERTa label order: contradiction=0, neutral=1, entailment=2
    """
    try:
        inputs = nli_tokenizer(
            context[:512], answer,
            return_tensors="pt",
            truncation=True,
            max_length=512,
        ).to(DEVICE)
        with torch.no_grad():
            logits = nli_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0].tolist()
        # DeBERTa MNLI label order: contradiction, neutral, entailment
        return {
            "contradiction": probs[0],
            "neutral":       probs[1],
            "entailment":    probs[2],
        }
    except Exception:
        return {"contradiction": 0.0, "neutral": 1.0, "entailment": 0.0}


def verify_chain(answer: str, docs: List[str]) -> dict:
    """
    Chain-of-verification using DeBERTa NLI across each retrieved document.
    Each document is verified independently to avoid dilution.

    FEVER-inspired verdict aggregation across the document chain:
      entailment_rate  : fraction of docs that SUPPORT the answer
      contradicted     : True if ANY doc REFUTES the answer (strict)
      max_entailment   : highest single-doc SUPPORTS score

    The contradiction check is deliberately strict — any single REFUTES
    verdict triggers regeneration, consistent with FEVER logic that a
    strong counter-evidence outweighs multiple weak supporting signals.
    """
    if not docs:
        return {"entailment_rate": 0.0, "contradicted": False,
                "max_entailment": 0.0}

    entailed_count = 0
    contradicted   = False
    max_entailment = 0.0

    for doc in docs:
        result = verify(answer, doc)
        if result["entailment"] >= NLI_THRESHOLD:
            entailed_count += 1
        if result["contradiction"] >= CONTRADICT_TRIGGER:
            contradicted = True
        max_entailment = max(max_entailment, result["entailment"])

    return {
        "entailment_rate": entailed_count / len(docs),
        "contradicted":    contradicted,
        "max_entailment":  max_entailment,
        "verifier_model":  PRIMARY_VERIFIER_MODEL,  # provenance tracking
    }


print("verify() and verify_chain() ready — using DeBERTa NLI.")
print(f"  Verifier family: deberta (independent from mistral + bge)")


In [ ]:
from difflib import SequenceMatcher


def normalise_for_exact(text: str) -> str:
    """
    Normalise answer surface form before exact match comparison.
    Handles the most common mismatches in HotPotQA:
    - Leading articles: "the minor basilica" → "minor basilica"
    - Yes/no variants: "yeah" → "yes"
    - Trailing punctuation: "Bob Seger." → "Bob Seger"
    - Extra whitespace
    """
    t = text.lower().strip()

    # Strip leading articles
    t = re.sub(r"^(the|a|an)\s+", "", t).strip()

    # Normalise yes/no variants
    if t in {"yes", "true", "correct", "yeah", "yep"}:
        return "yes"
    if t in {"no", "false", "incorrect", "nope"}:
        return "no"

    # Strip trailing punctuation
    t = re.sub(r"[.!?,;:]+$", "", t).strip()

    # Strip trailing parenthetical
    t = re.sub(r"\s*\([^)]*\).*$", "", t).strip()

    # Strip apposition
    t = re.sub(r",\s+(?:an?|the)\s+\w+.*$", "", t).strip()

    # Collapse multiple spaces
    t = re.sub(r"\s+", " ", t).strip()

    return t


def token_f1(pred: str, gold: str) -> float:
    pred_tokens = set(pred.lower().split())
    gold_tokens = set(gold.lower().split())
    if not pred_tokens or not gold_tokens:
        return 0.0
    common = pred_tokens & gold_tokens
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall    = len(common) / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def semantic_sim(pred: str, gold: str) -> float:
    try:
        embs = embedder.encode(
            [pred.lower(), gold.lower()],
            normalize_embeddings=True,
            convert_to_numpy=True,
        ).astype("float32")
        return float(np.dot(embs[0], embs[1]))
    except Exception:
        return 0.0


def char_sim(pred: str, gold: str) -> float:
    return SequenceMatcher(None, pred.lower(), gold.lower()).ratio()


def lcs_ratio(pred: str, gold: str) -> float:
    p, g = pred.lower().split(), gold.lower().split()
    if not p or not g:
        return 0.0
    m, n = len(p), len(g)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if p[i-1] == g[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[m][n] / max(m, n)


def containment(pred: str, gold: str) -> float:
    pred_tokens = set(pred.lower().split())
    gold_tokens = set(gold.lower().split())
    if not gold_tokens:
        return 0.0
    return len(pred_tokens & gold_tokens) / len(gold_tokens)


def compute_scores(pred: str, gold: str) -> dict:
    """
    Compute all five similarity metrics and their weighted ensemble.

    Phase 4 change: normalise_for_exact() is applied before exact match
    comparison so surface-form mismatches like leading articles
    ("the minor basilica" vs "a minor basilica") count as exact matches.
    approx_match now also fires when exact=1 (ensures consistency).
    """
    if not pred:
        return {
            "exact_match":   0, "approx_match":  0,
            "f1":            0.0, "ensemble_score": 0.0,
            "match_reason":  "none",
            "scores": {"token_f1": 0.0, "semantic": 0.0,
                       "char_sim": 0.0, "lcs": 0.0, "containment": 0.0},
        }

    # ── Exact match with normalisation ────────────────────────────────────────
    pred_n = normalise_for_exact(pred)
    gold_n = normalise_for_exact(gold)
    exact  = int(pred_n == gold_n)

    # ── Five metrics ──────────────────────────────────────────────────────────
    tf1  = token_f1(pred, gold)
    sem  = semantic_sim(pred, gold)
    cs   = char_sim(pred, gold)
    lcs  = lcs_ratio(pred, gold)
    cont = containment(pred, gold)

    ensemble = (
        SCORE_WEIGHTS["token_f1"]    * tf1  +
        SCORE_WEIGHTS["semantic"]    * sem  +
        SCORE_WEIGHTS["char_sim"]    * cs   +
        SCORE_WEIGHTS["lcs"]         * lcs  +
        SCORE_WEIGHTS["containment"] * cont
    )

    # approx fires on ensemble threshold alone — independent of exact
    # exact match does NOT force approx to tick; they are separate metrics
    # This gives a cleaner picture of how many answers are "close enough"
    # vs "exactly right" vs "roughly right"
    approx = int(ensemble >= APPROX_THRESHOLD)

    # ── Match reason ──────────────────────────────────────────────────────────
    if exact:
        reason = "exact"
    elif cont >= 0.8:
        reason = "containment"
    elif tf1 >= 0.5:
        reason = "token_f1"
    elif sem >= 0.65:
        reason = "semantic"
    elif lcs >= 0.5:
        reason = "lcs"
    elif ensemble >= APPROX_THRESHOLD:
        reason = "ensemble"
    else:
        reason = "no-match"

    return {
        "exact_match":    exact,
        "approx_match":   approx,
        "f1":             tf1,
        "ensemble_score": ensemble,
        "match_reason":   reason,
        "scores": {
            "token_f1":   tf1,  "semantic": sem,
            "char_sim":   cs,   "lcs":      lcs,
            "containment": cont,
        },
    }


In [ ]:
def compute_perplexity(text: str) -> float:
    """
    Perplexity computation disabled — no local model available with local Mistral model.
    ICA now uses BGE cosine similarity as a proxy instead.
    """
    return 0.0


def ica_score_documents(query: str, docs: List[str]) -> List[Tuple[str, float]]:
    """
    ICA approximation using BGE embedding similarity.

    Original PPL-based ICA required a local model forward pass which is
    unavailable with Mistral locally. BGE cosine similarity is a valid
    proxy — higher similarity = document is more informative for the query.

    This restores meaningful document ranking that was lost when ICA
    was returning score=0.0 for all documents.
    """
    if not docs:
        return []
    try:
        q_emb = embedder.encode(
            [query], normalize_embeddings=True, convert_to_numpy=True
        ).astype("float32")
        d_emb = embedder.encode(
            docs, normalize_embeddings=True, convert_to_numpy=True
        ).astype("float32")
        scores = np.dot(d_emb, q_emb.T).flatten()
        return [(doc, float(score)) for doc, score in zip(docs, scores)]
    except Exception:
        # Fallback: return docs in original order if embedder fails
        return [(doc, 0.0) for doc in docs]


In [ ]:
def filter_irrelevant_docs(question: str, docs: List[str],
                            min_docs: int = 3) -> List[str]:
    """
    NegTrain filter — disabled on A100.
    Cross-encoder reranker already handles relevance with higher accuracy.
    Kept as pass-through for compatibility; re-enable by replacing return.
    """
    return docs if docs else []


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# HALLUCINATION CLASSIFICATION — FEVER-INSPIRED THREE-LABEL SCHEME
#
# The FEVER dataset (Thorne et al., 2018) introduced a three-way annotation
# scheme for fact verification:
#   SUPPORTS         → evidence entails the claim
#   REFUTES          → evidence contradicts the claim
#   NOT ENOUGH INFO  → evidence is neutral / insufficient
#
# We adapt this directly to our RAG hallucination taxonomy:
#
#   FEVER label        →  Our label      →  Meaning in pipeline context
#   ─────────────────────────────────────────────────────────────────────
#   SUPPORTS           →  "none"         →  Answer grounded in context
#   REFUTES            →  "fabrication"  →  Answer not found in context
#   NOT ENOUGH INFO    →  "omission"     →  Model refused / hedged
#
# The three-label framework lets us distinguish two qualitatively different
# failure modes that a binary correct/incorrect metric would conflate:
#   - Fabrication: model invented something (active hallucination)
#   - Omission:    model admitted it didn't know (passive failure)
#
# This distinction matters for regeneration: fabrication triggers
# regeneration; omission signals a retrieval problem, not a generation one.
#
# 27 omission phrases from DePaC Appendix G (Shi et al., 2024)
# ═══════════════════════════════════════════════════════════════════════════════
OMISSION_PHRASES = [
    "the context does not", "the context doesn't", "the provided context",
    "no information", "not mentioned", "not provided", "not specified",
    "cannot be determined", "can't be determined", "cannot determine",
    "unable to determine", "not stated", "not given", "not available",
    "insufficient information", "no evidence", "no mention",
    "the passage does not", "the text does not", "the document does not",
    "there is no", "there's no", "i don't know", "i do not know",
    "unknown", "not found", "no relevant",
]


def classify_hallucination(answer: str, context: str) -> str:
    """
    Classify the answer into one of three labels inspired by FEVER:

    FEVER label        →  Our label      →  Detection method
    ─────────────────────────────────────────────────────────
    NOT ENOUGH INFO    →  "omission"     →  Empty answer OR DePaC omission phrase
    REFUTES            →  "fabrication"  →  Answer not found in retrieved context
    SUPPORTS           →  "none"         →  Answer grounded (word overlap or semantic)

    Steps (in order of precedence):
      0. Empty / near-empty answer          → omission
      1. DePaC Appendix G omission phrase   → omission  (27 phrases)
      2. >50% word overlap with context     → none       (grounded)
      3. BGE semantic similarity ≥ 0.4      → none       (semantically grounded)
      4. Otherwise                          → fabrication

    The fabrication / omission split allows targeted diagnosis:
      High fabrication → retrieval is finding wrong documents
      High omission    → model is over-refusing / context is too sparse
    """
    answer_lower  = answer.strip().lower()
    context_lower = context.lower()

    # ── Step 0: empty or near-empty answer = omission ─────────────────────
    # This was the root cause of omission always showing 0%:
    # the pipeline returned empty strings for failed generations
    # but classify_hallucination was only checking for omission phrases,
    # not empty strings — so they silently fell through to "fabrication".
    if not answer_lower or len(answer_lower) <= 1:
        return "omission"

    # ── Step 1: DePaC omission phrases ────────────────────────────────────
    for phrase in OMISSION_PHRASES:
        if phrase in answer_lower:
            return "omission"

    # Check if answer is grounded (appears in context)
    answer_words = set(answer_lower.split())
    context_words = set(context_lower.split())
    overlap = answer_words & context_words

    # If >50% of answer words appear in context → not hallucination
    if len(answer_words) > 0 and len(overlap) / len(answer_words) > 0.5:
        return "none"

    # Semantic check as secondary gate
    if semantic_sim(answer_lower, context_lower[:512]) >= 0.4:
        return "none"

    return "fabrication"

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LANGGRAPH STATE DEFINITION
#
# PipelineState is the single shared object that flows through every node
# in the LangGraph. Each node receives the full state, makes its changes,
# and returns a dict of updated keys.
#
# LangGraph merges the returned dict back into the state automatically —
# nodes never need to copy unchanged fields.
# ═══════════════════════════════════════════════════════════════════════════════

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langchain_core.messages import BaseMessage
import operator

class PipelineState(TypedDict, total=False):
    # ── Input ─────────────────────────────────────────────────────────────────
    question:          str
    gold_answer:       str
    verbose:           bool

    # ── Stage 1 — Query Decomposition ────────────────────────────────────────
    sub_questions:     List[str]

    # ── Stage 2+3 — Retrieval + Quality Gate ─────────────────────────────────
    retrieved_docs:    List[str]
    wiki_docs:         List[str]
    wiki_fallback:     bool
    rq_score:          float
    retrieval_source:  str           # "corpus" | "wikipedia" | "web"

    # ── Stage 4+5 — Reranking + GraphRAG ─────────────────────────────────────
    reranked_docs:     List[str]

    # ── Stage 6+7 — ICA + Sentence Selection ─────────────────────────────────
    relevant_docs:     List[str]
    ica_scores:        List[float]
    final_context:     str

    # ── Stage 8 — Answer Generation ──────────────────────────────────────────
    answer:            str
    raw_candidates:    List[str]

    # ── Stage 9 — Verification ────────────────────────────────────────────────
    verification:      dict          # {entailment_rate, contradicted, max_entailment}
    entailment_rate:   float
    contradicted:      bool
    confidence:        float

    # ── Stage 10 — Regeneration ───────────────────────────────────────────────
    regen_attempt:     int

    # ── HITL ─────────────────────────────────────────────────────────────────
    hitl_mode:         bool       # True when HITL is active for this run
    hitl_points:       List[str]  # checkpoint names to pause at
    hitl_session_id:   str        # unique run ID for session lookup
    hitl_checkpoints:  List[dict] # [{checkpoint, decision, auto}, ...]

    # ── Output ────────────────────────────────────────────────────────────────
    predicted_answer:  str
    hallucination_type:str
    scores:            dict
    timings:           dict


print("PipelineState TypedDict defined.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# HUMAN IN THE LOOP — SESSION MANAGEMENT
#
# Uses threading.Event (NOT asyncio.Event) so the pipeline thread and the
# FastAPI request thread can communicate reliably across event loops.
#
# Root cause of previous hang:
#   asyncio.Event is bound to the event loop it was created in.
#   run_pipeline_hitl() runs in a ThreadPoolExecutor → new event loop.
#   FastAPI /hitl/decide runs in its own event loop.
#   sess.event.set() from FastAPI never woke up sess.event.wait() in the
#   pipeline thread because they were in different event loops.
#
# Fix: threading.Event works across threads with no event loop dependency.
# ═══════════════════════════════════════════════════════════════════════════════

import threading, uuid

# In-process session store — keyed by session_id
hitl_sessions: dict = {}


class HITLSession:
    """
    Manages a single HITL pause point using threading.Event.
    Works reliably across threads and asyncio event loops.
    """
    def __init__(self, session_id: str, checkpoint: str, data: dict):
        self.session_id = session_id
        self.checkpoint = checkpoint
        self.data       = data
        self.event      = threading.Event()   # ← threading not asyncio
        self.decision   = None

    def resolve(self, action: str, payload: dict = {}):
        """Called by FastAPI /hitl/decide — unblocks the pipeline thread."""
        self.decision = {"action": action, **payload}
        self.event.set()

    def wait(self, timeout: float = 120.0) -> dict:
        """
        Blocking wait — runs in the pipeline thread.
        Auto-approves after timeout seconds.
        """
        fired = self.event.wait(timeout=timeout)
        if not fired:
            self.decision = {"action": "approve", "auto": True}
        return self.decision


def hitl_checkpoint(
    session_id: str,
    checkpoint: str,
    data:       dict,
    timeout:    float = 120.0,
) -> dict:
    """
    Synchronous checkpoint — register session, block until human decides.
    No async/await needed since threading.Event.wait() is a plain blocking call.
    """
    sess = HITLSession(session_id, checkpoint, data)
    hitl_sessions[session_id] = sess
    decision = sess.wait(timeout=timeout)
    hitl_sessions.pop(session_id, None)
    return decision


print("HITL session management ready (threading.Event).")
print("  hitl_sessions dict registered for FastAPI backend access.")


In [ ]:
# ── Live progress callback ────────────────────────────────────────────────────
def _cb(name: str, elapsed: float, info: dict = None):
    """Notify Flask /progress endpoint that an agent has completed."""
    try:
        import sys as _sys
        _fn = getattr(_sys.modules.get("pipeline_globals"), "ON_AGENT_DONE", None)
        if callable(_fn):
            _fn(name, elapsed, info or {})
    except Exception:
        pass


# ═══════════════════════════════════════════════════════════════════════════════
# LANGGRAPH NODE FUNCTIONS
#
# Each function:
#   - Takes the full PipelineState
#   - Returns a dict of ONLY the keys it modifies
#   - Wraps its core logic in latency_tracker.track()
#   - Calls the corresponding Phase 4 Agent for business logic
#
# Node routing:
#   query_decomposition_node
#           ↓
#   hybrid_retrieval_node
#           ↓
#   quality_gate_node ──────────────────→ [route_quality_gate]
#           ↓ (strong retrieval)                  ↓ (weak → fallback)
#   reranking_node  ←──────────────── wikipedia_fallback_node
#           ↓
#   graphrag_node
#           ↓
#   ica_scoring_node
#           ↓
#   sentence_selection_node
#           ↓
#   answer_generation_node
#           ↓
#   verification_node ──────────────────→ [route_verification]
#           ↓ (confident)                         ↓ (uncertain/contradicted)
#   finalise_node   ←──────────────── regeneration_node
# ═══════════════════════════════════════════════════════════════════════════════


# ── Node 1: Query Decomposition ───────────────────────────────────────────────
def query_decomposition_node(state: PipelineState) -> dict:
    """Decomposes multi-hop question into 2 sub-questions."""
    with latency_tracker.track("QueryDecompositionAgent"):
        sub_qs = decompose_query(state["question"])

    if state.get("verbose"):
        _vprint("① QueryDecompositionAgent  [mistral-7b]")
        for j, sq in enumerate(sub_qs, 1):
            _vprint(f"   Sub-question {j}:", sq)
        _vsep()

    _cb("QueryDecompositionAgent",
        latency_tracker._current.get("QueryDecompositionAgent", 0),
        {"sub_questions": sub_qs})
    return {"sub_questions": sub_qs}


# ── Node 2: Hybrid Retrieval ──────────────────────────────────────────────────
def hybrid_retrieval_node(state: PipelineState) -> dict:
    """Retrieves candidate documents via FAISS + BM25 fusion."""
    with latency_tracker.track("HybridRetrievalAgent"):
        docs = hybrid_retrieve(state["question"], top_k_docs=TOP_K_DOCS)
    _cb("HybridRetrievalAgent",
        latency_tracker._current.get("HybridRetrievalAgent", 0),
        {"doc_count": len(docs)})
    return {"retrieved_docs": docs}


# ── Node 3: Quality Gate ──────────────────────────────────────────────────────
def quality_gate_node(state: PipelineState) -> dict:
    """
    Scores retrieval quality. Implements full real-time retrieval cascade.

    Source cascade (automatic, priority order):
      ① FAISS + BM25 (corpus)  — always runs, scored by BGE cosine
      ② Wikipedia API          — always fetched as supplementary source
      ③ Web search             — when rq_score < threshold AND wiki < 300 chars
                                  Provider cascade: Tavily → DuckDuckGo (free)

    Decision logic:
      rq_score >= threshold    → corpus is sufficient, proceed to reranking
      rq_score <  threshold    → wikipedia_fallback_node (augmented with web)

    Retrieval source reported to frontend for explainability badge display.
    """
    q      = state["question"]
    sub_qs = state.get("sub_questions", [q])

    with latency_tracker.track("RetrievalQualityGateAgent"):

        # ── Score corpus quality ───────────────────────────────────────────────
        rq_score = retrieval_quality_score(q, state["retrieved_docs"])

        # ── Wikipedia: always fetch for all sub-questions ─────────────────────
        wiki_docs = []
        for sq in sub_qs:
            w = wiki_search(sq)
            if w:
                wiki_docs.append(w)

        # ── Web search: low corpus quality + thin Wikipedia ───────────────────
        web_docs = []
        web_used = False
        if rq_score < RETRIEVAL_QUALITY_THRESHOLD and WEB_RETRIEVAL_ENABLED:
            wiki_total = sum(len(d) for d in wiki_docs)
            if wiki_total < WEB_MIN_WIKI_CHARS:
                for sq in sub_qs:
                    snippets = web_search(sq, max_results=WEB_MAX_RESULTS)
                    web_docs.extend(snippets)
                web_used = len(web_docs) > 0
                if web_used and state.get("verbose"):
                    provider = "Tavily" if TAVILY_API_KEY else "DuckDuckGo"
                    _vprint("   Web provider:", provider)


    # ── Determine retrieval source label ──────────────────────────────────────
    if web_used:
        retrieval_source = "web"
    elif wiki_docs:
        retrieval_source = "wikipedia"
    else:
        retrieval_source = "corpus"

    if state.get("verbose"):
        _vprint("③ RetrievalQualityGateAgent  [bge]")
        _vprint("   Retrieval quality score:",
                f"{rq_score:.3f}  (threshold={RETRIEVAL_QUALITY_THRESHOLD})")
        _vprint("   Wiki docs fetched:",    str(len(wiki_docs)))
        _vprint("   Web snippets:",         str(len(web_docs)) +
                (" (Tavily)" if TAVILY_API_KEY and web_used else
                 " (DDG)"    if web_used       else " (not triggered)"))
        _vprint("   Retrieval source:",     retrieval_source)
        _vsep()

    _cb("RetrievalQualityGateAgent",
        latency_tracker._current.get("RetrievalQualityGateAgent", 0),
        {"quality_score": round(rq_score, 3),
         "wiki_needed":   rq_score < RETRIEVAL_QUALITY_THRESHOLD,
         "retrieval_source": retrieval_source})
    return {
        "rq_score":         rq_score,
        "wiki_docs":        wiki_docs + web_docs,
        "web_docs":         web_docs,
        "wiki_fallback":    rq_score < RETRIEVAL_QUALITY_THRESHOLD,
        "retrieval_source": retrieval_source,
    }

# ── Node 3a: Wikipedia Fallback ───────────────────────────────────────────────
def wikipedia_fallback_node(state: PipelineState) -> dict:
    """
    Fires when retrieval quality is weak.
    Prepends Wikipedia docs to retrieved_docs.
    """
    wiki_docs      = state.get("wiki_docs", [])
    retrieved_docs = wiki_docs + state.get("retrieved_docs", [])[:2]

    if state.get("verbose"):
        _vprint("③a wikipedia_fallback_node  — weak retrieval, using Wikipedia")
        _vprint("   Docs after fallback:", str(len(retrieved_docs)))
        _vsep()

    return {
        "retrieved_docs": retrieved_docs,
        "wiki_fallback":  True,
    }


# ── Router: quality gate ──────────────────────────────────────────────────────
def route_quality_gate(state: PipelineState) -> str:
    """
    Conditional edge after quality_gate_node.
    Routes to wikipedia_fallback_node if retrieval is weak,
    otherwise directly to reranking_node.
    """
    if state.get("rq_score", 1.0) < RETRIEVAL_QUALITY_THRESHOLD:
        return "wikipedia_fallback"
    return "reranking"


# ── Node 4: Reranking ─────────────────────────────────────────────────────────
def reranking_node(state: PipelineState) -> dict:
    """Reranks candidates with MS-MARCO cross-encoder."""
    with latency_tracker.track("RerankingAgent"):
        all_candidates = (state.get("retrieved_docs", []) +
                          state.get("wiki_docs", []))
        seen_texts, deduped = set(), []
        for doc in all_candidates:
            key = doc[:100]
            if key not in seen_texts:
                seen_texts.add(key)
                deduped.append(doc)
        reranked = rerank_docs(
            state["question"], deduped, top_k=TOP_K_RERANK
        )

    if state.get("verbose"):
        _vprint("④ RerankingAgent  [cross_encoder]")
        _vprint("   Docs after rerank:", str(len(reranked)))
        _vsep()

    _cb("RerankingAgent",
        latency_tracker._current.get("RerankingAgent", 0),
        {"doc_count": len(reranked)})
    return {"reranked_docs": reranked}


# ── Node 5: GraphRAG ──────────────────────────────────────────────────────────
def graphrag_node(state: PipelineState) -> dict:
    """Expands evidence via entity co-occurrence graph (BFS hop-2)."""
    with latency_tracker.track("GraphRAGAgent"):
        graph_docs    = multihop_retrieve(state["question"])
        reranked_docs = state.get("reranked_docs", [])
        combined      = (reranked_docs + graph_docs[:2])[:TOP_K_RERANK + 2]

    if state.get("verbose"):
        _vprint("⑤ GraphRAGAgent  [none]")
        _vprint("   GraphRAG docs added:", str(len(graph_docs)))
        _vsep()

    _cb("GraphRAGAgent",
        latency_tracker._current.get("GraphRAGAgent", 0),
        {"hop_docs": len(graph_docs)})
    return {"reranked_docs": combined}


# ── Node 6: ICA Scoring ───────────────────────────────────────────────────────
def ica_scoring_node(state: PipelineState) -> dict:
    """Ranks documents by information increment (perplexity-based ICA)."""
    with latency_tracker.track("ICAAgent"):
        ica_scored = ica_score_documents(
            state["question"], state.get("reranked_docs", [])
        )
        ica_scored.sort(key=lambda x: -x[1])
        ranked_docs = [doc for doc, _ in ica_scored[:TOP_K_RERANK]]
        ica_scores  = [s   for _, s   in ica_scored[:TOP_K_RERANK]]

    if state.get("verbose"):
        ica_str = ", ".join(f"{s:.2f}" for s in ica_scores)
        _vprint("⑥ ICAAgent  [mistral-7b — ranking only, not verification]")
        _vprint("   ICA scores:", f"[{ica_str}]")
        _vsep()

    _cb("ICAAgent",
        latency_tracker._current.get("ICAAgent", 0),
        {"top_score": round(max(ica_scores) if ica_scores else 0.0, 3)})
    return {"relevant_docs": ranked_docs, "ica_scores": ica_scores}


# ── Node 7: Sentence Selection ────────────────────────────────────────────────
def sentence_selection_node(state: PipelineState) -> dict:
    """Selects top-10 sentences per-document using TF-IDF scoring."""
    with latency_tracker.track("SentenceSelectionAgent"):
        ctx = select_sentences_from_docs(
            state["question"], state.get("relevant_docs", []), top_n=10
        )
        if not ctx:
            ctx = " ".join(state.get("relevant_docs", []))

    if state.get("verbose"):
        snippet = ctx[:120].replace("\n", " ") + "…"
        _vprint("⑦ SentenceSelectionAgent  [tfidf]")
        _vprint("   Context snippet:", snippet)
        _vsep()

    _cb("SentenceSelectionAgent",
        latency_tracker._current.get("SentenceSelectionAgent", 0),
        {"snippet": ctx[:120] if ctx else ""})
    return {"final_context": ctx}


# ── Node 8: Answer Generation ─────────────────────────────────────────────────
def answer_generation_node(state: PipelineState) -> dict:
    """
    Generates N candidate answers with self-consistency.
    Tie-break uses BGE embedding similarity (NOT NLI) → no circular verification.
    """
    with latency_tracker.track("AnswerGenerationAgent"):
        answer, raw_candidates = generation_agent(
            state["question"], state["final_context"]
        )

        # Empty answer fallback chain
        if not answer:
            wiki_docs = state.get("wiki_docs", [])
            if wiki_docs:
                wiki_ctx  = select_sentences_from_docs(
                    state["question"], wiki_docs, top_n=10
                )
                answer, _ = generation_agent(state["question"], wiki_ctx or
                                             " ".join(wiki_docs))
                answer    = post_process_answer(answer)
        if not answer:
            for sq in state.get("sub_questions", []):
                w = wiki_search(sq, max_chars=2000)
                if w:
                    fresh, _ = generation_agent(state["question"], w)
                    fresh    = post_process_answer(fresh)
                    if fresh:
                        answer = fresh
                        break

    if state.get("verbose"):
        _vprint("⑧ AnswerGenerationAgent  [mistral-7b]")
        _vprint("   Tie-break: BGE cosine similarity (independent of generation model)")
        for k, cand in enumerate(raw_candidates, 1):
            marker = " ◀ selected" if cand == answer else ""
            _vprint(f"   Candidate {k}:", f'"{cand}"{marker}')
        _vsep()

    _cb("AnswerGenerationAgent",
        latency_tracker._current.get("AnswerGenerationAgent", 0),
        {"answer": answer, "candidates": len(raw_candidates)})
    return {"answer": answer, "raw_candidates": raw_candidates}


# ── Node 9: Verification ──────────────────────────────────────────────────────
def verification_node(state: PipelineState) -> dict:
    """
    Verifies the answer using DeBERTa-NLI.
    DeBERTa is INDEPENDENT from Llama-3.1 (generation) and BGE (retrieval).
    Raises an error if model family matches the generator — no circular verification.
    """
    with latency_tracker.track("VerificationAgent"):
        # Independence guard
        assert verification_agent.MODEL_FAMILY != generation_agent.MODEL_FAMILY, (
            "CIRCULAR VERIFICATION DETECTED: DeBERTa cannot verify Llama-3.1 output"
        )
        verif = verify_chain(state["answer"], state.get("relevant_docs", []))

    entailment_rate = verif.get("entailment_rate", 0.0)
    contradicted    = verif.get("contradicted",    False)
    confidence      = verif.get("max_entailment",  0.0)

    if state.get("verbose"):
        flag = "⚠ YES" if contradicted else "no"
        _vprint("⑨ VerificationAgent  [deberta ← independent of generation model]")
        _vprint("   Verifier:", PRIMARY_VERIFIER_MODEL)
        _vprint("   Entailment rate:", f"{entailment_rate:.2f}")
        _vprint("   Max entailment:",
                f"{confidence:.3f}  (threshold={CONFIDENCE_THRESHOLD})")
        _vprint("   Contradiction:", flag)
        _vsep()

    _cb("VerificationAgent",
        latency_tracker._current.get("VerificationAgent", 0),
        {"entailment": round(entailment_rate, 3), "contradicted": contradicted})
    return {
        "verification":    verif,
        "entailment_rate": entailment_rate,
        "contradicted":    contradicted,
        "confidence":      confidence,
    }


# ── Router: verification ──────────────────────────────────────────────────────
def route_verification(state: PipelineState) -> str:
    """
    Only regenerate when BOTH conditions are true:
    - confidence is low (answer not well-supported), AND
    - contradiction is detected
    High confidence + contradiction means the answer is a correct short form
    of a longer gold entity (e.g. "Bob Seger" vs "Bob Seger and the Silver Bullet Band").
    """
    confidence   = state.get("confidence", 1.0)
    contradicted = state.get("contradicted", False)
    within_budget = state.get("regen_attempt", 0) < MAX_REGEN_RETRIES

    # Only regenerate if confidence is low — ignore contradiction alone
    # when the model is highly confident (entailment=0.999)
    needs_regen = confidence < CONFIDENCE_THRESHOLD

    if needs_regen and within_budget:
        return "regeneration"
    return "finalise"


# ── Node 10: Regeneration ─────────────────────────────────────────────────────
def regeneration_node(state: PipelineState) -> dict:
    """
    Regenerates the answer with a stricter prompt.
    Regeneration is CONTROLLED by DeBERTa (external decision) →
    Llama-3.1 is not self-triggering → no circular self-verification.
    After regeneration, routes back to verification_node for re-check.
    """
    regen_attempt = state.get("regen_attempt", 0) + 1

    with latency_tracker.track("RegenerationAgent"):
        if regen_attempt == 1:
            # Attempt 1: strict context
            strict_ctx = (
                "Use ONLY the evidence below. "
                "Do NOT add outside knowledge.\n\n"
                + state["final_context"]
            )
            new_ans, _ = generation_agent(state["question"], strict_ctx)
        else:
            # Attempt 2: Wikipedia context
            wiki_docs  = state.get("wiki_docs", [])
            wiki_ctx   = (select_sentences_from_docs(
                              state["question"], wiki_docs, top_n=10)
                          if wiki_docs else state["final_context"])
            new_ans, _ = generation_agent(state["question"], wiki_ctx)

        new_ans = post_process_answer(new_ans)

    if state.get("verbose"):
        _vprint(f"⑩ RegenerationAgent  attempt {regen_attempt}"
                "  [mistral — triggered by deberta]")
        _vprint("   New answer:", f'"{new_ans}"')
        _vsep()

    return {
        "answer":        new_ans,
        "regen_attempt": regen_attempt,
    }


# ── Node 11: Finalise ─────────────────────────────────────────────────────────
def finalise_node(state: PipelineState) -> dict:
    """
    Computes final scores, hallucination type, commits latency.
    This is always the last node before END.
    """
    answer       = state.get("answer", "")
    gold_answer  = state.get("gold_answer", "")
    final_ctx    = state.get("final_context", "")

    hallucination_type = classify_hallucination(answer, final_ctx)
    scores             = compute_scores(answer, gold_answer)

    latency_tracker.commit_run()
    timings = latency_tracker._runs[-1] if latency_tracker._runs else {}

    if state.get("verbose"):
        em_icon  = "✓" if scores["exact_match"]  else "✗"
        apx_icon = "✓" if scores["approx_match"] else "✗"
        _vprint("FINAL RESULT")
        _vprint("   Predicted answer:", f'"{answer}"')
        _vprint("   Gold answer:",      f'"{gold_answer}"')
        _vprint("   Exact Match:",
                f'{em_icon}   F1: {scores["f1"]:.3f}   Approx: {apx_icon}')
        _vprint("   Hallucination:", hallucination_type)
        _vprint("   Regen attempts:", str(state.get("regen_attempt", 0)))
        _vsep()
        latency_tracker.report()
        _vheader("END")
        print()

    return {
        "predicted_answer":   answer,
        "hallucination_type": hallucination_type,
        "scores":             scores,
        "timings":            timings,
    }


print("All LangGraph node functions defined.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# GRAPHRAG — Entity Co-occurrence Graph + Multi-hop Retrieval
#
# Builds an undirected graph where nodes = documents and edges = shared
# named entities (title words ≥ 4 chars). BFS from the top retrieved
# document expands to related documents at hop depth 2.
# ═══════════════════════════════════════════════════════════════════════════════

import networkx as nx

# ── Build entity co-occurrence graph over document titles ─────────────────────
G = nx.Graph()

for idx, meta in chunk_meta.items():
    doc_id = meta["doc_id"]
    if not G.has_node(doc_id):
        G.add_node(doc_id, text=corpus[idx])

# Connect documents that share named entities (title words ≥ 4 chars)
doc_ids = list(corpus_map.keys())
for i, doc_a in enumerate(doc_ids):
    words_a = set(w.lower() for w in doc_a.split() if len(w) >= 4)
    for doc_b in doc_ids[i + 1:]:
        words_b = set(w.lower() for w in doc_b.split() if len(w) >= 4)
        if words_a & words_b:
            G.add_edge(doc_a, doc_b)

print(f"GraphRAG: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")


def multihop_retrieve(
    query:     str,
    top_k:     int = 3,
    hop_depth: int = 2,
) -> List[str]:
    """
    BFS expansion from the top dense-retrieved seed document.

    For multi-hop questions, the answer often lives in a document connected
    to (but not directly retrieved for) the seed. hop_depth=2 captures
    one additional hop without exploding the candidate set.
    """
    # Seed: top-1 from dense retrieval
    seed_texts, seed_ids = retrieve_chunks_deduplicated(query, top_k_docs=1)
    if not seed_ids:
        return []

    seed_node = seed_ids[0]
    if seed_node not in G:
        return []

    # BFS up to hop_depth
    visited  = {seed_node}
    frontier = {seed_node}
    for _ in range(hop_depth):
        new_frontier = set()
        for node in frontier:
            for neighbor in G.neighbors(node):
                if neighbor not in visited:
                    new_frontier.add(neighbor)
                    visited.add(neighbor)
        frontier = new_frontier

    # Return text of top_k BFS-discovered nodes (exclude seed)
    hop_docs = [
        corpus_map[n] for n in list(visited - {seed_node})
        if n in corpus_map
    ]
    return hop_docs[:top_k]


print("multihop_retrieve() defined.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LANGGRAPH — BUILD AND COMPILE
#
# Graph topology:
#
#   query_decomposition
#          |
#   hybrid_retrieval
#          |
#     quality_gate ──────[weak]──────> wikipedia_fallback
#          |                                    |
#          └──────────[strong]─────────────────>|
#                                               |
#                                          reranking
#                                               |
#                                           graphrag
#                                               |
#                                         ica_scoring
#                                               |
#                                    sentence_selection
#                                               |
#                                    answer_generation
#                                               |
#                                         verification ──[uncertain/contradicted]──>
#                                               |                                   |
#                                           finalise  <─────── regeneration ────────>
#                                               |           (loops back to verify)
#                                             END
# ═══════════════════════════════════════════════════════════════════════════════

from langgraph.graph import StateGraph, END

# ── Build graph ───────────────────────────────────────────────────────────────
builder = StateGraph(PipelineState)

# Add all nodes
builder.add_node("query_decomposition",  query_decomposition_node)
builder.add_node("hybrid_retrieval",     hybrid_retrieval_node)
builder.add_node("quality_gate",         quality_gate_node)
builder.add_node("wikipedia_fallback",   wikipedia_fallback_node)
builder.add_node("reranking",            reranking_node)
builder.add_node("graphrag",             graphrag_node)
builder.add_node("ica_scoring",          ica_scoring_node)
builder.add_node("sentence_selection",   sentence_selection_node)
builder.add_node("answer_generation",    answer_generation_node)
builder.add_node("verification",         verification_node)
builder.add_node("regeneration",         regeneration_node)
builder.add_node("finalise",             finalise_node)

# ── Linear edges ──────────────────────────────────────────────────────────────
builder.set_entry_point("query_decomposition")
builder.add_edge("query_decomposition", "hybrid_retrieval")
builder.add_edge("hybrid_retrieval",    "quality_gate")

# ── Conditional edge: quality gate ────────────────────────────────────────────
builder.add_conditional_edges(
    "quality_gate",
    route_quality_gate,
    {
        "wikipedia_fallback": "wikipedia_fallback",
        "reranking":          "reranking",
    }
)
builder.add_edge("wikipedia_fallback", "reranking")

# ── Linear edges continue ─────────────────────────────────────────────────────
builder.add_edge("reranking",          "graphrag")
builder.add_edge("graphrag",           "ica_scoring")
builder.add_edge("ica_scoring",        "sentence_selection")
builder.add_edge("sentence_selection", "answer_generation")
builder.add_edge("answer_generation",  "verification")

# ── Conditional edge: verification ────────────────────────────────────────────
builder.add_conditional_edges(
    "verification",
    route_verification,
    {
        "regeneration": "regeneration",
        "finalise":     "finalise",
    }
)

# ── Regeneration loops back to verification (not generation) ──────────────────
# This is the correct agentic loop: regenerate → re-verify → decide again.
# The loop terminates when route_verification returns "finalise" OR
# regen_attempt >= MAX_REGEN_RETRIES.
builder.add_edge("regeneration", "verification")

# ── Final edge ────────────────────────────────────────────────────────────────
builder.add_edge("finalise", END)

# ── Compile ───────────────────────────────────────────────────────────────────
rag_graph = builder.compile()

print("LangGraph compiled successfully.")
print()
print("Graph topology:")
print("  query_decomposition")
print("        ↓")
print("  hybrid_retrieval")
print("        ↓")
print("  quality_gate ──[weak]──> wikipedia_fallback ──>┐")
print("        ↓ [strong]                                ↓")
print("        └──────────────────────────────────> reranking")
print("                                                  ↓")
print("                                             graphrag")
print("                                                  ↓")
print("                                           ica_scoring")
print("                                                  ↓")
print("                                      sentence_selection")
print("                                                  ↓")
print("                                      answer_generation")
print("                                                  ↓")
print("                           ┌─── verification ────────────────┐")
print("                           ↓ [confident]       [uncertain]   ↓")
print("                        finalise           regeneration ──────┘")
print("                           ↓               (→ re-verify)")
print("                          END")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LANGGRAPH PIPELINE WRAPPER
#
# run_pipeline() now invokes the compiled LangGraph instead of manually
# sequencing agent calls. The graph handles all control flow including
# the quality gate conditional and the regeneration loop.
# ═══════════════════════════════════════════════════════════════════════════════

def _vprint(label: str, value: str = "", width: int = 60) -> None:
    if value:
        print(f"  {label:<26} {value}")
    else:
        print(f"  {label}")

def _vsep(char: str = "─", width: int = 62) -> None:
    print("  " + char * width)

def _vheader(title: str, width: int = 62) -> None:
    pad = (width - len(title) - 2) // 2
    print("  " + "═" * pad + " " + title + " " +
          "═" * (width - pad - len(title) - 2))


def run_pipeline(sample: dict, verbose: bool = False,
                 hitl_mode: bool = False,
                 hitl_points: list = None) -> dict:
    """
    Invoke the compiled LangGraph with the sample as initial state.
    Supports optional HITL mode — use run_pipeline_hitl() for pausing.

    The graph handles all routing — quality gate fallback and
    verification-triggered regeneration are conditional edges,
    not imperative if-statements.
    """
    if hitl_points is None:
        hitl_points = ["decomposition", "generation", "verification"]
    question    = sample["question"]
    gold_answer = sample["answer"]

    if verbose:
        _vheader("PIPELINE TRACE  [LangGraph]")
        _vprint("❓ Question:", question)
        _vprint("✅ Gold Answer:", gold_answer)
        _vsep()

    # Build initial state — only question/gold/verbose are set.
    # All other fields are populated by graph nodes.
    initial_state: PipelineState = {
        "question":         question,
        "gold_answer":      gold_answer,
        "verbose":          verbose,
        "regen_attempt":    0,
        "hitl_mode":        hitl_mode,
        "hitl_points":      hitl_points,
        "hitl_session_id":  "",
        "hitl_checkpoints": [],
    }

    # ── Invoke the graph ───────────────────────────────────────────────────────
    final_state = rag_graph.invoke(initial_state)

    # ── Extract results from final state ──────────────────────────────────────
    scores             = final_state.get("scores", {})
    verification       = final_state.get("verification", {})

    # ── Cannot-answer path ────────────────────────────────────────────────────
    # When entailment stays below 0.25 after regeneration, the pipeline
    # does not have enough grounded context to answer reliably.
    # Return a structured "cannot answer" instead of a low-confidence fabrication.
    predicted = final_state.get("predicted_answer", "")
    entail    = verification.get("entailment_rate", 0.0)
    regen_n   = final_state.get("regen_attempt", 0)
    cannot_answer = False

    if entail < 0.25 and regen_n >= 1 and predicted:
        # Answer exists but DeBERTa cannot verify it even after regeneration
        predicted     = (
            "I cannot confidently answer this question from available sources. "
            "The retrieved context does not sufficiently support any answer."
        )
        cannot_answer = True
        if verbose:
            _vprint("  ⚠ CANNOT ANSWER: entailment below 0.25 after regeneration.")

    return {
        "question":           question,
        "gold_answer":        gold_answer,
        "predicted_answer":   predicted,
        "exact_match":        scores.get("exact_match", 0),
        "approx_match":       scores.get("approx_match", 0),
        "f1":                 scores.get("f1", 0.0),
        "ensemble_score":     scores.get("ensemble_score", 0.0),
        "match_reason":       scores.get("match_reason", "none"),
        "entailment_rate":    verification.get("entailment_rate", 0.0),
        "regen_attempts":     final_state.get("regen_attempt", 0),
        "retrieval_quality":  final_state.get("rq_score", 0.0),
        "hallucination_type": final_state.get("hallucination_type", "none"),
        "verifier_model":     PRIMARY_VERIFIER_MODEL,
        "retrieval_source":   final_state.get("retrieval_source", "corpus"),
        "web_snippets_used":  len(final_state.get("web_docs", [])),
        "cannot_answer":      cannot_answer,
        "timings":            final_state.get("timings", {}),
        "hitl_checkpoints":   final_state.get("hitl_checkpoints", []),
    }


print("run_pipeline() wired to LangGraph.")
print("Call run_pipeline(sample, verbose=True) to execute.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPLAINABILITY MODULE
#
# Provides four explainability functions that work on top of the existing
# pipeline without modifying any agent:
#
#   1. explain_pipeline()       — full per-stage audit trail for one result
#   2. find_answer_source()     — which sentence in which doc produced the answer
#   3. compute_confidence()     — weighted confidence score from 3 independent signals
#   4. hallucination_by_type()  — fabrication rate breakdown by question type
#
# All functions use data already produced by the pipeline (PipelineState fields)
# — no extra LLM calls needed for 1-3. Function 4 post-processes eval results.
#
# Reference: DePaC arXiv:2412.14905 — hallucination classification methodology
# ═══════════════════════════════════════════════════════════════════════════════

from difflib import SequenceMatcher


# ── 1. Full per-stage audit trail ─────────────────────────────────────────────
def explain_pipeline(result: dict, state: dict = None) -> dict:
    """
    Build a full explainability report for one pipeline result.

    Parameters
    ----------
    result : dict   — output of run_pipeline()
    state  : dict   — final_state from LangGraph (optional, adds richer detail)

    Returns
    -------
    dict with one key per agent explaining its decision.
    """
    q   = result.get("question",          "")
    ans = result.get("predicted_answer",  "")
    rq  = result.get("retrieval_quality", 0.0)
    ent = result.get("entailment_rate",   0.0)
    hal = result.get("hallucination_type","none")
    reg = result.get("regen_attempts",    0)

    explanation = {
        "question": q,
        "answer":   ans,

        "stage_1_decomposition": {
            "sub_questions":  state.get("sub_questions", []) if state else [],
            "explanation":    "Multi-hop question split into sub-questions for targeted retrieval.",
            "auditable":      True,
            "hitl_editable":  True,
        },

        "stage_2_retrieval": {
            "doc_count":      len(state.get("retrieved_docs", [])) if state else "N/A",
            "wiki_fallback":  state.get("wiki_fallback", False)    if state else "N/A",
            "explanation":    "BGE FAISS (dense) + BM25 (sparse) fusion retrieved candidate documents.",
            "auditable":      True,
        },

        "stage_3_quality_gate": {
            "quality_score":  round(rq, 4),
            "threshold":      RETRIEVAL_QUALITY_THRESHOLD,
            "wiki_triggered": rq < RETRIEVAL_QUALITY_THRESHOLD,
            "explanation":    (
                f"Retrieval quality {rq:.2f} {'< threshold — Wikipedia fallback triggered.' if rq < RETRIEVAL_QUALITY_THRESHOLD else '>= threshold — corpus docs used directly.'}"
            ),
            "auditable":      True,
        },

        "stage_4_reranking": {
            "docs_after_rerank": len(state.get("reranked_docs", [])) if state else "N/A",
            "explanation":    "MS-MARCO cross-encoder re-scored and re-ordered retrieved documents.",
            "auditable":      True,
        },

        "stage_6_ica": {
            "ica_scores":     state.get("ica_scores", []) if state else [],
            "explanation":    (
                "Information-Calibrated Aggregation (DePaC §3.2): "
                "documents ranked by BGE cosine alignment with query. "
                "Higher score = more relevant context passed to generator."
            ),
            "auditable":      True,
        },

        "stage_7_context": {
            "context_length": len(state.get("final_context", "")) if state else "N/A",
            "snippet":        (state.get("final_context", "")[:150] + "...") if state else "N/A",
            "explanation":    "TF-IDF selected top-2 sentences per document to build final context.",
            "auditable":      True,
        },

        "stage_8_generation": {
            "candidates":     state.get("raw_candidates", []) if state else [],
            "selected":       ans,
            "agreement":      _candidate_agreement(state.get("raw_candidates", [ans])),
            "explanation":    (
                f"Self-consistency N={N_SAMPLES}: "
                f"{_candidate_agreement(state.get('raw_candidates', [ans])):.0%} of candidates agreed. "
                "BGE cosine selected best candidate."
            ),
            "auditable":      True,
        },

        "stage_9_verification": {
            "entailment_rate":  round(ent, 4),
            "contradicted":     state.get("contradicted", False) if state else "N/A",
            "verifier_model":   PRIMARY_VERIFIER_MODEL,
            "explanation":      (
                f"DeBERTa-NLI (independent of Mistral) verified answer. "
                f"Entailment: {ent:.0%}. "
                f"{'Contradiction detected — regeneration triggered.' if reg > 0 else 'No contradiction detected.'}"
            ),
            "auditable":        True,
            "independent":      True,   # key: verifier is independent of generator
        },

        "stage_10_regeneration": {
            "triggered":        reg > 0,
            "attempts":         reg,
            "explanation":      (
                f"Regeneration {'triggered (' + str(reg) + ' attempt(s)) due to low entailment / contradiction.' if reg > 0 else 'not triggered — answer passed verification.'}"
            ),
            "auditable":        True,
        },

        "hallucination_classification": {
            "type":         hal,
            "method":       "DePaC Appendix G (27 omission phrases + context containment check)",
            "explanation":  (
                "Omission: answer contains DePaC refusal phrase. "
                "Fabrication: answer not found in retrieved context. "
                "None: answer is grounded in retrieved context."
            ),
        },
    }
    return explanation


def _candidate_agreement(candidates: list) -> float:
    """Fraction of candidates that match the most common answer."""
    if not candidates:
        return 1.0
    from collections import Counter
    counts = Counter(c.strip().lower() for c in candidates)
    most_common_count = counts.most_common(1)[0][1]
    return most_common_count / len(candidates)


# ── 2. Answer source attribution ──────────────────────────────────────────────
def find_answer_source(answer: str, docs: list) -> dict:
    """
    Find which retrieved document and sentence contains the answer.

    Uses exact match first, then fuzzy SequenceMatcher for partial matches.
    Returns the source sentence and document index for attribution.
    """
    if not answer or not docs:
        return {"found": False, "doc_index": None,
                "sentence": None, "match_score": 0.0}

    ans_lower = answer.strip().lower()
    best      = {"found": False, "doc_index": None,
                 "sentence": None, "match_score": 0.0}

    for i, doc in enumerate(docs):
        sentences = [s.strip() for s in doc.replace(".", ". ").split(". ") if s.strip()]
        for sent in sentences:
            sent_lower = sent.lower()

            # Exact containment
            if ans_lower in sent_lower:
                return {
                    "found":       True,
                    "doc_index":   i,
                    "sentence":    sent,
                    "doc_snippet": doc[:200],
                    "match_score": 1.0,
                    "match_type":  "exact",
                }

            # Fuzzy match
            score = SequenceMatcher(None, ans_lower, sent_lower).ratio()
            if score > best["match_score"]:
                best = {
                    "found":       score > 0.4,
                    "doc_index":   i,
                    "sentence":    sent,
                    "doc_snippet": doc[:200],
                    "match_score": round(score, 3),
                    "match_type":  "fuzzy",
                }

    return best


# ── 3. Pipeline confidence score ──────────────────────────────────────────────
def compute_confidence(result: dict, state: dict = None) -> dict:
    """
    Aggregate three independent signals into an overall confidence score.

    Weights:
      - Retrieval quality  (stage 3):  30% — how good was the evidence?
      - Candidate agreement (stage 8): 30% — how consistent was generation?
      - Entailment rate    (stage 9):  40% — did DeBERTa verify the answer?

    All three signals are from independent models (BGE, Mistral, DeBERTa).
    """
    retrieval_conf  = result.get("retrieval_quality", 0.0)
    entailment_conf = result.get("entailment_rate",   0.0)
    candidates      = state.get("raw_candidates", []) if state else []
    agreement_conf  = _candidate_agreement(candidates) if candidates else entailment_conf

    overall = (
        retrieval_conf  * 0.30 +
        agreement_conf  * 0.30 +
        entailment_conf * 0.40
    )

    level = (
        "HIGH"   if overall >= 0.70 else
        "MEDIUM" if overall >= 0.45 else
        "LOW"
    )

    return {
        "retrieval_confidence":   round(retrieval_conf,  3),
        "generation_agreement":   round(agreement_conf,  3),
        "verification_confidence":round(entailment_conf, 3),
        "overall":                round(overall,         3),
        "level":                  level,
        "weights":                {"retrieval": 0.30, "agreement": 0.30, "entailment": 0.40},
    }


# ── 4. Hallucination breakdown by question type ───────────────────────────────
def hallucination_by_type(results: list) -> dict:
    """
    Break down fabrication rate by question type across an evaluation run.

    Identifies where the pipeline struggles most — useful for targeted
    improvement and for paper analysis.
    """
    by_type = {}
    for r in results:
        qt  = detect_question_type(r.get("question", ""))
        hal = r.get("hallucination_type", "none")
        by_type.setdefault(qt, {"total": 0, "fabrication": 0, "omission": 0, "none": 0})
        by_type[qt]["total"] += 1
        by_type[qt][hal if hal in ("fabrication","omission") else "none"] += 1

    breakdown = {}
    for qt, counts in sorted(by_type.items(), key=lambda x: -x[1]["total"]):
        total = counts["total"]
        breakdown[qt] = {
            "total":           total,
            "fabrication_pct": round(counts["fabrication"] / total * 100, 1),
            "omission_pct":    round(counts["omission"]    / total * 100, 1),
            "clean_pct":       round(counts["none"]        / total * 100, 1),
        }
    return breakdown


# ── 5. Print explain report ───────────────────────────────────────────────────
def print_explain(result: dict, state: dict = None):
    """Pretty-print the full explainability report for one result."""
    exp  = explain_pipeline(result, state)
    conf = compute_confidence(result, state)
    src  = find_answer_source(
        result.get("predicted_answer",""),
        state.get("retrieved_docs", []) if state else []
    )

    print("\n" + "═"*60)
    print(f"  EXPLAINABILITY REPORT")
    print("═"*60)
    print(f"  Question : {exp['question'][:70]}")
    print(f"  Answer   : {exp['answer']}")
    print(f"  Confidence: {conf['overall']:.0%} ({conf['level']})")
    print()

    stages = [
        ("① Decomposition",   "stage_1_decomposition"),
        ("③ Quality Gate",    "stage_3_quality_gate"),
        ("⑥ ICA",             "stage_6_ica"),
        ("⑧ Generation",      "stage_8_generation"),
        ("⑨ Verification",    "stage_9_verification"),
        ("⑩ Regeneration",    "stage_10_regeneration"),
    ]
    for label, key in stages:
        s = exp.get(key, {})
        print(f"  {label}")
        print(f"    → {s.get('explanation','')}")
        print()

    print(f"  CONFIDENCE BREAKDOWN")
    print(f"    Retrieval   (30%) : {conf['retrieval_confidence']:.0%}")
    print(f"    Agreement   (30%) : {conf['generation_agreement']:.0%}")
    print(f"    Entailment  (40%) : {conf['verification_confidence']:.0%}")
    print(f"    Overall           : {conf['overall']:.0%}  [{conf['level']}]")
    print()

    if src["found"]:
        print(f"  ANSWER SOURCE")
        print(f"    Doc index  : {src['doc_index']}")
        print(f"    Sentence   : {src['sentence'][:120]}")
        print(f"    Match type : {src['match_type']}  (score={src['match_score']})")
    else:
        print(f"  ANSWER SOURCE: not found in retrieved docs (possible fabrication)")
    print("═"*60)


# ── Quick test ────────────────────────────────────────────────────────────────
print("Explainability module loaded. Available functions:")
print("  explain_pipeline(result, state)   — full per-stage audit trail")
print("  find_answer_source(answer, docs)  — which sentence produced the answer")
print("  compute_confidence(result, state) — weighted 3-signal confidence score")
print("  hallucination_by_type(results)    — fabrication breakdown by question type")
print("  print_explain(result, state)      — pretty-print full report")
print()
print("Usage after run_pipeline():")
print("  result = run_pipeline(sample)")
print("  print_explain(result)")
print("  # or for eval set:")
print("  breakdown = hallucination_by_type(eval_results)")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# HUMAN IN THE LOOP — PIPELINE FUNCTIONS (synchronous)
#
# Now fully synchronous — no async/await, no ThreadPoolExecutor.
# hitl_checkpoint() blocks the calling thread using threading.Event.wait().
# This works correctly whether called from:
#   - Colab notebook cell (direct call)
#   - FastAPI background thread (via run_pipeline_hitl in /query endpoint)
#   - Notebook manual test cell
# ═══════════════════════════════════════════════════════════════════════════════

import threading, uuid, sys, time


def run_pipeline_hitl(
    sample:      dict,
    hitl_points: list = None,
    verbose:     bool = False,
) -> dict:
    """
    Synchronous HITL pipeline.

    Wraps run_pipeline() with three checkpoint pauses using
    threading.Event — no asyncio required.

    Usage (notebook):
        result = run_pipeline_hitl(
            {"question": "...", "answer": ""},
            hitl_points=["decomposition","generation","verification"],
            verbose=True,
        )
    """
    if hitl_points is None:
        hitl_points = ["decomposition", "generation", "verification"]

    q           = sample["question"]
    sid         = str(uuid.uuid4())[:8]
    checkpoints = []

    # ── Checkpoint 1: Query Decomposition ─────────────────────────────────────
    sub_questions = decompose_query(q)

    if "decomposition" in hitl_points:
        decision = hitl_checkpoint(
            f"{sid}_decomp", "decomposition", {
                "question":      q,
                "sub_questions": sub_questions,
                "message": "Review the sub-questions before retrieval. Approve or edit.",
            }
        )
        if decision["action"] == "edit":
            sub_questions = decision.get("sub_questions", sub_questions)
        elif decision["action"] == "reject":
            return {
                "predicted_answer": "", "error": "Rejected at decomposition",
                "hitl_checkpoints": checkpoints, "regen_attempts": 0,
                "entailment_rate": 0.0, "retrieval_quality": 0.0,
                "hallucination_type": "omission", "timings": {},
            }
        checkpoints.append({
            "checkpoint": "decomposition", "decision": decision["action"],
            "auto": decision.get("auto", False), "sub_questions": sub_questions,
        })

    # ── Run LangGraph with approved sub-questions ─────────────────────────────
    _orig = decompose_query
    def _patched(_q): return sub_questions
    patched_mods = []
    for mod in list(sys.modules.values()):
        if mod is not None and hasattr(mod,"decompose_query") and mod.decompose_query is _orig:
            mod.decompose_query = _patched
            patched_mods.append(mod)
    try:
        result = run_pipeline(sample, verbose=verbose)
    finally:
        for mod in patched_mods:
            mod.decompose_query = _orig

    answer = result.get("predicted_answer", "")

    # ── Checkpoint 2: Answer Generation ───────────────────────────────────────
    if "generation" in hitl_points:
        decision = hitl_checkpoint(
            f"{sid}_gen", "generation", {
                "question":        q,
                "answer":          answer,
                "entailment_rate": result.get("entailment_rate", 0),
                "regen_attempts":  result.get("regen_attempts",  0),
                "hallucination":   result.get("hallucination_type", "none"),
                "message": "Review the answer. Approve, edit, or regenerate.",
            }
        )
        if decision["action"] == "edit":
            answer = decision.get("answer", answer)
        elif decision["action"] == "regenerate":
            result = run_pipeline(sample, verbose=False)
            answer = result.get("predicted_answer", "")
        elif decision["action"] == "reject":
            return {
                "predicted_answer": "", "error": "Rejected at generation",
                "hitl_checkpoints": checkpoints, "regen_attempts": 0,
                "entailment_rate": 0.0, "retrieval_quality": 0.0,
                "hallucination_type": "omission", "timings": {},
            }
        checkpoints.append({
            "checkpoint": "generation", "decision": decision["action"],
            "auto": decision.get("auto", False), "final_answer": answer,
        })

    # ── Checkpoint 3: Verification (low confidence only) ─────────────────────
    if "verification" in hitl_points and result.get("entailment_rate", 1.0) < 0.3:
        er = result.get("entailment_rate", 0)
        decision = hitl_checkpoint(
            f"{sid}_verif", "verification", {
                "question":        q,
                "answer":          answer,
                "entailment_rate": er,
                "hallucination":   result.get("hallucination_type", "none"),
                "message": f"Low confidence ({er:.0%}). Approve or regenerate.",
            }
        )
        if decision["action"] == "regenerate":
            result = run_pipeline(sample, verbose=False)
            answer = result.get("predicted_answer", "")
        checkpoints.append({
            "checkpoint": "verification", "decision": decision["action"],
            "auto": decision.get("auto", False),
        })

    result["predicted_answer"] = answer
    result["hitl_checkpoints"] = checkpoints
    result["hitl_session_id"]  = sid
    return result


print("run_pipeline_hitl() defined (synchronous, threading.Event).")
print("  Usage: result = run_pipeline_hitl(sample, hitl_points=[...])")


In [ ]:
from torch.utils.data import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from transformers import TrainingArguments, Trainer
import gc

# ── Build QLoRA instruction-tuning dataset from HotPotQA training split ───────
# Positive: teach model to answer from context
# Negative (30%): teach model to refuse when context is irrelevant
#   → directly targets the fabrication hallucination rate

def build_finetune_sample(sample: dict, negative: bool = False) -> str:
    question = sample["question"]
    answer   = sample["answer"]
    context  = " ".join(
        " ".join(sents)
        for title, sents in zip(
            sample["context"]["title"],
            sample["context"]["sentences"]
        )
    )[:800]

    instruction = (
        "Answer the question using ONLY the context below. "
        "Give a SHORT, DIRECT answer. "
        "If the context does not contain the answer, say "
        "'I cannot determine the answer from the provided context.'"
    )
    if negative:
        return (
            f"[INST] {instruction}\n\n"
            f"Context: {context[:200]}... [unrelated passage]\n\n"
            f"Question: {question} [/INST] "
            f"I cannot determine the answer from the provided context."
        )
    return (
        f"[INST] {instruction}\n\n"
        f"Context: {context}\n\n"
        f"Question: {question} [/INST] {answer}"
    )


class HotPotQAFinetuneDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=512,
                 neg_ratio=QLORA_NEG_RATIO):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.texts      = []
        for sample in samples:
            self.texts.append(build_finetune_sample(sample, negative=False))
            if np.random.random() < neg_ratio:
                self.texts.append(build_finetune_sample(sample, negative=True))
        n_neg = sum(1 for t in self.texts if "cannot determine" in t)
        print(f"  Dataset: {len(self.texts)} samples  ({n_neg} negative refusals)")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].squeeze()
        labels    = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids":      input_ids,
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels":         labels,
        }


print("Building QLoRA fine-tuning dataset...")
finetune_dataset = HotPotQAFinetuneDataset(
    samples    = hotpot_train,
    tokenizer  = tokenizer,
    max_length = 512,
)
print(f"Fine-tuning dataset ready: {len(finetune_dataset)} samples")

# ── LoRA config ───────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r              = QLORA_RANK,
    lora_alpha     = QLORA_ALPHA,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout   = 0.05,
    bias           = "none",
    task_type      = TaskType.CAUSAL_LM,
)

print("Preparing model for QLoRA training...")
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir                  = QLORA_OUTPUT_DIR,
    num_train_epochs            = QLORA_EPOCHS,
    per_device_train_batch_size = QLORA_BATCH_SIZE,
    gradient_accumulation_steps = QLORA_GRAD_ACCUM,
    learning_rate               = QLORA_LR,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.05,
    fp16                        = True,
    logging_steps               = 10,
    save_strategy               = "epoch",
    optim                       = "paged_adamw_8bit",
    report_to                   = "none",
    dataloader_pin_memory       = False,
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = finetune_dataset,
)

print(f"Starting QLoRA fine-tuning...")
print(f"  Epochs         : {QLORA_EPOCHS}")
print(f"  Effective batch: {QLORA_BATCH_SIZE * QLORA_GRAD_ACCUM}")
print(f"  Learning rate  : {QLORA_LR}  |  Rank: {QLORA_RANK}")
print()

trainer.train()

model.save_pretrained(QLORA_OUTPUT_DIR)
tokenizer.save_pretrained(QLORA_OUTPUT_DIR)
print(f"LoRA adapter saved to {QLORA_OUTPUT_DIR}")

gc.collect()
torch.cuda.empty_cache()


In [ ]:
from peft import PeftModel

# ── Reload base model and merge LoRA adapter for inference ────────────────────
print("Reloading base model for clean inference copy...")

from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model_for_inference = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.1",
    quantization_config = None          if IS_A100 else bnb_cfg,
    torch_dtype         = torch.float16 if IS_A100 else None,
    device_map          = "auto",
    trust_remote_code   = True,
)

print("Merging LoRA adapter...")
finetuned_model = PeftModel.from_pretrained(base_model_for_inference, QLORA_OUTPUT_DIR)
finetuned_model = finetuned_model.merge_and_unload()
finetuned_model.eval()

# Store both for comparison
base_model_weights     = model   # pre-QLoRA
model                  = finetuned_model
finetuned_model_weights = finetuned_model

vram = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0
print(f"Fine-tuned model ready.  VRAM: {vram:.1f} GB")


In [ ]:
from tqdm import tqdm
import random


def run_evaluation(samples: list, label: str,
                   verbose_n: int = VERBOSE_SAMPLES) -> dict:
    """Run the full pipeline over samples and return aggregated metrics."""
    results = []
    print(f"\n{'='*55}")
    print(f"  Evaluating: {label}  ({len(samples)} samples)")
    print(f"  Verbose trace for first {verbose_n} samples.")
    print(f"{'='*55}\n")

    for i, sample in enumerate(tqdm(samples, desc=label)):
        try:
            result = run_pipeline(sample, verbose=(i < verbose_n))
            results.append(result)
            if (i + 1) % 5 == 0:
                em  = np.mean([r["exact_match"]  for r in results])
                apx = np.mean([r["approx_match"] for r in results])
                f1  = np.mean([r["f1"]           for r in results])
                rr  = np.mean([r["regen_attempts"] > 0 for r in results])
                print(f"  [{i+1:2d}/{len(samples)}]  EM={em:.3f}  "
                      f"Approx={apx:.3f}  F1={f1:.3f}  RegenRate={rr:.2%}")
        except Exception as e:
            print(f"  Sample {i} failed: {e}")
            results.append({
                "question":          sample["question"],
                "gold_answer":       sample["answer"],
                "predicted_answer":  "",
                "exact_match":       0, "approx_match":      0,
                "f1":                0.0, "ensemble_score":    0.0,
                "match_reason":      "error", "entailment_rate":   0.0,
                "regen_attempts":    0, "retrieval_quality": 0.0,
                "hallucination_type": "fabrication",
            })

    n             = len(results)
    halluc_types  = [r["hallucination_type"] for r in results]
    omission_rate = halluc_types.count("omission")    / n * 100
    fabr_rate     = halluc_types.count("fabrication") / n * 100

    metrics = {
        "exact_match":         float(np.mean([r["exact_match"]    for r in results])),
        "approx_match":        float(np.mean([r["approx_match"]   for r in results])),
        "f1":                  float(np.mean([r["f1"]             for r in results])),
        "ensemble_score":      float(np.mean([r["ensemble_score"] for r in results])),
        "regen_rate":          float(np.mean([r["regen_attempts"] > 0 for r in results])),
        "entailment_rate":     float(np.mean([r["entailment_rate"]for r in results])),
        "omission_rate":       omission_rate,
        "fabrication_rate":    fabr_rate,
        "total_hallucination": omission_rate + fabr_rate,
        "match_reasons":       dict(Counter(r["match_reason"] for r in results)),
    }

    print(f"\n{'='*55}")
    print(f"  RESULTS — {label}")
    print(f"{'='*55}")
    print(f"  Exact Match         : {metrics['exact_match']:.4f}")
    print(f"  Approx Match        : {metrics['approx_match']:.4f}")
    print(f"  F1 Score            : {metrics['f1']:.4f}")
    print(f"  Ensemble Score      : {metrics['ensemble_score']:.4f}")
    print(f"  Regeneration Rate   : {metrics['regen_rate']:.4f}")
    print(f"  Entailment Rate     : {metrics['entailment_rate']:.4f}")
    print(f"  Fact Omission       : {metrics['omission_rate']:.1f}%")
    print(f"  Fact Fabrication    : {metrics['fabrication_rate']:.1f}%")
    print(f"  Total Hallucination : {metrics['total_hallucination']:.1f}%")
    print()
    print("  Match reason breakdown:")
    for reason, count in sorted(
            metrics["match_reasons"].items(), key=lambda x: -x[1]):
        print(f"    {reason:<22}: {count:2d}/{n}  {'█' * count}")
    print(f"{'='*55}")
    return {"metrics": metrics, "results": results}


# ── Fixed seed for reproducible evaluation ────────────────────────────────────
random.seed(42)
eval_samples = random.sample(hotpot_eval, min(NUM_EVAL, len(hotpot_eval)))

# ── Mistral-7B evaluation ───────────────────────────────────────────────────────
# No model weights to set — model is already loaded.
# No GPU needed. Rate limit: 15 req/min on free tier.
ft_run = run_evaluation(eval_samples, label="HotPotQA — Mistral-7B + QLoRA")

# ── Expose metrics for chart cell ─────────────────────────────────────────────
ft_metrics    = ft_run["metrics"]
# Note: "ft" prefix kept for compatibility with chart cell
results       = ft_run["results"]
em_score      = ft_metrics["exact_match"]
approx_score  = ft_metrics["approx_match"]
f1_score      = ft_metrics["f1"]
ens_score     = ft_metrics["ensemble_score"]
regen_rate    = ft_metrics["regen_rate"]
entail_rate   = ft_metrics["entailment_rate"]
omission_rate = ft_metrics["omission_rate"]
fabr_rate     = ft_metrics["fabrication_rate"]
total_halluc  = ft_metrics["total_hallucination"]
match_reasons = Counter(ft_metrics["match_reasons"])

# ── Aggregate latency report ──────────────────────────────────────────────────
latency_tracker.report_aggregate()

import json as _json
with open(f"{OUTPUT_DIR}/latency_report.json", "w") as f:
    _json.dump(latency_tracker.to_dict(), f, indent=2)
print(f"Latency report saved to {OUTPUT_DIR}/latency_report.json")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FLASK BACKEND WITH HITL
#
# Why Flask instead of FastAPI:
#   FastAPI uses asyncio — threading.Event.set() from one async context
#   does not reliably wake threading.Event.wait() in another thread.
#   Flask with threaded=True uses plain OS threads throughout, so
#   set() always wakes wait() with no event-loop interference.
#
# Run this cell AFTER evaluation. Keep running while using the frontend UI.
# ═══════════════════════════════════════════════════════════════════════════════

import subprocess, sys, types as _types, threading, time

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "flask", "flask-cors", "pyngrok"])

# ── Embed HTML frontend source ────────────────────────────────────────────────
# The HTML is embedded here so Flask can serve it directly.
HTML_SOURCE = r'''
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width,initial-scale=1.0"/>
<title>An Agentic Framework for Reliable Retrieval-Augmented Text Generation using Verification, Correction, and Multi-Model Evaluation</title>
<link href="https://fonts.googleapis.com/css2?family=DM+Mono:wght@400;500&family=Fraunces:ital,wght@0,300;0,700;1,300&family=DM+Sans:wght@300;400;500&display=swap" rel="stylesheet"/>
<style>
:root{--bg:#0a0c0f;--surface:#111418;--surface2:#181c22;--border:#232830;--accent:#00e5b0;--accent2:#7c6dfa;--warn:#f5a623;--danger:#ff4d6d;--text:#e8eaf0;--muted:#6b7280;--mono:'DM Mono',monospace;--serif:'Fraunces',Georgia,serif;--sans:'DM Sans',sans-serif}
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
body{background:var(--bg);color:var(--text);font-family:var(--sans);font-size:14px;line-height:1.6;min-height:100vh;overflow-x:hidden}
body::before{content:'';position:fixed;inset:0;pointer-events:none;z-index:0;background-image:linear-gradient(rgba(0,229,176,.03) 1px,transparent 1px),linear-gradient(90deg,rgba(0,229,176,.03) 1px,transparent 1px);background-size:40px 40px}
.shell{position:relative;z-index:1;display:grid;grid-template-columns:300px 1fr;grid-template-rows:56px 1fr;height:100vh}
header{grid-column:1/-1;display:flex;align-items:center;padding:0 24px;gap:14px;border-bottom:1px solid var(--border);background:rgba(10,12,15,.94);backdrop-filter:blur(12px)}
.logo{font-family:var(--serif);font-size:20px;font-weight:700;color:var(--accent);letter-spacing:-.5px}
.logo span{color:var(--text);font-weight:300;font-style:italic}
.htag{font-family:var(--mono);font-size:10px;color:var(--muted);background:var(--surface2);border:1px solid var(--border);padding:2px 8px;border-radius:3px;letter-spacing:1px;text-transform:uppercase}
.hright{margin-left:auto;display:flex;align-items:center;gap:10px}
.pbadge{font-family:var(--mono);font-size:10px;padding:2px 8px;border-radius:3px;border:1px solid rgba(124,109,250,.4);background:rgba(124,109,250,.08);color:var(--accent2)}
.sdot{width:7px;height:7px;border-radius:50%;background:var(--accent);box-shadow:0 0 8px var(--accent);animation:pulse 2s infinite}
@keyframes pulse{0%,100%{opacity:1}50%{opacity:.4}}
.slabel{font-family:var(--mono);font-size:11px;color:var(--muted)}
.mbadge{font-family:var(--mono);font-size:10px;padding:2px 8px;border-radius:3px;border:1px solid;transition:all .2s}
.mbadge.demo{color:var(--warn);border-color:rgba(245,166,35,.3);background:rgba(245,166,35,.06)}
.mbadge.live{color:var(--accent);border-color:rgba(0,229,176,.3);background:rgba(0,229,176,.06)}
.hitl-ind{font-family:var(--mono);font-size:10px;padding:2px 8px;border-radius:3px;border:1px solid rgba(124,109,250,.4);background:rgba(124,109,250,.08);color:var(--accent2);display:none}
aside{border-right:1px solid var(--border);background:var(--surface);overflow-y:auto;padding:20px 0}
.ssec{padding:0 16px 18px;border-bottom:1px solid var(--border);margin-bottom:14px}
.ssec:last-child{border-bottom:none}
.slbl{font-family:var(--mono);font-size:10px;text-transform:uppercase;letter-spacing:1.5px;color:var(--muted);margin-bottom:10px}
.persona-cards{display:flex;flex-direction:column;gap:6px}
.pcard{background:var(--surface2);border:2px solid var(--border);border-radius:7px;padding:10px 12px;cursor:pointer;transition:all .15s}
.pcard:hover{border-color:rgba(124,109,250,.4)}
.pcard.active{border-color:var(--accent2);background:rgba(124,109,250,.08)}
.pcard-name{font-family:var(--mono);font-size:12px;color:var(--accent2);margin-bottom:3px;display:flex;align-items:center;gap:6px}
.pcard.active .pcard-name{color:var(--accent)}
.adot{width:5px;height:5px;border-radius:50%;background:var(--accent);display:none}
.pcard.active .adot{display:block}
.pcard-desc{font-size:11px;color:var(--muted);line-height:1.4}
.trow{display:flex;align-items:center;justify-content:space-between;margin-bottom:8px}
.trow label.tlbl{font-size:12px;color:#9ca3af}
.tgl{position:relative;width:36px;height:20px;cursor:pointer}
.tgl input{opacity:0;width:0;height:0}
.ttrack{position:absolute;inset:0;background:var(--surface2);border:1px solid var(--border);border-radius:20px;transition:.2s}
.tgl input:checked+.ttrack{background:var(--accent2);border-color:var(--accent2)}
.tthumb{position:absolute;height:14px;width:14px;left:3px;bottom:3px;background:var(--muted);border-radius:50%;transition:.2s}
.tgl input:checked~.tthumb{transform:translateX(16px);background:#fff}
.urlrow{display:flex;gap:6px;margin-top:8px}
.urlinput{flex:1;background:var(--surface2);border:1px solid var(--border);color:var(--text);font-family:var(--mono);font-size:10px;padding:6px 10px;border-radius:4px;outline:none;transition:border-color .2s}
.urlinput:focus{border-color:var(--accent)}
.urlinput::placeholder{color:var(--muted)}
.cbtn{background:var(--accent2);color:#fff;border:none;font-family:var(--mono);font-size:10px;padding:6px 10px;border-radius:4px;cursor:pointer;white-space:nowrap;transition:all .15s}
.cbtn:hover{background:#9580ff}
.bnote{font-size:10px;color:var(--muted);line-height:1.5;margin-top:6px}
.hcb-group{margin-top:8px;padding:10px 12px;background:var(--surface2);border:1px solid var(--border);border-radius:6px}
.hcb-title{font-family:var(--mono);font-size:10px;color:var(--accent2);text-transform:uppercase;letter-spacing:1px;margin-bottom:8px}
.hcb-row{display:flex;align-items:center;gap:8px;font-size:12px;color:#9ca3af;margin-bottom:6px;cursor:pointer}
.hcb-row:last-child{margin-bottom:0}
.hcb-row input[type=checkbox]{accent-color:var(--accent2);width:13px;height:13px}
.crow{display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;gap:8px}
.crow label{font-size:12px;color:#9ca3af;flex-shrink:0}
.crow input[type=range]{flex:1;accent-color:var(--accent);height:3px}
.cval{font-family:var(--mono);font-size:11px;color:var(--accent);min-width:36px;text-align:right}
.badge{display:inline-flex;align-items:center;gap:5px;font-family:var(--mono);font-size:10px;padding:3px 8px;border-radius:3px;border:1px solid;margin:3px 3px 3px 0}
.bg{color:var(--accent);border-color:rgba(0,229,176,.3);background:rgba(0,229,176,.06)}
.bp{color:var(--accent2);border-color:rgba(124,109,250,.3);background:rgba(124,109,250,.06)}
.bo{color:var(--warn);border-color:rgba(245,166,35,.3);background:rgba(245,166,35,.06)}
.agent-list{display:flex;flex-wrap:wrap}
.sbtn{background:var(--surface2);border:1px solid var(--border);color:#9ca3af;font-family:var(--sans);font-size:12px;padding:6px 10px;border-radius:4px;cursor:pointer;text-align:left;transition:all .15s;width:100%;margin-bottom:5px;line-height:1.4}
.sbtn:hover{border-color:var(--accent);color:var(--text);background:rgba(0,229,176,.05)}
main{display:flex;flex-direction:column;overflow:hidden}
.qarea{padding:20px 24px;border-bottom:1px solid var(--border);background:var(--surface)}
.qlabel{font-family:var(--serif);font-size:20px;font-weight:300;font-style:italic;color:var(--text);margin-bottom:12px}
.qbox{display:flex;gap:10px;align-items:flex-start}
textarea{flex:1;background:var(--surface2);border:1px solid var(--border);color:var(--text);font-family:var(--sans);font-size:14px;padding:11px 14px;border-radius:6px;resize:none;height:72px;outline:none;transition:border-color .2s;line-height:1.5}
textarea::placeholder{color:var(--muted)}
textarea:focus{border-color:var(--accent)}
.btngrp{display:flex;flex-direction:column;gap:6px}
.runbtn{background:var(--accent);color:#0a0c0f;border:none;font-family:var(--mono);font-size:12px;font-weight:500;letter-spacing:.5px;padding:11px 20px;border-radius:6px;cursor:pointer;transition:all .2s;white-space:nowrap;text-transform:uppercase;min-width:90px}
.runbtn:hover:not(:disabled){background:#00ffca;transform:translateY(-1px);box-shadow:0 4px 20px rgba(0,229,176,.3)}
.runbtn:disabled{opacity:.4;cursor:not-allowed;transform:none}
.clrbtn{background:transparent;color:var(--muted);border:1px solid var(--border);font-family:var(--mono);font-size:11px;padding:6px 12px;border-radius:6px;cursor:pointer;transition:all .2s}
.clrbtn:hover{border-color:var(--danger);color:var(--danger)}
.oarea{flex:1;overflow-y:auto;padding:18px 24px;display:flex;flex-direction:column;gap:10px}
.empty{display:flex;flex-direction:column;align-items:center;justify-content:center;gap:14px;color:var(--muted);text-align:center;padding:60px 20px}
.empty-g{font-size:44px;opacity:.18;font-family:var(--mono);color:var(--accent);line-height:1}
.empty p{font-size:13px;max-width:320px}

/* ── Agent stage card ── */
.acard{background:var(--surface);border:1px solid var(--border);border-radius:8px;animation:fadeIn .25s ease both}
@keyframes fadeIn{from{opacity:0;transform:translateY(5px)}to{opacity:1;transform:translateY(0)}}
.ahead{display:flex;align-items:center;padding:10px 14px;gap:10px;min-height:40px;background:var(--surface2);cursor:pointer;user-select:none;border-bottom:1px solid transparent;transition:background .15s}
.ahead:hover{background:#1e242c}
.ahead.open{border-bottom-color:var(--border)}
.anum{font-family:var(--mono);font-size:10px;color:var(--muted);min-width:18px}
.aname{font-family:var(--mono);font-size:12px;color:var(--text);flex:1}
.afam{font-family:var(--mono);font-size:10px;padding:2px 6px;border-radius:3px;border:1px solid}
.fgemini{color:#4285f4;border-color:rgba(66,133,244,.3);background:rgba(66,133,244,.06)}
.fmistral{color:#00e5b0;border-color:rgba(0,229,176,.3);background:rgba(0,229,176,.06)}
.fdeberta{color:var(--accent2);border-color:rgba(124,109,250,.3);background:rgba(124,109,250,.06)}
.fbge{color:var(--accent);border-color:rgba(0,229,176,.3);background:rgba(0,229,176,.06)}
.fcross{color:var(--warn);border-color:rgba(245,166,35,.3);background:rgba(245,166,35,.06)}
.ftfidf{color:#9ca3af;border-color:rgba(156,163,175,.3);background:rgba(156,163,175,.06)}
.fnone{color:#6b7280;border-color:rgba(107,114,128,.2);background:rgba(107,114,128,.04)}
.alat{font-family:var(--mono);font-size:11px;color:var(--muted);min-width:50px;text-align:right}
.alat.warn{color:var(--warn)}
.achev{font-size:9px;color:var(--muted);transition:transform .2s}
.ahead.open .achev{transform:rotate(90deg)}
.abody{padding:0 14px;max-height:0;overflow:hidden;transition:max-height .3s ease,padding .3s ease;box-sizing:border-box}
.abody.open{padding:10px 14px;max-height:2000px}
.lattrack{height:5px;background:var(--surface2);border-radius:3px;overflow:hidden;margin:4px 0 10px}
.latfill{height:100%;border-radius:3px;background:var(--accent);transition:width .5s ease}
.latfill.warn{background:var(--warn)}
.kvrow{display:flex;gap:8px;margin-bottom:5px;font-size:12px;align-items:baseline}
.kvkey{font-family:var(--mono);color:var(--muted);min-width:130px;flex-shrink:0;font-size:11px}
.kvval{color:var(--text);word-break:break-word;line-height:1.5}
.kvval.good{color:var(--accent)}
.kvval.warn{color:var(--warn)}

/* ── RUNNING state for active agent ── */
.acard.running .ahead{background:rgba(0,229,176,.05);border-bottom-color:rgba(0,229,176,.2)}
.acard.running .aname{color:var(--accent)}
.acard.running .alat{color:var(--accent)}
.running-dot{display:inline-block;width:7px;height:7px;border-radius:50%;background:var(--accent);animation:rpulse 1s ease-in-out infinite;margin-right:6px}
@keyframes rpulse{0%,100%{transform:scale(1);opacity:1}50%{transform:scale(1.4);opacity:.6}}

/* ── Status timeline ── */
.timeline{display:flex;flex-direction:column;gap:0;margin-bottom:10px}
.tl-item{display:flex;align-items:flex-start;gap:10px;padding:4px 0}
.tl-dot{width:10px;height:10px;border-radius:50%;flex-shrink:0;margin-top:5px}
.tl-dot.done{background:var(--accent)}
.tl-dot.running{background:var(--accent);animation:rpulse 1s infinite}
.tl-dot.pending{background:var(--border);border:1px solid var(--muted)}
.tl-dot.skipped{background:var(--muted)}
.tl-line{width:1px;height:16px;background:var(--border);margin-left:4px}
.tl-label{font-family:var(--mono);font-size:11px;color:var(--muted)}
.tl-label.done{color:var(--text)}
.tl-label.running{color:var(--accent)}

/* ── Result card ── */
.src-badge{font-family:var(--mono);font-size:10px;padding:2px 8px;border-radius:3px;margin-left:8px;display:inline-block}
.src-corpus{color:var(--accent);border:1px solid rgba(0,229,176,.3);background:rgba(0,229,176,.06)}
.src-wikipedia{color:var(--accent2);border:1px solid rgba(124,109,250,.3);background:rgba(124,109,250,.06)}
.src-web{color:var(--warn);border:1px solid rgba(245,166,35,.3);background:rgba(245,166,35,.06)}
.cannot-banner{background:rgba(255,77,109,.08);border:1px solid rgba(255,77,109,.3);border-radius:8px;padding:10px 14px;margin:8px 0;font-family:var(--mono);font-size:11px;color:#ff4d6d}
.rcard{background:var(--surface);border:1px solid var(--border);border-radius:8px;padding:20px;animation:fadeIn .35s ease both;border-left:3px solid var(--accent)}
.rtitle{font-family:var(--serif);font-size:12px;font-weight:300;color:var(--muted);text-transform:uppercase;letter-spacing:1px;margin-bottom:10px}
.ranswer{font-family:var(--serif);font-size:28px;font-weight:700;color:var(--accent);letter-spacing:-.5px;margin-bottom:16px;line-height:1.2}
.mrow{display:flex;flex-wrap:wrap;gap:8px}
.mchip{background:var(--surface2);border:1px solid var(--border);border-radius:5px;padding:6px 12px;display:flex;flex-direction:column;align-items:center;gap:2px;min-width:80px}
.mval{font-family:var(--mono);font-size:16px;font-weight:500;color:var(--text);line-height:1}
.mval.good{color:var(--accent)}
.mval.warn{color:var(--warn)}
.mlbl{font-size:10px;color:var(--muted);font-family:var(--mono);text-transform:uppercase;letter-spacing:.5px}
.lcard{background:var(--surface);border:1px solid var(--border);border-radius:8px;padding:18px;display:flex;align-items:center;gap:14px}
.spin{width:18px;height:18px;border:2px solid var(--border);border-top-color:var(--accent);border-radius:50%;animation:spin .8s linear infinite;flex-shrink:0}
@keyframes spin{to{transform:rotate(360deg)}}
.ltxt{font-family:var(--mono);font-size:12px;color:var(--muted)}
.lstg{color:var(--accent)}

/* ── HITL panel ── */
#hitlOverlay{display:none;position:fixed;inset:0;background:rgba(0,0,0,.6);z-index:199;backdrop-filter:blur(3px)}#hitlPanel{display:none;position:fixed;bottom:24px;right:24px;width:440px;max-width:95vw;background:var(--surface);border:2px solid var(--accent2);border-radius:10px;padding:20px;z-index:200;box-shadow:0 8px 40px rgba(124,109,250,.25);animation:fadeIn .3s ease both}
.hitl-header{font-family:var(--mono);font-size:10px;text-transform:uppercase;letter-spacing:1.5px;color:var(--accent2);margin-bottom:10px;display:flex;align-items:center;gap:8px}
.hitl-title{font-family:var(--serif);font-size:17px;font-weight:700;color:var(--text);margin-bottom:6px}
.hitl-msg{font-size:12px;color:var(--muted);line-height:1.6;margin-bottom:12px}
.hitl-data{background:var(--surface2);border:1px solid var(--border);border-radius:6px;padding:12px;margin-bottom:14px;font-size:12px;font-family:var(--mono);color:var(--text);line-height:1.8}
.hitl-edit{display:none;margin-bottom:12px}
.hitl-edit textarea{width:100%;background:var(--surface2);border:1px solid var(--accent2);color:var(--text);font-family:var(--mono);font-size:12px;padding:10px;border-radius:6px;resize:vertical;min-height:80px;outline:none;line-height:1.6}
.hitl-edit-label{font-family:var(--mono);font-size:10px;color:var(--accent2);margin-bottom:6px;text-transform:uppercase;letter-spacing:.5px}
.hitl-btns{display:flex;gap:8px;flex-wrap:wrap;align-items:center}
.hbtn{border:none;font-family:var(--mono);font-size:11px;padding:8px 14px;border-radius:5px;cursor:pointer;font-weight:500;transition:all .15s}
.hbtn-approve{background:var(--accent);color:#0a0c0f}
.hbtn-approve:hover{background:#00ffca}
.hbtn-edit{background:var(--accent2);color:#fff}
.hbtn-edit:hover{background:#9580ff}
.hbtn-regen{background:var(--surface2);color:var(--warn);border:1px solid rgba(245,166,35,.4)}
.hbtn-regen:hover{background:rgba(245,166,35,.1)}
.hbtn-reject{background:transparent;color:var(--danger);border:1px solid rgba(255,77,109,.4)}
.hbtn-reject:hover{background:rgba(255,77,109,.08)}
.hitl-timer{font-family:var(--mono);font-size:10px;color:var(--muted);margin-left:auto}
::-webkit-scrollbar{width:4px}
::-webkit-scrollbar-track{background:transparent}
::-webkit-scrollbar-thumb{background:var(--border);border-radius:99px}

.step-row{display:flex;align-items:baseline;gap:8px;padding:4px 0;
          border-bottom:1px solid rgba(35,40,48,.6);font-size:11px;}
.step-row:last-child{border-bottom:none;}
.step-icon{font-family:var(--mono);color:var(--muted);font-size:12px;
           min-width:14px;flex-shrink:0;}
.step-label{font-family:var(--mono);color:var(--muted);font-size:10px;text-transform:uppercase;letter-spacing:.8px;white-space:nowrap;width:160px;flex-shrink:0;overflow:hidden;text-overflow:ellipsis}
.step-val{font-family:var(--mono);font-size:11px;color:var(--text);
          flex:1;word-break:break-word;}
.step-val.pending{color:var(--muted);font-style:italic;}
.step-val.good{color:var(--accent);}
.step-val.warn{color:var(--warn);}
</style>
</head>
<body>
<div class="shell">
  <header>
    <div class="logo" style="font-size:11px;letter-spacing:-.2px;max-width:260px;line-height:1.3">An Agentic Framework <span style="color:var(--accent2)">for Reliable RAG using Verification, Correction and Multi-Model Evaluation</span></div>
    <span class="htag">LangGraph &middot; Mistral-7B &middot; DeBERTa-NLI</span>
    <div class="hright">
      <span class="pbadge" id="headerPersona">encyclopedic</span>
      <span class="hitl-ind" id="hitlIndicator">&#x1F464; HITL ON</span>
      <div class="sdot"></div>
      <span class="slabel">Mistral-7B &middot; DeBERTa-NLI</span>
      <span class="mbadge demo" id="modeBadge">demo mode</span>
    </div>
  </header>

  <aside>
    <div class="ssec">
      <div class="slbl">Persona</div>
      <div class="persona-cards">
        <div class="pcard active" data-persona="encyclopedic" onclick="selectPersona(this)">
          <div class="pcard-name"><span class="adot"></span>encyclopedic</div>
          <div class="pcard-desc">Expert assistant across history, science &amp; culture.</div>
        </div>
        <div class="pcard" data-persona="academic" onclick="selectPersona(this)">
          <div class="pcard-name"><span class="adot"></span>academic</div>
          <div class="pcard-desc">Research assistant &mdash; precise, citation-quality answers.</div>
        </div>
        <div class="pcard" data-persona="concise" onclick="selectPersona(this)">
          <div class="pcard-name"><span class="adot"></span>concise</div>
          <div class="pcard-desc">Minimal assistant &mdash; fewest words possible.</div>
        </div>
        <div class="pcard" data-persona="custom" onclick="selectPersona(this)">
          <div class="pcard-name"><span class="adot"></span>custom</div>
          <div class="pcard-desc">Define your own persona instructions below.</div>
        </div>
      </div>
    </div>

    <!-- Custom persona input — shown only when custom selected -->
    <div id="customPersonaBox" style="display:none;margin:0 0 12px">
      <div class="slbl" style="margin-bottom:4px">✏ Custom Persona Instructions</div>
      <textarea id="customPersonaInput" rows="4"
        placeholder="e.g. You are a medical expert. Answer with clinical precision and cite evidence."
        style="width:100%;background:var(--surface2);border:1px solid var(--border);
               color:var(--text);font-family:var(--mono);font-size:11px;
               padding:8px 10px;border-radius:6px;resize:vertical;box-sizing:border-box">
      </textarea>
      <div style="font-size:10px;color:var(--muted);margin-top:3px">
        Enter instructions before running.
      </div>
    </div>

    <div class="ssec">
      <div class="slbl">Backend</div>
      <div class="trow">
        <label class="tlbl">Connect to pipeline</label>
        <label class="tgl">
          <input type="checkbox" id="backendToggle" onchange="onBackendToggle()"/>
          <span class="ttrack"></span><span class="tthumb"></span>
        </label>
      </div>
      <div id="backendPanel" style="display:none">
        <div class="urlrow">
          <input class="urlinput" id="backendUrl" placeholder="https://xxxx.ngrok-free.app"/>
          <button class="cbtn" onclick="checkHealth()">Test</button>
        </div>
      </div>
      <p class="bnote" id="modeNote">Demo mode &mdash; results are simulated. Enable toggle to connect your notebook pipeline.</p>
      <div id="hitlControls" style="display:none;margin-top:12px">
        <div class="trow" style="margin-bottom:4px">
          <label class="tlbl">Human in the Loop</label>
          <label class="tgl">
            <input type="checkbox" id="hitlToggle" onchange="onHitlToggle()"/>
            <span class="ttrack"></span><span class="tthumb"></span>
          </label>
        </div>
        <div id="hitlPointsPanel" style="display:none">
          <div class="hcb-group">
            <div class="hcb-title">Pause at checkpoints:</div>
            <label class="hcb-row"><input type="checkbox" id="hcDecomp" checked/> &nbsp;&#x2460; Query Decomposition</label>
            <label class="hcb-row"><input type="checkbox" id="hcGen"    checked/> &nbsp;&#x2467; Answer Generation</label>
            <label class="hcb-row"><input type="checkbox" id="hcVerif"  checked/> &nbsp;&#x2468; Verification (low confidence)</label>
          </div>
        </div>
      </div>
    </div>

    <div class="ssec">
      <div class="slbl">Model Config</div>
      <div class="crow"><label>N Samples</label><input type="range" min="1" max="7" value="5" oninput="document.getElementById('nSV').textContent=this.value"/><span class="cval" id="nSV">5</span></div>
      <div class="crow"><label>Temperature</label><input type="range" min="1" max="10" value="7" oninput="document.getElementById('tV').textContent='0.'+this.value"/><span class="cval" id="tV">0.7</span></div>
      <div class="crow"><label>Max Tokens</label><input type="range" min="5" max="30" value="20" oninput="document.getElementById('mV').textContent=this.value"/><span class="cval" id="mV">20</span></div>
    </div>

    <div class="ssec">
      <div class="slbl">Agent Model Families</div>
      <div class="agent-list">
        <span class="badge" style="color:#00e5b0;border-color:rgba(0,229,176,.3);background:rgba(0,229,176,.06)">Mistral-7B</span>
        <span class="badge bp">DeBERTa-NLI</span>
        <span class="badge bg">BGE-large</span>
        <span class="badge bo">MS-MARCO</span>
      </div>
      <div style="font-size:11px;color:var(--muted);line-height:1.6;margin-top:8px"><strong style="color:var(--accent2)">DeBERTa</strong> verifies <strong style="color:var(--accent)">Mistral-7B</strong> output independently &mdash; no circular verification.</div>
    </div>

    <div class="ssec">
      <div class="slbl">Sample Questions</div>
      <button class="sbtn" onclick="setQ(this)">Were Scott Derrickson and Ed Wood of the same nationality?</button>
      <button class="sbtn" onclick="setQ(this)">Which other Mexican Formula One driver held the podium besides the Force India driver born in 1990?</button>
      <button class="sbtn" onclick="setQ(this)">Rock Springs is a collection of short stories by an author born in what year?</button>
      <button class="sbtn" onclick="setQ(this)">Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?</button>
      <button class="sbtn" onclick="setQ(this)">Are Local H and For Against both from the United States?</button>
    </div>
  </aside>

  <main>
    <div class="qarea">
      <div class="qlabel">Ask a multi-hop question</div>
      <div class="qbox">
        <textarea id="qInput" placeholder="e.g. Which studio album did Kanye West record with Roc-A-Fella Records and soul singer Dwele?"></textarea>
        <div class="btngrp">
          <button class="runbtn" id="runBtn" onclick="run()">&#x25B6; Run</button>
          <button class="clrbtn" onclick="clearOutput()">&#x2715; Clear</button>
        </div>
      </div>
    </div>
    <div class="oarea" id="oArea">
      <div class="empty" id="emptyState">
        <div class="empty-g">&#x2B23;</div>
        <p>Choose a persona and enter a multi-hop question. Each pipeline stage will appear as it runs.</p>
        <p style="font-size:11px;color:#374151">Enable the backend toggle for real results, or run in demo mode to see the pipeline flow.</p>
      </div>
    </div>
  </main>
</div>

<!-- HITL overlay + panel -->
<div id="hitlOverlay" onclick="decide('approve')"></div>
<div id="hitlPanel">
  <div class="hitl-header">&#x1F464; Human Review Required <span id="hitlCheckpoint" style="opacity:.6"></span></div>
  <div class="hitl-title" id="hitlTitle"></div>
  <div class="hitl-msg"   id="hitlMsg"></div>
  <div class="hitl-data"  id="hitlData"></div>
  <div class="hitl-edit"  id="hitlEdit">
    <div class="hitl-edit-label" id="hitlEditLabel">Edit below:</div>
    <textarea id="hitlField" rows="3"></textarea>
  </div>
  <div class="hitl-btns">
    <button class="hbtn hbtn-approve" onclick="decide('approve')">&#x2713; Approve</button>
    <button class="hbtn hbtn-edit"    onclick="decide('edit')"    id="btnEdit">&#x270E; Edit</button>
    <button class="hbtn hbtn-regen"   onclick="decide('regenerate')" id="btnRegen">&#x21BA; Regenerate</button>
    <button class="hbtn hbtn-reject"  onclick="decide('reject')">&#x2715; Reject</button>
    <span class="hitl-timer" id="hitlTimer">Auto-approves in 120s</span>
  </div>
</div>

<script>
// ══════════════════════════════════════════════════════
// AGENT METADATA
// ══════════════════════════════════════════════════════
const AGENT_META={
  QueryDecompositionAgent:  {cls:"fmistral",  lbl:"mistral-7b",    budget:15, icon:"①"},
  HybridRetrievalAgent:     {cls:"fbge",     lbl:"bge-large",     budget:2,  icon:"②"},
  RetrievalQualityGateAgent:{cls:"fbge",     lbl:"bge-large",     budget:5,  icon:"③"},
  WikipediaFallbackAgent:   {cls:"fbge",     lbl:"wikipedia",     budget:5,  icon:"③"},
  RerankingAgent:           {cls:"fcross",   lbl:"cross-encoder", budget:2,  icon:"④"},
  GraphRAGAgent:            {cls:"fnone",    lbl:"no-model",      budget:1,  icon:"⑤"},
  ICAAgent:                 {cls:"fmistral",  lbl:"mistral-7b",    budget:3,  icon:"⑥"},
  SentenceSelectionAgent:   {cls:"ftfidf",   lbl:"tfidf",         budget:1,  icon:"⑦"},
  AnswerGenerationAgent:    {cls:"fmistral",  lbl:"mistral-7b",    budget:45, icon:"⑧"},
  VerificationAgent:        {cls:"fdeberta", lbl:"deberta-nli",   budget:3,  icon:"⑨"},
  RegenerationAgent:        {cls:"fmistral",  lbl:"mistral-7b",    budget:45, icon:"⑩"},
};

const AGENT_DETAILS={
  QueryDecompositionAgent: {
    desc:"Decomposes the question into 2 atomic sub-questions using Mistral-7B few-shot CoT.",
    infoKeys:{sub_questions:"Sub-questions generated"}
  },
  HybridRetrievalAgent:{
    desc:"Fuses FAISS dense retrieval (BGE-large) with BM25 sparse retrieval.",
    infoKeys:{doc_count:"Documents retrieved"}
  },
  RetrievalQualityGateAgent:{
    desc:"Scores retrieval quality via BGE cosine similarity. Routes to Wikipedia if score < 0.3.",
    infoKeys:{quality_score:"Quality score",wiki_needed:"Wikipedia triggered"}
  },
  WikipediaFallbackAgent:{
    desc:"Fetches evidence from Wikipedia REST API when retrieval quality is insufficient.",
    infoKeys:{doc_count:"Docs fetched"}
  },
  RerankingAgent:{
    desc:"Reranks candidates using MS-MARCO MiniLM cross-encoder for relevance.",
    infoKeys:{doc_count:"Docs after rerank"}
  },
  GraphRAGAgent:{
    desc:"Expands evidence via BFS on entity co-occurrence graph (hop depth 2).",
    infoKeys:{hop_docs:"Docs added via graph"}
  },
  ICAAgent:{
    desc:"Information-Calibrated Aggregation (DePaC §3.2): ranks docs by BGE cosine similarity — independent of generation model.",
    infoKeys:{top_score:"Top ICA score"}
  },
  SentenceSelectionAgent:{
    desc:"Selects top-2 sentences per document using TF-IDF relevance scoring.",
    infoKeys:{snippet:"Context snippet"}
  },
  AnswerGenerationAgent:{
    desc:"Generates N=5 self-consistency candidates using Mistral-7B. BGE cosine tie-breaks.",
    infoKeys:{answer:"Selected answer",candidates:"Candidates generated"}
  },
  VerificationAgent:{
    desc:"Verifies answer via DeBERTa-v3 NLI — independent of Mistral-7B (FEVER-inspired: SUPPORTS/REFUTES/NOT ENOUGH INFO).",
    infoKeys:{entailment:"Entailment rate",contradicted:"Contradiction detected"}
  },
  RegenerationAgent:{
    desc:"Triggered by DeBERTa contradiction. Regenerates with stricter Mistral-7B prompt.",
    infoKeys:{answer:"New answer",attempt:"Attempt number"}
  },
};

const PSYS={
  encyclopedic:"Expert encyclopedic assistant with deep knowledge across history, science, culture...",
  academic:    "Academic research assistant — precise, citation-quality factual answers...",
  concise:     "Concise factual assistant. Always answers in fewest words possible...",
  custom:      "Define your own persona instructions in the text box below.",
};

const DEMO_LAT={
  QueryDecompositionAgent:0.82,HybridRetrievalAgent:.10,
  RetrievalQualityGateAgent:1.16,RerankingAgent:.02,GraphRAGAgent:.03,
  ICAAgent:.40,SentenceSelectionAgent:.03,AnswerGenerationAgent:1.20,
  VerificationAgent:1.13
};

const DEMO_INFO={
  QueryDecompositionAgent: d=>({sub_questions:["Who is related to "+d.q.split(" ").slice(-2).join(" ")+"?","What is the timeframe or context?"]}),
  HybridRetrievalAgent:    ()=>({doc_count:7}),
  RetrievalQualityGateAgent:()=>({quality_score:0.691,wiki_needed:false}),
  RerankingAgent:          ()=>({doc_count:5}),
  GraphRAGAgent:           ()=>({hop_docs:3}),
  ICAAgent:                ()=>({top_score:0.847}),
  SentenceSelectionAgent:  d=>({snippet:d.q.slice(0,60)+"..."}),
  AnswerGenerationAgent:   d=>({answer:getDemoAns(d.q),candidates:5}),
  VerificationAgent:       ()=>({entailment:0.77,contradicted:false}),
};

const DEMO_ANS_MAP={nationality:"yes",pedro:"Pedro Rodriguez",
  "rock springs":"1944",kaiser:"Henry J. Kaiser",
  "local h":"yes",seger:"Bob Seger",laleli:"no",beckham:"Sir Alex Ferguson"};

// ══════════════════════════════════════════════════════
// STATE
// ══════════════════════════════════════════════════════
const ST={
  running:false,persona:"encyclopedic",useBackend:false,backendUrl:"",
  hitlMode:false,pollTimer:null,countdownTimer:null,
  timeLeft:120,editing:false,hitlSession:null,
  shownSessions:new Set(),  // track sessions already shown to prevent duplicates
  activeAgentCard:null,
};

// ══════════════════════════════════════════════════════
// SIDEBAR CONTROLS
// ══════════════════════════════════════════════════════
// ── Global state for agent card timers ───────────────────────────────────────
var _agentStartTimes = {};
var _agentTimers     = {};


function selectPersona(c){
  document.querySelectorAll(".pcard").forEach(x=>x.classList.remove("active"));
  c.classList.add("active");
  ST.persona=c.dataset.persona;
  document.getElementById("headerPersona").textContent=ST.persona;
  // Show custom persona textbox only when custom is selected
  var box=document.getElementById("customPersonaBox");
  if(box) box.style.display=(ST.persona==="custom")?"block":"none";
}
function onBackendToggle(){
  ST.useBackend=document.getElementById("backendToggle").checked;
  document.getElementById("backendPanel").style.display=ST.useBackend?"block":"none";
  document.getElementById("hitlControls").style.display=ST.useBackend?"block":"none";
  const b=document.getElementById("modeBadge");
  if(ST.useBackend){b.className="mbadge live";b.textContent="live mode"}
  else{b.className="mbadge demo";b.textContent="demo mode"}
}
function onHitlToggle(){
  ST.hitlMode=document.getElementById("hitlToggle").checked;
  document.getElementById("hitlPointsPanel").style.display=ST.hitlMode?"block":"none";
  document.getElementById("hitlIndicator").style.display=ST.hitlMode?"inline":"none";
}
async function checkHealth(){
  const url=document.getElementById("backendUrl").value.trim().replace(/\/$/,"");
  if(!url)return;
  ST.backendUrl=url;
  const n=document.getElementById("modeNote");
  n.textContent="Checking...";
  try{
    const r=await fetch(url+"/health",{
      headers:{"ngrok-skip-browser-warning":"true"},
      signal:AbortSignal.timeout(6000)
    });
    if(r.ok){
      const d=await r.json();
      ST.useBackend=true;
      document.getElementById("modeBadge").className="mbadge live";
      document.getElementById("modeBadge").textContent="live mode";
      document.getElementById("hitlControls").style.display="block";
      n.innerHTML='&#x2713; Connected &middot; <strong style="color:var(--accent)">'+(d.model||"gpt-4o-mini")+'</strong>';
    } else {
      n.innerHTML='<span style="color:var(--danger)">&#x2715; Error '+r.status+'</span>';
    }
  }catch(err){
    ST.useBackend=false;
    const msg=err&&err.message?err.message:"connection refused";
    n.innerHTML='<span style="color:var(--danger)">&#x2715; Unreachable &mdash; '+msg+'</span>';
    console.error("Health check failed:",err);
  }
}
document.getElementById("backendUrl").addEventListener("change",checkHealth);
function setQ(b){document.getElementById("qInput").value=b.textContent.trim()}
function getHitlPoints(){const p=[];if(document.getElementById("hcDecomp").checked)p.push("decomposition");if(document.getElementById("hcGen").checked)p.push("generation");if(document.getElementById("hcVerif").checked)p.push("verification");return p}

// ══════════════════════════════════════════════════════
// OUTPUT MANAGEMENT
// ══════════════════════════════════════════════════════
function resetOutput(){
  const a=document.getElementById("oArea");
  Array.from(a.children).forEach(c=>{if(c.id!=="emptyState")a.removeChild(c)});
  document.getElementById("emptyState").style.display="none";
  a.scrollTop=0;
  ST.activeAgentCard=null;
}
function clearOutput(){
  resetOutput();
  document.getElementById("qInput").value="";
  document.getElementById("emptyState").style.display="flex";
  ST.running=false;
  const b=document.getElementById("runBtn");b.disabled=false;b.textContent="\u25B6 Run";
  stopPoll();
}
function appendEl(el){
  const a=document.getElementById("oArea");
  a.appendChild(el);a.scrollTop=a.scrollHeight;
}
function toggleCard(h){h.classList.toggle("open");h.nextElementSibling.classList.toggle("open")}
function getDemoAns(q){const lq=q.toLowerCase();for(const[k,v]of Object.entries(DEMO_ANS_MAP))if(lq.includes(k))return v;const w=q.match(/[A-Z][a-zA-Z]+(?:\s[A-Z][a-zA-Z]+)*/g);return w?w[w.length-1]:"Verified Answer"}

// ══════════════════════════════════════════════════════
// AGENT CARD FACTORY
// ══════════════════════════════════════════════════════
function makeAgentCard(name, state){
  state = state || "pending";
  var m   = AGENT_META[name] || {cls:"fnone", lbl:"unknown", budget:20, icon:"·"};
  var det = AGENT_DETAILS[name] || {desc:"", infoKeys:{}};
  var d   = document.createElement("div");
  d.className = "acard" + (state==="running" ? " running" : "");
  d.id = "card_" + name;

  var isOpen = (state==="running" || state==="done");

  // Build step rows without nested template literals
  var stepRowsHtml = "";
  var infoKeys = Object.keys(det.infoKeys);
  for(var ki=0; ki<infoKeys.length; ki++){
    var k2    = infoKeys[ki];
    var lbl2  = det.infoKeys[k2];
    stepRowsHtml +=
      '<div class="step-row" id="step_'+name+'_'+k2+'">'+
        '<span class="step-icon"></span>'+
        '<span class="step-label">'+lbl2+'</span>'+
        '<span class="step-val pending" id="stepval_'+name+'_'+k2+'">—</span>'+
      '</div>';
  }
  var stepBlock = stepRowsHtml
    ? '<div style="margin:6px 0 2px">'+stepRowsHtml+'</div>'
    : "";

  var latHtml  = (state==="running")
    ? '<span id="timer_'+name+'" class="live-timer">0.0s</span>'
    : "&mdash;";
  var nameHtml = (state==="running")
    ? '<span class="running-dot"></span> '+name
    : name;
  var chevSt   = isOpen ? 'style="transform:rotate(90deg)"' : "";
  var barCls   = (state==="running") ? " running" : "";
  var barW     = (state==="running") ? "100" : "0";

  d.innerHTML =
    '<div class="'+(isOpen?"ahead open":"ahead")+'" onclick="toggleCard(this)">'+
      '<span class="anum">'+m.icon+'</span>'+
      '<span class="aname" id="aname_'+name+'">'+nameHtml+'</span>'+
      '<span class="afam '+m.cls+'">'+m.lbl+'</span>'+
      '<span class="alat" id="lat_'+name+'">'+latHtml+'</span>'+
      '<span class="achev" '+chevSt+'>&#x25B6;</span>'+
    '</div>'+
    '<div class="'+(isOpen?"abody open":"abody")+'">'+
      '<p style="font-size:11px;color:var(--muted);margin-bottom:10px;line-height:1.5">'+det.desc+'</p>'+
      '<div class="lattrack" style="margin-bottom:10px">'+
        '<div class="latfill'+barCls+'" id="bar_'+name+'" style="width:'+barW+'%"></div>'+
      '</div>'+
      stepBlock+
    '</div>';

  if(state==="running"){
    _agentStartTimes[name] = performance.now();
    _agentTimers[name] = setInterval(function(){
      var el = document.getElementById("timer_"+name);
      if(el) el.textContent = ((performance.now()-_agentStartTimes[name])/1000).toFixed(1)+"s";
    }, 100);
  }
  return d;
}

function formatVal(k, v){
  if(k==="sub_questions"&&Array.isArray(v)){
    var r="";
    for(var i=0;i<v.length;i++) r+='<span style="display:block;color:var(--accent2);margin-bottom:2px">'+(i+1)+". "+v[i]+"</span>";
    return r;
  }
  if(k==="contradicted") return v?'<span class="step-val warn">&#x26A0; YES</span>':'<span class="step-val good">no</span>';
  if(k==="entailment")   return '<span class="step-val '+(v>=0.5?"good":"warn")+'">'+(v*100).toFixed(0)+'%</span>';
  if(k==="wiki_needed")  return v?'<span class="step-val warn">YES &#x2014; fallback triggered</span>':'<span class="step-val good">no</span>';
  if(k==="quality_score")return '<span class="step-val '+(v>=0.3?"good":"warn")+'">'+v+'</span>';
  if(["top_score","doc_count","hop_docs","candidates","attempt","context_len"].indexOf(k)>=0)
    return '<span class="step-val good">'+v+'</span>';
  if(k==="snippet"||k==="answer")
    return '<span class="step-val" style="color:var(--accent);max-width:220px;display:inline-block;white-space:nowrap;overflow:hidden;text-overflow:ellipsis" title="'+String(v).replace(/"/g,"&quot;")+'">'+v+'</span>';
  return '<span class="step-val">'+v+'</span>';
}

function updateAgentDone(name, elapsed, info){
  const card=document.getElementById("card_"+name);
  if(!card)return;
  card.classList.remove("running");

  const m=AGENT_META[name]||{budget:20};
  const over=elapsed>m.budget;
  const pct=Math.min((elapsed/Math.max(m.budget,elapsed,.01))*100,100).toFixed(1);

  // Update latency label
  const latEl=document.getElementById("lat_"+name);
  if(latEl){latEl.textContent=elapsed.toFixed(2)+"s";if(over)latEl.classList.add("warn")}

  // Update bar
  const barEl=document.getElementById("bar_"+name);
  if(barEl){barEl.style.width=pct+"%";if(over)barEl.classList.add("warn")}

  // Update step-val spans directly — one span per info key
  if(info){
    var ks=Object.keys(info);
    for(var ki2=0;ki2<ks.length;ki2++){
      var k2=ks[ki2], v2=info[k2];
      var sv=document.getElementById("stepval_"+name+"_"+k2);
      if(!sv) continue;
      sv.className="step-val";
      // Format value
      var vs2;
      if(k2==="sub_questions"&&Array.isArray(v2)){
        var parts2=[];
        for(var si2=0;si2<v2.length;si2++) parts2.push((si2+1)+". "+v2[si2]);
        vs2=parts2.join("<br>");
      } else if(k2==="contradicted"){
        vs2=v2?'<span style="color:var(--warn)">&#x26A0; YES</span>':
               '<span style="color:var(--accent)">no</span>';
      } else if(k2==="entailment"){
        vs2='<span style="color:'+(v2>=0.5?"var(--accent)":"var(--warn)")+'">'+
            (v2*100).toFixed(0)+'%</span>';
      } else if(k2==="wiki_needed"){
        vs2=v2?'<span style="color:var(--warn)">yes — Wikipedia triggered</span>':
               '<span style="color:var(--accent)">no</span>';
      } else if(k2==="retrieval_source"){
        var smap={"corpus":"&#x1F4DA; corpus","wikipedia":"&#x1F310; wikipedia","web":"&#x1F50D; web"};
        var sc={"corpus":"var(--accent)","web":"var(--warn)","wikipedia":"var(--accent2)"}[v2]||"var(--text)";
        vs2='<span style="color:'+sc+'">'+(smap[v2]||v2)+'</span>';
      } else if(k2==="quality_score"){
        vs2='<span style="color:'+(v2>=0.3?"var(--accent)":"var(--warn)")+'">'+v2+'</span>';
      } else if(k2==="snippet"){
        vs2='<div style="color:var(--accent);font-size:11px;line-height:1.6;'+
            'white-space:pre-wrap;word-break:break-word;margin-top:4px;'+
            'padding:6px 8px;background:rgba(0,229,176,.04);'+
            'border-left:2px solid var(--accent);border-radius:0 4px 4px 0">'+
            String(v2)+'</div>';
      } else if(k2==="answer"){
        vs2='<span style="color:var(--accent);font-weight:500">'+String(v2)+'</span>';
      } else if(k2==="scores"&&Array.isArray(v2)){
        vs2=v2.join(", ");
      } else {
        vs2=String(v2);
      }
      sv.innerHTML=vs2;
    }
  }

  // Update header to show name without spinner
  const ahead=card.querySelector(".ahead");
  const aname=ahead.querySelector(".aname");
  if(aname)aname.innerHTML=" "+name;

  // Auto-open if it has interesting info
  const body=card.querySelector(".abody");
  if(body&&info&&Object.keys(info).length>0)body.classList.add("open");
  const hdr=card.querySelector(".ahead");
  if(hdr&&info&&Object.keys(info).length>0)hdr.classList.add("open");
}

function makeResultCard(data, isDemo){
  // Rejected pipeline — show error notice instead of result card
  if(data.error && !data.answer){
    var d=document.createElement("div");
    d.style.cssText="background:rgba(255,77,109,.06);border:1px solid rgba(255,77,109,.3);"
      +"border-radius:8px;padding:14px 16px;margin-top:8px;"
      +"font-family:var(--mono);font-size:11px;color:var(--danger)";
    d.textContent="Pipeline stopped: "+data.error;
    return d;
  }
  var hg  = (data.hallucination==="none");
  var rg  = (data.retrieval_quality>0.5);
  var dt  = isDemo?'<span style="color:var(--muted);font-size:10px;margin-left:8px;">[demo]</span>':"";
  var cps = data.hitl_checkpoints||[];
  var cpInner="";
  for(var ci=0;ci<cps.length;ci++){
    var cp=cps[ci];
    cpInner+='<div class="kvrow"><span class="kvkey">&#x1F464; '+cp.checkpoint+'</span>'+
             '<span class="kvval" style="color:var(--accent2)">'+cp.decision+(cp.auto?" (auto)":"")+'</span></div>';
  }
  var cpBlock=cps.length
    ?'<div style="margin-top:14px;border-top:1px solid var(--border);padding-top:12px">'+
     '<div style="font-family:var(--mono);font-size:10px;color:var(--accent2);text-transform:uppercase;letter-spacing:1px;margin-bottom:8px">&#x1F464; HITL Checkpoint Summary</div>'+
     cpInner+'</div>'
    :"";
  var d=document.createElement("div");
  d.className="rcard";
  d.innerHTML=
    (data.cannot_answer
      ? '<div class="cannot-banner">&#x26A0; CANNOT ANSWER &mdash; entailment below threshold after regeneration. Retrieved sources do not support a confident answer.</div>'
      : "")+
    '<div class="rtitle">Verified Answer &middot; <span style="color:var(--accent2)">'+ST.persona+'</span> persona '+
      '<span class="src-badge src-'+(data.retrieval_source||"corpus")+'">'+
        ({"corpus":"&#x1F4DA; corpus","wikipedia":"&#x1F310; wikipedia","web":"&#x1F50D; web"}[data.retrieval_source||"corpus"]||"corpus")+
      '</span>'+dt+'</div>'+
    '<div class="ranswer">'+(data.cannot_answer?"&mdash;":(data.answer||"&mdash;"))+'</div>'+
    (data.gold_answer
      ? (function(){
          var match = data.answer && data.gold_answer &&
            data.answer.trim().toLowerCase() === data.gold_answer.trim().toLowerCase();
          return '<div style="margin:10px 0 14px">'+
            '<div style="font-family:var(--mono);font-size:9px;text-transform:uppercase;'+
            'letter-spacing:1.5px;color:var(--muted);margin-bottom:4px">Gold Answer</div>'+
            '<div style="font-family:var(--serif);font-size:18px;font-weight:300;color:'+
            (match ? 'var(--accent)' : 'var(--warn)')+'">'+
            (data.gold_answer||'&mdash;')+'</div></div>';
        })()
      : "")+
    '<div class="mrow">'+
      '<div class="mchip"><div class="mval good">'+((data.entailment_rate||0)*100).toFixed(0)+'%</div><div class="mlbl">Entailment</div></div>'+
      '<div class="mchip"><div class="mval '+(rg?"good":"warn")+'">'+((data.retrieval_quality||0).toFixed(2))+'</div><div class="mlbl">Retrieval</div></div>'+
      '<div class="mchip"><div class="mval">'+(data.regen_attempts||0)+'</div><div class="mlbl">Regen</div></div>'+
      '<div class="mchip"><div class="mval '+(hg?"good":"warn")+'">'+(data.hallucination||"none")+'</div><div class="mlbl">Hallucin.</div></div>'+
      '<div class="mchip"><div class="mval">'+((data.wall_time||0).toFixed(1))+'s</div><div class="mlbl">Total</div></div>'+
    '</div>'+cpBlock;
  return d;
}

function mkErrCard(msg){
  const d=document.createElement("div");d.className="rcard";d.style.borderLeftColor="var(--danger)";
  d.innerHTML="<div class='rtitle' style='color:var(--danger)'>Pipeline Error</div><div style='font-family:var(--mono);font-size:12px;color:var(--muted);margin-top:8px;line-height:1.8;'>"+msg+"</div>";
  return d;
}

// ══════════════════════════════════════════════════════
// MAIN RUN
// ══════════════════════════════════════════════════════
async function run(){
  if(ST.running)return;
  const q=document.getElementById("qInput").value.trim();
  if(!q)return;
  ST.running=true;
  const btn=document.getElementById("runBtn");
  btn.disabled=true;btn.textContent="\u23F3";
  resetOutput();
  try{
    if(ST.useBackend&&ST.backendUrl) await runReal(q);
    else await runDemo(q);
  }catch(e){appendEl(mkErrCard(e.message))}
  finally{ST.running=false;btn.disabled=false;btn.textContent="\u25B6 Run";stopPoll()}
}

// ══════════════════════════════════════════════════════
// REAL BACKEND — SSE streaming
// ══════════════════════════════════════════════════════
// ── Backend helpers — fetch with ngrok header ─────────────────────────────────
async function apiPost(path, body){
  const resp = await fetch(ST.backendUrl.replace(/\/+$/,"")+path,{
    method:"POST",
    headers:{
      "Content-Type":"application/json",
      "ngrok-skip-browser-warning":"true",
    },
    body: JSON.stringify(body),
    signal: AbortSignal.timeout(60000),
  });
  if(!resp.ok){
    const e=await resp.json().catch(()=>({detail:resp.statusText}));
    throw new Error(e.detail||"Server error "+resp.status);
  }
  return resp.json();
}

async function apiGet(path){
  const resp = await fetch(ST.backendUrl.replace(/\/+$/,"")+path,{
    headers:{"ngrok-skip-browser-warning":"true"},
    signal: AbortSignal.timeout(10000),
  });
  if(!resp.ok) throw new Error("GET "+path+" failed: "+resp.status);
  return resp.json();
}

async function pollJobUntilDone(jobId){
  for(let i=0;i<600;i++){
    await new Promise(r=>setTimeout(r,2000));
    let job;
    try{ job=await apiGet("/job/"+jobId); }catch(e){ continue; }
    if(job.status==="done")   return job.result;
    if(job.status==="error")  throw new Error(job.error||"Pipeline error");
  }
  throw new Error("Pipeline timed out after 20 minutes.");
}

async function runReal(q){
  const hitl  = ST.hitlMode;
  const pts   = getHitlPoints();
  const cpEl  = document.getElementById("customPersonaInput");
  const custP = cpEl ? cpEl.value.trim() : "";

  // ── Step 1: POST /query → get job_id immediately ──────────────────────────
  const payload = {
    question:       q,
    persona:        ST.persona,
    hitl_mode:      hitl,
    hitl_points:    pts,
    custom_persona: custP,
  };

  let jobId;
  try {
    const resp = await apiPost("/query", payload);
    jobId = resp.job_id;
    if(!jobId) throw new Error("No job_id returned from backend");
  } catch(e) {
    appendEl(mkErrCard(e.message));
    return;
  }

  // Running banner
  const banner = document.createElement("div");
  banner.id = "run-banner";
  banner.style.cssText = "font-family:var(--mono);font-size:11px;color:var(--accent);" +
    "padding:8px 0;border-bottom:1px solid var(--border);margin-bottom:4px";
  banner.innerHTML = "&#x25B6; Pipeline running &middot; " +
    "<span style=\"color:var(--accent2)\">" + ST.persona + "</span> persona" +
    (hitl ? " &middot; <span style=\"color:var(--accent2)\">&#x1F464; HITL ON</span>" : "");
  appendEl(banner);

  // ── Step 2: Poll /progress every 600ms — show agents as they complete ──────
  const shownAgents = new Set();
  let finalData     = null;
  let attempts      = 0;
  const MAX_POLLS   = 1200;  // 20 min timeout

  if(hitl) startPoll();

  while(attempts < MAX_POLLS) {
    await new Promise(r => setTimeout(r, 600));
    attempts++;

    let prog;
    try { prog = await apiGet("/job/" + jobId + "/progress"); }
    catch(e) { continue; }

    // Render any newly-completed agents
    const agents = prog.agents || [];
    for(const agentData of agents) {
      if(shownAgents.has(agentData.agent)) continue;
      shownAgents.add(agentData.agent);

      // Brief "running" flash then update to done
      const card = makeAgentCard(agentData.agent, "running");
      appendEl(card);
      await new Promise(r => setTimeout(r, 400));
      updateAgentDone(agentData.agent, agentData.elapsed, agentData.info || {});
    }

    if(prog.status === "done") {
      // Fetch full result
      try {
        const job = await apiGet("/job/" + jobId);
        finalData = job.result;
      } catch(e) { appendEl(mkErrCard(e.message)); return; }
      break;
    }
    if(prog.status === "error") {
      const job = await apiGet("/job/" + jobId).catch(()=>({error:"Pipeline failed"}));
      appendEl(mkErrCard(job.error || "Pipeline error"));
      return;
    }
  }

  if(!finalData) { appendEl(mkErrCard("Pipeline timed out.")); return; }

  // HITL complete — hide panel but keep loop running for result
  if(hitl){
    document.getElementById("hitlPanel").style.display="none";
    document.getElementById("hitlOverlay").style.display="none";
    if(ST.countdownTimer){clearInterval(ST.countdownTimer);ST.countdownTimer=null}
    if(ST.pollTimer){clearInterval(ST.pollTimer);ST.pollTimer=null}
    ST.shownSessions=new Set();
  }

  // Remove banner
  const b = document.getElementById("run-banner");
  if(b) b.remove();

  // Done banner
  const done = document.createElement("div");
  done.style.cssText = "font-family:var(--mono);font-size:11px;color:var(--accent);" +
    "padding:8px 0;border-bottom:1px solid var(--border);margin-bottom:4px";
  done.innerHTML = "&#x2714; Pipeline complete &middot; " +
    "<span style=\"color:var(--accent2)\">" + ST.persona + "</span> persona" +
    " &middot; " + ((finalData.wall_time||0).toFixed(1)) + "s";
  appendEl(done);

  const AGENT_ORDER = [
    "QueryDecompositionAgent","HybridRetrievalAgent","RetrievalQualityGateAgent",
    "WikipediaFallbackAgent","RerankingAgent","GraphRAGAgent","ICAAgent",
    "SentenceSelectionAgent","AnswerGenerationAgent","VerificationAgent","RegenerationAgent"
  ];

  // Render agents not yet shown (missed between polls)
  for(const agent of AGENT_ORDER) {
    if(shownAgents.has(agent)) continue;
    const t = (finalData.timings||{})[agent];
    if(!t || t===0) continue;
    const card = makeAgentCard(agent, "done");
    appendEl(card);
    updateAgentDone(agent, t, buildInfo(agent, finalData));
  }

  appendEl(makeResultCard(finalData, false));
}

// Build info dict for an agent from final result data
function buildInfo(agent, data) {
  const m = {};
  if(agent==="QueryDecompositionAgent" && data.sub_questions && data.sub_questions.length)
    m.sub_questions = data.sub_questions;
  if(agent==="HybridRetrievalAgent" && data.doc_count)
    m.doc_count = data.doc_count;
  if(agent==="RetrievalQualityGateAgent"){
    m.quality_score    = +(data.retrieval_quality||0).toFixed(3);
    m.retrieval_source = data.retrieval_source||"corpus";
    m.wiki_needed      = (data.retrieval_source||"corpus")!=="corpus";
  }
  if(agent==="RerankingAgent" && data.reranked_count)
    m.doc_count = data.reranked_count;
  if(agent==="GraphRAGAgent" && data.graph_docs_added)
    m.hop_docs = data.graph_docs_added;
  if(agent==="ICAAgent" && data.top_ica_score)
    m.top_score = data.top_ica_score;
  if(agent==="SentenceSelectionAgent" && data.context_snippet)
    m.snippet = data.context_snippet;
  if(agent==="AnswerGenerationAgent"){
    m.answer     = data.answer||"";
    m.candidates = data.n_samples||5;
  }
  if(agent==="VerificationAgent"){
    m.entailment   = data.entailment_rate||0;
    m.contradicted = (data.regen_attempts||0)>0;
  }
  if(agent==="RegenerationAgent" && (data.regen_attempts||0)>0){
    m.attempt = data.regen_attempts;
    m.answer  = data.answer||"";
  }
  return m;
}


async function runDemo(q){
  const banner=document.createElement("div");
  banner.style.cssText="font-family:var(--mono);font-size:11px;color:var(--warn);padding:8px 0;border-bottom:1px solid var(--border);margin-bottom:4px";
  banner.innerHTML="&#x25B6; Demo mode &middot; <span style=\"color:var(--accent2)\">"+ST.persona+"</span> persona &middot; <span style=\"opacity:.5\">connect backend for real results</span>";
  appendEl(banner);

  for(const[agent,base]of Object.entries(DEMO_LAT)){
    // Show agent card in running state
    const card=makeAgentCard(agent,"running");
    appendEl(card);

    // Simulate processing
    const delay=250+Math.random()*200;
    await new Promise(r=>setTimeout(r,delay));
    if(!ST.running){card.remove();return}

    const lat=base+(Math.random()-.5)*.3;
    const infoFn=DEMO_INFO[agent];
    const info=infoFn?infoFn({q}):{};
    updateAgentDone(agent,lat,info);
  }

  appendEl(makeResultCard({
    answer:getDemoAns(q),
    entailment_rate:.6+Math.random()*.3,
    retrieval_quality:.65+Math.random()*.2,
    regen_attempts:0,hallucination:"none",
    wall_time:Object.values(DEMO_LAT).reduce((a,b)=>a+b,0),
    hitl_checkpoints:[],
  },true));
}

// ══════════════════════════════════════════════════════
// HITL POLLING + PANEL
// ══════════════════════════════════════════════════════
function startPoll(){
  if(ST.pollTimer)return;
  ST.pollTimer=setInterval(async()=>{
    if(!ST.backendUrl||!ST.running)return;
    try{
      const r=await fetch(ST.backendUrl+"/hitl/pending",{headers:{"ngrok-skip-browser-warning":"true"}});
      if(!r.ok)return;
      const d=await r.json();
      if(d.count>0&&!ST.hitlSession)showPanel(d.pending[0]);
    }catch{}
  },1500);
}
function stopPoll(){
  if(ST.pollTimer){clearInterval(ST.pollTimer);ST.pollTimer=null}
  if(ST.countdownTimer){clearInterval(ST.countdownTimer);ST.countdownTimer=null}
  document.getElementById("hitlPanel").style.display="none";
  document.getElementById("hitlOverlay").style.display="none";
  ST.shownSessions=new Set();
  ST.hitlSession=null;ST.editing=false;
  // NOTE: do NOT set ST.running=false — runReal loop handles that
}

function showPanel(cp){
  if(ST.shownSessions.has(cp.session_id))return;  // already shown
  ST.shownSessions.add(cp.session_id);
  ST.hitlSession=cp.session_id;ST.editing=false;ST.timeLeft=120;
  const TITLES={decomposition:"① Query Decomposition Review",generation:"⑧ Answer Generation Review",verification:"⑨ Verification Review"};
  document.getElementById("hitlCheckpoint").textContent="checkpoint: "+cp.checkpoint;
  document.getElementById("hitlTitle").textContent=TITLES[cp.checkpoint]||cp.checkpoint;
  document.getElementById("hitlMsg").textContent=cp.data.message||"";
  const dd=document.getElementById("hitlData");
  if(cp.checkpoint==="decomposition"){
    dd.innerHTML='<div style="color:var(--muted);margin-bottom:4px">Sub-questions:</div>'+(cp.data.sub_questions||[]).map((sq,i)=>'<div style="color:var(--accent2)">'+(i+1)+". "+sq+"</div>").join("");
    document.getElementById("hitlField").value=(cp.data.sub_questions||[]).join("\n");
    document.getElementById("hitlEditLabel").textContent="Edit sub-questions (one per line):";
    document.getElementById("btnEdit").style.display="inline-block";
    document.getElementById("btnRegen").style.display="none";
  }else if(cp.checkpoint==="generation"){
    dd.innerHTML='<div style="color:var(--muted);margin-bottom:4px">Answer:</div><div style="color:var(--accent);font-family:var(--serif);font-size:18px;margin-bottom:8px">'+cp.data.answer+'</div><div style="color:var(--muted);font-size:11px">Entailment: '+(cp.data.entailment_rate*100).toFixed(0)+'%</div>';
    document.getElementById("hitlField").value=cp.data.answer;
    document.getElementById("hitlEditLabel").textContent="Edit the answer:";
    document.getElementById("btnEdit").style.display="inline-block";
    document.getElementById("btnRegen").style.display="inline-block";
  }else if(cp.checkpoint==="verification"){
    dd.innerHTML='<div style="color:var(--muted);margin-bottom:4px">Answer:</div><div style="color:var(--accent);margin-bottom:8px">'+cp.data.answer+'</div><div style="color:var(--warn)">&#x26A0; Low entailment: '+(cp.data.entailment_rate*100).toFixed(0)+'%</div><div style="color:var(--muted);font-size:11px;margin-top:4px">Hallucination: '+cp.data.hallucination+'</div>';
    document.getElementById("btnEdit").style.display="none";
    document.getElementById("btnRegen").style.display="inline-block";
  }
  document.getElementById("hitlEdit").style.display="none";
  document.getElementById("hitlTimer").textContent="Auto-approves in "+ST.timeLeft+"s";
  document.getElementById("hitlPanel").style.display="block";
  document.getElementById("hitlOverlay").style.display="block";
  if(ST.countdownTimer)clearInterval(ST.countdownTimer);
  ST.countdownTimer=setInterval(()=>{ST.timeLeft--;document.getElementById("hitlTimer").textContent="Auto-approves in "+ST.timeLeft+"s";if(ST.timeLeft<=0){clearInterval(ST.countdownTimer);ST.countdownTimer=null}},1000);
}

function decide(action){
  if(!ST.hitlSession)return;

  // Edit button — show edit field first, wait for confirm
  if(action==="edit"&&!ST.editing){
    ST.editing=true;
    document.getElementById("hitlEdit").style.display="block";
    document.getElementById("hitlPanel").querySelector(".hbtn-approve").textContent="\u2713 Confirm Edit";
    return;
  }

  // Build payload
  const payload={};
  if(ST.editing){
    const val=document.getElementById("hitlField").value.trim();
    const lines=val.split("\n").map(l=>l.trim()).filter(Boolean);
    if(lines.length>1) payload.sub_questions=lines;
    else               payload.answer=val;
    if(action==="approve") action="edit";
  }

  // Send decision to backend
  const sid = ST.hitlSession;
  fetch(ST.backendUrl+"/hitl/decide",{
    method:"POST",
    headers:{"Content-Type":"application/json","ngrok-skip-browser-warning":"true"},
    body:JSON.stringify({session_id:sid, action:action, payload:payload})
  }).catch(console.error);

  // Hide panel
  document.getElementById("hitlPanel").style.display="none";
  document.getElementById("hitlOverlay").style.display="none";
  document.getElementById("hitlPanel").querySelector(".hbtn-approve").textContent="\u2713 Approve";
  if(ST.countdownTimer){clearInterval(ST.countdownTimer);ST.countdownTimer=null}

  // Clear current session so next checkpoint can show
  ST.hitlSession=null;
  ST.editing=false;

  // For reject: show banner immediately so user knows pipeline stopped
  if(action==="reject"){
    const banner=document.createElement("div");
    banner.style.cssText="font-family:var(--mono);font-size:11px;color:var(--danger);"+
      "padding:8px 0;border-bottom:1px solid var(--border);margin:4px 0";
    banner.textContent="✕ Pipeline rejected at checkpoint — run stopped.";
    appendEl(banner);
  }

  // For regenerate: show banner so user knows it is re-running
  if(action==="regenerate"){
    const banner=document.createElement("div");
    banner.style.cssText="font-family:var(--mono);font-size:11px;color:var(--accent2);"+
      "padding:8px 0;border-bottom:1px solid var(--border);margin:4px 0";
    banner.textContent="↺ Regenerating — pipeline continuing with fresh answer...";
    appendEl(banner);
  }
}
</script>
</body>
</html>

'''

# ── Expose pipeline globals to backend module ─────────────────────────────────
_g = _types.ModuleType("pipeline_globals")
_g.PERSONA              = PERSONA
_g.PERSONA_MAP          = PERSONA_MAP
_g.run_pipeline         = run_pipeline
_g.run_pipeline_hitl    = run_pipeline_hitl
_g.decompose_query      = decompose_query
_g.hitl_sessions        = hitl_sessions
_g.rag_graph            = rag_graph
_g.embedder             = embedder
_g.MODEL_ID             = MODEL_ID
_g.HTML_SOURCE          = HTML_SOURCE
_g.N_SAMPLES            = N_SAMPLES
_g.TEMPERATURE          = TEMPERATURE
_g.MAX_NEW_TOKENS       = MAX_NEW_TOKENS
_g.CONFIDENCE_THRESHOLD = CONFIDENCE_THRESHOLD
_g.RETRIEVAL_QUALITY_THRESHOLD = RETRIEVAL_QUALITY_THRESHOLD
_g.ON_AGENT_DONE        = None
sys.modules["pipeline_globals"] = _g


BACKEND_CODE = r'''
import uuid, time, threading
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app, resources={r"/*": {"origins": "*"}})

# ── Job store ────────────────────────────────────────────────────────────────
_jobs:     dict = {}
_progress: dict = {}   # job_id → list of completed agents


@app.route("/health")
def health():
    import pipeline_globals as g
    return jsonify({"status": "ok", "model": g.MODEL_ID, "persona": g.PERSONA})


@app.route("/")
def serve_frontend():
    """Serve the HTML frontend with backend URL auto-injected."""
    import pipeline_globals as g
    content = g.HTML_SOURCE.replace(
        'backendUrl:""',
        'backendUrl:"' + request.host_url.rstrip("/") + '"',
    )
    from flask import Response
    return Response(content, mimetype="text/html")


@app.route("/personas")
def get_personas():
    import pipeline_globals as g
    return jsonify({"active": g.PERSONA, "available": list(g.PERSONA_MAP.keys())})


@app.route("/hitl/pending")
def hitl_pending():
    """Frontend polls this every 1.5s to find active checkpoints."""
    import pipeline_globals as g
    pending = [
        {"session_id": sid, "checkpoint": s.checkpoint, "data": s.data}
        for sid, s in g.hitl_sessions.items()
        if not s.event.is_set()
    ]
    return jsonify({"pending": pending, "count": len(pending)})


@app.route("/hitl/decide", methods=["POST"])
def hitl_decide():
    """
    Human submits decision. Calls sess.resolve() → threading.Event.set().
    Directly wakes the pipeline thread — no async involved.
    """
    import pipeline_globals as g
    body    = request.get_json(force=True)
    sid     = body.get("session_id", "")
    action  = body.get("action", "approve")
    payload = body.get("payload", {})

    sess = g.hitl_sessions.get(sid)
    if not sess:
        return jsonify({"error": "Session not found or already resolved"}), 404

    sess.resolve(action, payload)
    return jsonify({"status": "ok", "session_id": sid, "action": action})


@app.route("/job/<job_id>")
def get_job(job_id):
    """Poll this to check if a pipeline job has finished."""
    job = _jobs.get(job_id)
    if not job:
        return jsonify({"error": "Job not found"}), 404
    return jsonify(job)


@app.route("/job/<job_id>/progress")
def get_progress(job_id):
    """Poll for live per-agent progress. Frontend polls every 600ms."""
    return jsonify({
        "status": _jobs.get(job_id, {}).get("status", "pending"),
        "agents": _progress.get(job_id, []),
    })


@app.route("/query", methods=["POST"])
def run_query():
    import pipeline_globals as g
    body       = request.get_json(force=True)
    question   = body.get("question", "").strip()
    persona    = body.get("persona", "encyclopedic")
    hitl_mode  = body.get("hitl_mode", False)
    hitl_pts   = body.get("hitl_points",
                          ["decomposition", "generation", "verification"])

    if persona not in g.PERSONA_MAP:
        return jsonify({"error": f"Unknown persona: {persona}"}), 400

    g.PERSONA = persona
    # Look up gold answer from full HotPotQA dataset if available
    sample    = {"question": question,
                 "answer":   g.HOTPOT_GOLD.get(question, "")}

    # All requests use background thread + job_id so frontend polls /progress
    job_id = str(uuid.uuid4())[:8]
    _jobs[job_id]     = {"status": "pending", "result": None, "error": None}
    _progress[job_id] = []

    def _on_agent_done(agent_name, elapsed, info=None):
        _progress[job_id].append({
            "agent":   agent_name,
            "elapsed": round(elapsed, 3),
            "info":    info or {},
        })

    def _pipeline_thread():
        import pipeline_globals as _pg
        _pg.ON_AGENT_DONE = _on_agent_done
        try:
            t0 = time.perf_counter()
            if hitl_mode:
                result = g.run_pipeline_hitl(sample, hitl_pts, verbose=False)
            else:
                result = g.run_pipeline(sample, verbose=False)
            wall = round(time.perf_counter() - t0, 3)
            _jobs[job_id] = {"status": "done", "result": _fmt(result, wall), "error": None}
        except Exception as e:
            _jobs[job_id] = {"status": "error", "result": None, "error": str(e)}
        finally:
            _pg.ON_AGENT_DONE = None

    threading.Thread(target=_pipeline_thread, daemon=False).start()
    return jsonify({"job_id": job_id, "status": "pending"})


def _fmt(result: dict, wall: float) -> dict:
    return {
        "answer":            result.get("predicted_answer",   ""),
        "gold_answer":       result.get("gold_answer",        ""),
        "persona":           result.get("persona",            ""),
        "entailment_rate":   result.get("entailment_rate",    0.0),
        "regen_attempts":    result.get("regen_attempts",     0),
        "retrieval_quality": result.get("retrieval_quality",  0.0),
        "hallucination":     result.get("hallucination_type", "none"),
        "timings":           result.get("timings",            {}),
        "wall_time":         wall,
        "hitl_checkpoints":  result.get("hitl_checkpoints",  []),
        "retrieval_source":  result.get("retrieval_source",  "corpus"),
        "cannot_answer":     result.get("cannot_answer",     False),
        "web_snippets_used": result.get("web_snippets_used", 0),
    }
'''


# ── Expose pipeline globals to backend module ─────────────────────────────────
_g = _types.ModuleType("pipeline_globals")
_g.PERSONA              = PERSONA
_g.PERSONA_MAP          = PERSONA_MAP
_g.run_pipeline         = run_pipeline
_g.run_pipeline_hitl    = run_pipeline_hitl
_g.decompose_query      = decompose_query
_g.hitl_sessions        = hitl_sessions
_g.rag_graph            = rag_graph
_g.embedder             = embedder
_g.MODEL_ID             = MODEL_ID
_g.HTML_SOURCE          = HTML_SOURCE
_g.N_SAMPLES            = N_SAMPLES
_g.TEMPERATURE          = TEMPERATURE
_g.MAX_NEW_TOKENS       = MAX_NEW_TOKENS
_g.CONFIDENCE_THRESHOLD = CONFIDENCE_THRESHOLD
_g.RETRIEVAL_QUALITY_THRESHOLD = RETRIEVAL_QUALITY_THRESHOLD
_g.ON_AGENT_DONE        = None
_g.HOTPOT_GOLD          = {q["question"]: q["answer"]
                            for q in list(hotpot_train) + list(hotpot_eval)}
sys.modules["pipeline_globals"] = _g


with open("backend.py", "w") as f:
    f.write(BACKEND_CODE)
print("backend.py written.")

def _start():
    import backend as _bk
    _bk.app.run(host="0.0.0.0", port=8000,
                use_reloader=False, threaded=True, debug=False)

_t = threading.Thread(target=_start, daemon=True)
_t.start()
time.sleep(3)
print("Flask server running on port 8000. (threaded=True)")

from pyngrok import ngrok
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"   # ← get free token at dashboard.ngrok.com
ngrok.set_auth_token(NGROK_TOKEN)
tunnel     = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url

print(f"\n{'='*55}")
print(f"  Backend URL (paste into frontend):")
print(f"  {PUBLIC_URL}")
print(f"{'='*55}")
print(f"  Health  : {PUBLIC_URL}/health")
print(f"  Docs    : {PUBLIC_URL}/")
print(f"  HITL    : {PUBLIC_URL}/hitl/pending")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# HTML FRONTEND
#
# The Flask backend (previous cell) serves the HTML frontend directly at PUBLIC_URL/
# No second ngrok tunnel needed — one URL for everything.
#
# Usage:
#   1. Run previous cell — Flask starts, ngrok URL printed as PUBLIC_URL
#   2. Run this cell — opens the frontend URL in a clickable link
#   3. OR: download the HTML file below and open in Chrome directly
#
# Routes:
#   PUBLIC_URL/        → HTML frontend (backend URL auto-injected)
#   PUBLIC_URL/health  → Flask health check
#   PUBLIC_URL/query   → Pipeline endpoint
# ═══════════════════════════════════════════════════════════════════════════════

import base64
from IPython.display import display, HTML as _HTML
from google.colab import files

# ── Option 1: Open via ngrok URL (recommended) ────────────────────────────────
try:
    FRONTEND_URL = PUBLIC_URL + "/"
    print(f"\n{'='*60}")
    print(f"  Frontend URL (open in Chrome):")
    print(f"  {FRONTEND_URL}")
    print(f"{'='*60}")
    print("  Backend is pre-connected automatically.")
    print("  Just open the URL above — no manual URL paste needed.")

    display(_HTML(f"""
    <div style="font-family:monospace;padding:14px 16px;background:#111418;
                border:1px solid #232830;border-radius:8px;margin-top:8px">
      <div style="color:#6b7280;font-size:10px;text-transform:uppercase;
                  letter-spacing:1.5px;margin-bottom:8px">Frontend URL</div>
      <a href="{FRONTEND_URL}" target="_blank"
         style="color:#00e5b0;font-size:13px;text-decoration:none;
                font-family:monospace">
        {FRONTEND_URL}
      </a>
      <div style="color:#6b7280;font-size:10px;margin-top:8px">
        Backend pre-connected · Mistral-7B · DeBERTa-NLI · LangGraph
      </div>
    </div>
    """))

except NameError:
    print("PUBLIC_URL not set — run the previous ell first.")

# ── Option 2: Download HTML and open locally ──────────────────────────────────
# This injects the PUBLIC_URL so the downloaded file also auto-connects.
try:
    html_local = HTML_SOURCE.replace(
        'backendUrl:""',
        f'backendUrl:"{PUBLIC_URL}"'
    )
except NameError:
    html_local = HTML_SOURCE

with open("/content/rag_ui_stream.html", "w") as _f:
    _f.write(html_local)

print("\nDownloading rag_ui_stream.html (open in Chrome for best experience)...")
files.download("/content/rag_ui_stream.html")
